# FSTT-Net v2 — evidence-first pipeline

**Frequency-Spatial Temporal Semantic Token Transformer for Deepfake Detection**

Rebuild of the project on **detected, aligned face crops**, with FaceForensics++ c23
added and a staged experiment runner that refuses to waste GPU hours.

Everything writes to `/content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2`. The old `fsttnet` root is untouched.

---

## Run order

| Cell | What it does | Time |
|---|---|---|
| **1 · Data** | download → index → detect faces → cache. Resumable. | 9–11 h |
| **2 · Train** | prerequisites + STT-Repro + FSTT-Net | 4–6 h |
| **3 · Transfer + robustness** | evaluation only | 15–30 min |
| **4 · Manuscript pack** | every table and figure, IEEE format | 5–10 min |
| **5 · Evidence tracker** | scores you against the venue's bar | seconds |
| **6 · Full study** | seeds, same-schedule baselines, ablation — **gated** | 30–40 h |

After every runtime restart: rerun cell 2 before cells 3–6. Cell 1 only needs to
finish once.

## The one thing that decides everything

Cell 5 prints a checklist. One row on it matters more than all the others:

> **Proposed beats its own reproduction**

On the old centre-cropped data this row failed: ΔAUC −0.52 pp, 95% CI [−1.52, +0.34],
p = 0.28. Until it passes, extra seeds, extra baselines and extra ablations are all
measuring a difference that is not there — which is why **cell 6 refuses to run** until
it does. That gate is deliberate. It is there to stop you spending forty GPU hours on
tables that cannot support a claim.

## Two levers if the gate keeps failing

Both live at the top of cell 6:

- **`MODALITY_DROPOUT = 0.15`** — randomly blanks one input stream during training so
  the network cannot ignore the spectral branch. Right now nothing forces it to use
  that branch, which is the most likely reason the contribution does not show up.
  Enable it and rerun cell 2.
- **`DEGRADE_AUG`** — trains on JPEG/blur/noise. It will improve the robustness table,
  but then that table is no longer evidence of generalisation, because you trained for
  it. If you use it, say so in the paper and evaluate on severities you did not train on.

## Before you start

Runtime → Change runtime type → **GPU**. Kaggle token if `kagglehub` asks:

```python
from google.colab import files; files.upload()   # kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
```


---
## 1 · Data rebuild — detected faces + FF++ c23

Run once. **Resumable** — if Colab disconnects, rerun and it continues. It stops itself
at `TIME_BUDGET_HOURS` so the session is never killed mid-write.

Set `QUICK_TEST = True` for a 5-minute dry run first. Watch for
`face detector backend: dnn-ssd` and a detection rate well above 50%. If it falls back
to `center`, the cell stops on purpose — centre crops are exactly what broke the last
attempt.

In [ ]:
# ############################################################################
# CELL 0 (v4) :: DATA REBUILD - detected faces + FF++ c23
# ----------------------------------------------------------------------------
# This is STEP 1 and STEP 2 in one cell. Run it in a FRESH notebook (see the
# note at the bottom on why), on a Colab GPU or CPU runtime.
#
# WHAT IT FIXES
#   STEP 1  Your current crops are plain centre squares - MTCNN and Haar both
#           failed, so no face was ever detected. Your semantic prior assumes
#           eyes/nose/mouth sit at fixed positions, so the proposed model has
#           been running with its core mechanism disabled. This cell detects
#           faces with OpenCV's own DNN SSD (no torch, no facenet-pytorch) and
#           REFUSES to run if the detector is not working.
#   STEP 2  Adds FaceForensics++ c23, the corpus the base paper uses as the
#           cross-dataset training source. Without it your Table V cannot be
#           compared to theirs.
#
# BUILT FOR A 12-HOUR COLAB SESSION
#   * fully resumable - existing crops are skipped, so rerunning continues
#   * TIME_BUDGET_HOURS stops cleanly before the session is killed and tells
#     you to rerun; nothing is lost
#   * threaded extraction (VideoCapture and cv2.dnn release the GIL)
#   * partial manifests are flushed every FLUSH_EVERY videos
#   * writes to a NEW root, so your existing centre-crop results and figures
#     stay untouched for comparison
#
# EXPECTED WALL CLOCK (Colab, 4 threads)
#   Celeb-DF      download ~8 min   + extraction ~2.0-2.5 h
#   WildDeepfake  download ~8 min   + extraction ~3.5-4.5 h
#   FF++ c23      download ~20 min  + extraction ~2.5-3.5 h
#   Total roughly 9-11 h. If it stops, rerun - it picks up where it left off.
# ############################################################################

# ---------------------------------------------------------------------------
# 0. OPTIONS
# ---------------------------------------------------------------------------
DRIVE_ROOT   = "/content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2"   # NEW root, old one untouched
TIME_BUDGET_HOURS = 11.0        # stop cleanly before Colab's 12 h limit
NUM_THREADS  = 4                # extraction threads
FLUSH_EVERY  = 200              # write the partial manifest every N items
QUICK_TEST   = False            # True -> 6 sequences per split/label, ~5 min

# which corpora to build, in this order
BUILD = ["celebdf", "ff++", "wilddeepfake"]

# frames kept per video, per corpus (FF++ has ~5k videos, so it gets fewer)
FRAME_BUDGET = {
    "celebdf":      dict(train=20, test=32, stride=24),
    "wilddeepfake": dict(train=20, test=32, stride=24),
    "ff++":         dict(train=10, test=20, stride=30),
}

MIN_DETECT_RATE = 0.50          # abort if the smoke test finds fewer faces
ALLOW_CENTER_FALLBACK = False   # True only if you knowingly accept unaligned crops

MANUAL_DATA_ROOTS = {           # fill in if kagglehub cannot download
    # "celebdf": "/content/celeb-df-v2",
    # "ff++":    "/content/ff-c23",
}

# ---------------------------------------------------------------------------
# 0b. PIN OpenCV TO 4.x  (must happen before `import cv2`)
# ---------------------------------------------------------------------------
import sys, subprocess, importlib


def _pip(*a):
    subprocess.run([sys.executable, "-m", "pip", *a], check=False,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def _cv_major():
    try:
        from importlib.metadata import version
    except Exception:
        return None
    for p in ("opencv-python-headless", "opencv-python",
              "opencv-contrib-python-headless", "opencv-contrib-python"):
        try:
            return int(version(p).split(".")[0])
        except Exception:
            continue
    return None


_m = _cv_major()
if _m is None or _m >= 5:
    print("pinning OpenCV to 4.x (5.x removed readNetFromCaffe) ...")
    _pip("install", "-q", "--force-reinstall", "opencv-python-headless<5")

_live = sys.modules.get("cv2")
if _live is not None and int(getattr(_live, "__version__", "0").split(".")[0]) >= 5:
    print("=" * 92)
    print("RESTART THE RUNTIME, THEN RUN THIS CELL AGAIN")
    print("=" * 92)
    print("OpenCV 4.x has been installed but a 5.x build is already loaded.")
    print("    Runtime -> Restart session")
    print("=" * 92)
    raise SystemExit("restart required")

for mod, pkg in {"kagglehub": "kagglehub", "pandas": "pandas", "tqdm": "tqdm"}.items():
    try:
        importlib.import_module(mod)
    except Exception:
        print(f"installing {pkg} ...")
        _pip("install", "-q", pkg)

import os, json, time, math, hashlib, shutil, threading, urllib.request, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

_CASCADE = (getattr(cv2, "CascadeClassifier", None)
            or getattr(getattr(cv2, "objdetect", None), "CascadeClassifier", None))
_READ_CAFFE = getattr(cv2.dnn, "readNetFromCaffe", None)

T0 = time.time()


def elapsed_h():
    return (time.time() - T0) / 3600.0


def budget_left():
    return TIME_BUDGET_HOURS - elapsed_h()


# ---------------------------------------------------------------------------
# 1. DRIVE + PATHS
# ---------------------------------------------------------------------------
try:
    from google.colab import drive
    if not Path("/content/drive/NIT-PATNA-P2/MyDrive").is_dir():
        drive.mount("/content/drive")
    ROOT = Path(DRIVE_ROOT)
except Exception as e:
    print("Drive not available, using local storage:", e)
    print("!! this cache will be lost when the VM is recycled !!")
    ROOT = Path(DRIVE_ROOT if not DRIVE_ROOT.startswith("/content/drive")
                else "./fsttnet_v2")

FACE_DIR, MODEL_DIR, STATE = ROOT / "faces", ROOT / "models", ROOT / "state"
for d in (ROOT, FACE_DIR, MODEL_DIR, STATE):
    d.mkdir(parents=True, exist_ok=True)

print("=" * 92)
print("CELL 0 (v4) :: DATA REBUILD")
print("=" * 92)
print(f"root         : {ROOT}")
print(f"opencv       : {cv2.__version__}")
print(f"threads      : {NUM_THREADS}")
print(f"time budget  : {TIME_BUDGET_HOURS} h")
print(f"corpora      : {BUILD}")
print("=" * 92)


# ---------------------------------------------------------------------------
# 2. FACE DETECTOR (OpenCV DNN SSD -> Haar -> centre), one net per thread
# ---------------------------------------------------------------------------
_DNN = {
    "deploy.prototxt":
        "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/dnn/"
        "face_detector/deploy.prototxt",
    "res10_300x300_ssd_iter_140000.caffemodel":
        "https://raw.githubusercontent.com/opencv/opencv_3rdparty/"
        "dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
}


def ensure_model():
    out = {}
    for n, u in _DNN.items():
        p = MODEL_DIR / n
        if not p.exists() or p.stat().st_size < 10_000:
            print(f"  downloading {n} ...")
            urllib.request.urlretrieve(u, str(p))
        out[n] = p
    return out


_TL = threading.local()
IMG_SIZE, MARGIN, CONF = 224, 0.30, 0.60
BACKEND = "center"
_HAAR_PATH = None

try:
    if _READ_CAFFE is None:
        raise RuntimeError("cv2.dnn.readNetFromCaffe missing")
    _F = ensure_model()
    _READ_CAFFE(str(_F["deploy.prototxt"]),
                str(_F["res10_300x300_ssd_iter_140000.caffemodel"]))
    BACKEND = "dnn-ssd"
except Exception as e:
    print("  DNN detector unavailable:", e)
    try:
        d = getattr(getattr(cv2, "data", None), "haarcascades", None)
        if d and _CASCADE is not None:
            p = d + "haarcascade_frontalface_default.xml"
            if not _CASCADE(p).empty():
                _HAAR_PATH, BACKEND = p, "haar"
    except Exception as e2:
        print("  Haar unavailable:", e2)

print("face detector backend:", BACKEND)
if BACKEND == "center" and not ALLOW_CENTER_FALLBACK:
    print("=" * 92)
    print("NO FACE DETECTOR - STOPPING")
    print("=" * 92)
    print("Every crop would be a centre square, which is exactly the problem this")
    print("cell exists to fix. Check network access to raw.githubusercontent.com,")
    print("or set ALLOW_CENTER_FALLBACK = True if you accept unaligned crops.")
    print("=" * 92)
    raise RuntimeError("no usable face detector")


def _net():
    if BACKEND != "dnn-ssd":
        return None
    n = getattr(_TL, "net", None)
    if n is None:
        f = {k: MODEL_DIR / k for k in _DNN}
        n = _READ_CAFFE(str(f["deploy.prototxt"]),
                        str(f["res10_300x300_ssd_iter_140000.caffemodel"]))
        _TL.net = n
    return n


def _haar():
    if BACKEND != "haar":
        return None
    h = getattr(_TL, "haar", None)
    if h is None:
        h = _CASCADE(_HAAR_PATH)
        _TL.haar = h
    return h


def detect(rgb):
    H, W = rgb.shape[:2]
    if BACKEND == "dnn-ssd":
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        blob = cv2.dnn.blobFromImage(cv2.resize(bgr, (300, 300)), 1.0,
                                     (300, 300), (104.0, 177.0, 123.0))
        net = _net()
        net.setInput(blob)
        det = net.forward()
        best, bc = None, 0.0
        for i in range(det.shape[2]):
            c = float(det[0, 0, i, 2])
            if c < CONF or c <= bc:
                continue
            x1, y1, x2, y2 = det[0, 0, i, 3:7] * np.array([W, H, W, H])
            if x2 - x1 < 24 or y2 - y1 < 24:
                continue
            best, bc = [x1, y1, x2, y2], c
        if best is not None:
            return best
    if BACKEND == "haar":
        g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        f = _haar().detectMultiScale(g, 1.15, 5, minSize=(40, 40))
        if len(f):
            x, y, w, h = max(f, key=lambda b: b[2] * b[3])
            return [x, y, x + w, y + h]
    return None


STATS = {"calls": 0, "hits": 0}
_LOCK = threading.Lock()


def crop_face(rgb):
    H, W = rgb.shape[:2]
    box = detect(rgb)
    with _LOCK:
        STATS["calls"] += 1
        STATS["hits"] += box is not None
    if box is None:
        s = min(H, W)
        box = [(W - s) / 2, (H - s) / 2, (W + s) / 2, (H + s) / 2]
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    cx, cy = x1 + w / 2, y1 + h / 2
    s = max(w, h) * (1 + MARGIN)
    a, b = int(max(0, cx - s / 2)), int(max(0, cy - s / 2))
    c, d = int(min(W, cx + s / 2)), int(min(H, cy + s / 2))
    crop = rgb[b:d, a:c]
    if crop.size == 0:
        crop = rgb
    return cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)


# ---------------------------------------------------------------------------
# 3. DOWNLOAD
# ---------------------------------------------------------------------------
SLUGS = {
    "celebdf": ["reubensuju/celeb-df-v2"],
    "wilddeepfake": ["maysuni/wild-deepfake", "doosexe/voxear-faces-wilddeepfake"],
    "ff++": ["xdxd003/ff-c23", "greatgamedota/faceforensics"],
}


def fetch(key):
    if key in MANUAL_DATA_ROOTS and Path(MANUAL_DATA_ROOTS[key]).exists():
        print(f"  {key}: using MANUAL_DATA_ROOTS")
        return MANUAL_DATA_ROOTS[key]
    import kagglehub
    for s in SLUGS.get(key, []):
        try:
            p = kagglehub.dataset_download(s)
            print(f"  {key}: downloaded {s}")
            return p
        except Exception as e:
            print(f"  {key}: {s} failed -> {e}")
    return None


# ---------------------------------------------------------------------------
# 4. INDEXING
# ---------------------------------------------------------------------------
VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv"}
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def label_of(rel):
    s = rel.lower().replace("\\", "/")
    if "celeb-synthesis" in s:
        return 1
    if "celeb-real" in s or "youtube-real" in s:
        return 0
    if "manipulated_sequences" in s or "manipulated" in s:
        return 1
    if "original_sequences" in s or "original" in s:
        return 0
    fk = any(k in s for k in ["synthesis", "fake", "deepfake", "forged", "/1/"])
    rk = any(k in s for k in ["real", "pristine", "genuine", "/0/"])
    if fk and not rk:
        return 1
    if rk and not fk:
        return 0
    if fk and rk:
        for p in reversed([p for p in s.split("/") if p]):
            if "fake" in p and "real" not in p:
                return 1
            if "real" in p and "fake" not in p:
                return 0
    return None


def identity_of(rel):
    stem = Path(rel).stem
    if stem.lower().startswith("id"):
        return stem.split("_")[0]
    head = stem.split("_")[0]
    return head if head.isdigit() else None      # FF++ 000_003.mp4 -> 000


def index_corpus(root, key):
    root = Path(root)
    rows = []
    for dp, _, fns in os.walk(root):
        for fn in fns:
            ext = Path(fn).suffix.lower()
            if ext not in VIDEO_EXT and ext not in IMAGE_EXT:
                continue
            full = Path(dp) / fn
            rel = str(full.relative_to(root))
            lab = label_of(rel)
            if lab is None:
                continue
            kind = "video" if ext in VIDEO_EXT else "image"
            if kind == "video":
                vid = str(Path(rel).with_suffix(""))
            else:
                stem = Path(rel).stem
                trimmed = stem.rstrip("0123456789").rstrip("_-") or stem
                vid = str(Path(rel).parent / trimmed)
            rows.append({"dataset": key, "path": str(full), "rel": rel, "kind": kind,
                         "label": lab, "video_id": f"{key}/{vid}",
                         "identity": identity_of(rel)})
    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise RuntimeError(f"no labelled media under {root}")
    print(f"  {key}: {len(df):,} items | {df.kind.value_counts().to_dict()} | "
          f"real={int((df.label==0).sum()):,} fake={int((df.label==1).sum()):,} | "
          f"sequences={df.video_id.nunique():,}")
    return df


def make_splits(df, key, seed=42, val_frac=0.12, test_frac=0.20):
    """Identity-disjoint. Celeb-DF uses the official test list when present."""
    df = df.copy()
    if key == "celebdf":
        cand = list(Path(df.path.iloc[0]).parents)
        lst = None
        for p in cand:
            g = list(p.rglob("List_of_testing_videos.txt"))
            if g:
                lst = g[0]
                break
        if lst is not None:
            names = set()
            for line in lst.read_text().splitlines():
                line = line.strip()
                if line:
                    names.add(Path(line.split()[-1].replace("\\", "/")).name)
            is_test = df.rel.map(lambda r: Path(r).name in names)
            df["split"] = np.where(is_test, "test", "train")
            print(f"  {key}: official test list applied "
                  f"({int(is_test.sum()):,} test items)")
        else:
            df["split"] = None
    else:
        df["split"] = None

    if df["split"].isna().all():
        grp = df["identity"].fillna(df["video_id"])
        u = sorted(grp.unique())
        rng = np.random.RandomState(seed); rng.shuffle(u)
        n_test = max(1, int(test_frac * len(u)))
        test_g = set(u[:n_test])
        df["split"] = np.where(grp.isin(test_g), "test", "train")

    tr = df[df.split == "train"]
    grp = tr["identity"].fillna(tr["video_id"])
    u = sorted(grp.unique())
    rng = np.random.RandomState(seed + 1); rng.shuffle(u)
    val_g = set(u[:max(1, int(val_frac * len(u)))])
    df.loc[df.split == "train", "split"] = np.where(grp.isin(val_g), "val", "train")
    print(f"  {key}: splits {df.split.value_counts().to_dict()}")
    return df


# ---------------------------------------------------------------------------
# 5. SMOKE TEST
# ---------------------------------------------------------------------------
def smoke(df, n=25):
    sub = df.sample(min(n, len(df)), random_state=0)
    hit = tot = 0
    for r in sub.itertuples():
        try:
            if r.kind == "image":
                im = cv2.imread(r.path)
                if im is None:
                    continue
                rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
            else:
                cap = cv2.VideoCapture(r.path)
                nf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, nf // 2))
                ok, fr = cap.read()
                cap.release()
                if not ok:
                    continue
                rgb = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
            tot += 1
            hit += detect(rgb) is not None
        except Exception:
            continue
    return hit, tot


# ---------------------------------------------------------------------------
# 6. EXTRACTION (threaded, resumable, budgeted)
# ---------------------------------------------------------------------------
def frame_indices(n, split, bud):
    if split == "train":
        idx = list(range(0, n, max(1, bud["stride"])))
        cap = bud["train"]
    else:
        cap = bud["test"]
        idx = np.linspace(0, max(0, n - 1), num=min(cap, max(1, n))).astype(int).tolist()
    if len(idx) > cap:
        idx = [idx[i] for i in np.linspace(0, len(idx) - 1, cap).astype(int)]
    return sorted(set(int(i) for i in idx)) or [0]


def do_item(r, out_root, bud):
    rows = []
    try:
        sub = out_root / r.split / str(r.label)
        sub.mkdir(parents=True, exist_ok=True)
        base = hashlib.md5(r.rel.encode()).hexdigest()[:12]
        if r.kind == "image":
            op = sub / f"{base}_00000.jpg"
            if not op.exists():
                im = cv2.imread(r.path)
                if im is None:
                    return rows
                face = crop_face(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
                cv2.imwrite(str(op), cv2.cvtColor(face, cv2.COLOR_RGB2BGR),
                            [int(cv2.IMWRITE_JPEG_QUALITY), 95])
            rows.append({"dataset": r.dataset, "face": str(op), "label": r.label,
                         "video_id": r.video_id, "split": r.split, "frame": 0,
                         "identity": r.identity, "src": r.path})
        else:
            cap = cv2.VideoCapture(r.path)
            n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if n <= 0:
                cap.release()
                return rows
            for fi in frame_indices(n, r.split, bud):
                op = sub / f"{base}_{fi:05d}.jpg"
                if not op.exists():
                    cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
                    ok, fr = cap.read()
                    if not ok:
                        continue
                    face = crop_face(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
                    cv2.imwrite(str(op), cv2.cvtColor(face, cv2.COLOR_RGB2BGR),
                                [int(cv2.IMWRITE_JPEG_QUALITY), 95])
                rows.append({"dataset": r.dataset, "face": str(op), "label": r.label,
                             "video_id": r.video_id, "split": r.split, "frame": fi,
                             "identity": r.identity, "src": r.path})
            cap.release()
    except Exception:
        pass
    return rows


def extract(df, key, bud):
    out_root = FACE_DIR / key
    out_root.mkdir(parents=True, exist_ok=True)
    man_p = out_root / "manifest.csv"
    part_p = STATE / f"{key}_partial.csv"
    done_p = STATE / f"{key}_done.json"

    if man_p.exists():
        m = pd.read_csv(man_p)
        if len(m) and m.face.head(20).map(lambda p: Path(str(p)).exists()).any():
            print(f"  {key}: complete manifest already present ({len(m):,} crops)")
            return m, True

    done = set()
    if done_p.exists():
        try:
            done = set(json.loads(done_p.read_text()))
        except Exception:
            done = set()
    rows = []
    if part_p.exists():
        try:
            rows = pd.read_csv(part_p).to_dict("records")
            print(f"  {key}: resuming with {len(rows):,} crops already recorded")
        except Exception:
            rows = []

    work = df[~df.rel.isin(done)].reset_index(drop=True)
    if QUICK_TEST:
        keep = []
        for _, g in work.groupby(["split", "label"], sort=False):
            keep += list(g.video_id.unique()[:6])
        work = work[work.video_id.isin(keep)].reset_index(drop=True)

    print(f"  {key}: {len(work):,} items to process "
          f"({len(done):,} already done), budget left {budget_left():.2f} h")

    if len(work) == 0:
        m = pd.DataFrame(rows)
        if len(m):
            m.to_csv(man_p, index=False)
        return m, True

    STATS["calls"] = STATS["hits"] = 0
    finished, stopped = 0, False
    with ThreadPoolExecutor(max_workers=NUM_THREADS) as ex:
        futs = {ex.submit(do_item, r, out_root, bud): r for r in work.itertuples()}
        pbar = tqdm(as_completed(futs), total=len(futs), desc=f"faces[{key}]")
        for fu in pbar:
            r = futs[fu]
            try:
                rows += fu.result()
            except Exception:
                pass
            done.add(r.rel)
            finished += 1
            if finished % FLUSH_EVERY == 0:
                pd.DataFrame(rows).to_csv(part_p, index=False)
                done_p.write_text(json.dumps(sorted(done)))
                rate = STATS["hits"] / max(STATS["calls"], 1)
                pbar.set_postfix(crops=len(rows), det=f"{rate:.0%}",
                                 h=f"{elapsed_h():.1f}")
            if budget_left() <= 0.25:
                stopped = True
                for f2 in futs:
                    f2.cancel()
                break

    pd.DataFrame(rows).to_csv(part_p, index=False)
    done_p.write_text(json.dumps(sorted(done)))
    m = pd.DataFrame(rows)

    if stopped:
        print(f"  {key}: TIME BUDGET REACHED at {len(rows):,} crops - progress saved")
        return m, False

    if len(m):
        m.to_csv(man_p, index=False)
        print(f"  {key}: complete, {len(m):,} crops -> {man_p}")
    return m, True


# ---------------------------------------------------------------------------
# 7. RUN
# ---------------------------------------------------------------------------
FRAMES, COMPLETE = {}, {}
for key in BUILD:
    print("\n" + "-" * 92)
    print(f"CORPUS: {key}   (elapsed {elapsed_h():.2f} h, left {budget_left():.2f} h)")
    print("-" * 92)

    if budget_left() <= 0.5:
        print("  time budget exhausted - rerun this cell to continue")
        break

    man_p = FACE_DIR / key / "manifest.csv"
    if man_p.exists():
        try:
            m = pd.read_csv(man_p)
            if len(m) and m.face.head(20).map(lambda p: Path(str(p)).exists()).any():
                FRAMES[key], COMPLETE[key] = m, True
                print(f"  already built: {len(m):,} crops")
                continue
        except Exception:
            pass

    root = fetch(key)
    if root is None:
        print(f"  {key}: no data source, skipped")
        continue

    try:
        idx = index_corpus(root, key)
    except Exception as e:
        print(f"  {key}: indexing failed -> {e}")
        continue

    idx = make_splits(idx, key)

    hit, tot = smoke(idx)
    rate = hit / max(tot, 1)
    print(f"  {key}: detector smoke test {hit}/{tot} = {rate:.0%}")
    if tot and rate < MIN_DETECT_RATE and not ALLOW_CENTER_FALLBACK:
        print(f"  {key}: detection rate below {MIN_DETECT_RATE:.0%} - SKIPPED.")
        print("       Lower CONF, or set ALLOW_CENTER_FALLBACK = True knowingly.")
        continue

    m, ok = extract(idx, key, FRAME_BUDGET.get(key, FRAME_BUDGET["celebdf"]))
    FRAMES[key], COMPLETE[key] = m, ok
    if STATS["calls"]:
        print(f"  {key}: detection rate {STATS['hits']/STATS['calls']:.1%} "
              f"({STATS['hits']:,}/{STATS['calls']:,} crops from a detected box)")
    if not ok:
        break


# ---------------------------------------------------------------------------
# 8. REPORT
# ---------------------------------------------------------------------------
print("\n" + "=" * 92)
print(f"DATA REBUILD STATUS   (elapsed {elapsed_h():.2f} h)")
print("=" * 92)
for k in BUILD:
    if k not in FRAMES:
        print(f"{k:<14} not built yet")
        continue
    m = FRAMES[k]
    tag = "COMPLETE" if COMPLETE.get(k) else "PARTIAL"
    print(f"{k:<14} {tag:<9} {len(m):,} crops | "
          f"splits={m.split.value_counts().to_dict() if len(m) else {}} | "
          f"labels={m.label.value_counts().to_dict() if len(m) else {}} | "
          f"sequences={m.video_id.nunique() if len(m) else 0:,}")

pending = [k for k in BUILD if not COMPLETE.get(k)]
print()
print(f"detector: {BACKEND}")
print(f"cache   : {FACE_DIR}")
if pending:
    print()
    print("-" * 92)
    print("NOT FINISHED - RUN THIS SAME CELL AGAIN")
    print("-" * 92)
    print(f"still to do: {pending}")
    print("Everything already extracted is on Drive and will be skipped next time.")
    print("If the session died, just reconnect and rerun; nothing is lost.")
else:
    print()
    print("ALL CORPORA BUILT.")
    print("Next: point the SINGLE CELL's Config at this root -")
    print(f'    ROOT: str = "{ROOT}"')
    print("then run SINGLE CELL -> CELL X -> CELL M as before.")
print("=" * 92)


pinning OpenCV to 4.x (5.x removed readNetFromCaffe) ...
Mounted at /content/drive
CELL 0 (v4) :: DATA REBUILD
root         : /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2
opencv       : 4.14.0
threads      : 4
time budget  : 11.0 h
corpora      : ['celebdf', 'ff++', 'wilddeepfake']
  downloading deploy.prototxt ...
  downloading res10_300x300_ssd_iter_140000.caffemodel ...
face detector backend: dnn-ssd

--------------------------------------------------------------------------------------------
CORPUS: celebdf   (elapsed 0.01 h, left 10.99 h)
--------------------------------------------------------------------------------------------
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
  celebdf: downloaded reubensuju/celeb-df-v2
  celebdf: 6,529 items | {'video': 6529} | real=890 fake=5,639 | sequences=6,529
  celebdf: official test list applied (518 test items)
  celebdf: splits {'train': 5520, 'test': 518, 'val': 491}
  celebdf: detector smoke test 25/25 = 100%
  cel

faces[celebdf]:   0%|          | 0/6529 [00:00<?, ?it/s]

  celebdf: complete, 121,175 crops -> /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2/faces/celebdf/manifest.csv
  celebdf: detection rate 98.8% (119,671/121,175 crops from a detected box)

--------------------------------------------------------------------------------------------
CORPUS: ff++   (elapsed 0.87 h, left 10.13 h)
--------------------------------------------------------------------------------------------


100%|██████████| 16.7G/16.7G [01:41<00:00, 177MB/s]

Extracting files...


...

100%|██████████| 10.1G/10.1G [01:11<00:00, 153MB/s]

Extracting files...


  wilddeepfake: downloaded maysuni/wild-deepfake
  wilddeepfake: 189,713 items | {'image': 189713} | real=97,055 fake=92,658 | sequences=1,716
  wilddeepfake: splits {'train': 137435, 'test': 35984, 'val': 16294}
  wilddeepfake: detector smoke test 24/25 = 96%
  wilddeepfake: 189,713 items to process (0 already done), budget left 8.46 h


faces[wilddeepfake]:   0%|          | 0/189713 [00:00<?, ?it/s]

  wilddeepfake: complete, 189,713 crops -> /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2/faces/wilddeepfake/manifest.csv
  wilddeepfake: detection rate 96.6% (183,326/189,713 crops from a detected box)

DATA REBUILD STATUS   (elapsed 3.92 h)
celebdf        COMPLETE  121,175 crops | splits={'train': 88887, 'test': 16576, 'val': 15712} | labels={1: 102437, 0: 18738} | sequences=6,529
ff++           COMPLETE  38,613 crops | splits={'train': 21033, 'test': 11980, 'val': 5600} | labels={1: 25703, 0: 12910} | sequences=3,000
wilddeepfake   COMPLETE  189,713 crops | splits={'train': 137435, 'test': 35984, 'val': 16294} | labels={0: 97055, 1: 92658} | sequences=1,716

detector: dnn-ssd
cache   : /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2/faces

ALL CORPORA BUILT.
Next: point the SINGLE CELL's Config at this root -
    ROOT: str = "/content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2"
then run SINGLE CELL -> CELL X -> CELL M as before.


---
## 2 · Prerequisites + training

Defines `CFG`, `FRAMES`, `FSTTNet`, `FSTTLoss` and the rest, then trains **STT-Repro**
(controlled base-paper configuration) and **FSTT-Net** (proposed).

Rerun after every runtime restart — cells 3–6 depend on what this creates. Resumable via
`/content/drive/MyDrive/fsttnet_v2/checkpoints/last2_v6`; set `FORCE_RESTART = False` to continue rather than
start over.

In [ ]:
!pip install -q --force-reinstall --no-cache-dir numpy==2.0.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 166.6 MB/s eta 0:00:00


In [ ]:
# ############################################################################
# FSTT-Net :: SINGLE CELL  (prerequisites + retrain STT-Repro and FSTT-Net)
# ----------------------------------------------------------------------------
# Paste this as ONE cell. Do not split it. Everything that used to be
# "Cell A" and "PART 2" is inside, so the ordering errors cannot happen.
#
# After every runtime restart: run THIS cell, then CELL C (results pack).
# ############################################################################

# ---- 0. Drive + numpy/torch bridge preflight -------------------------------
import sys, os, subprocess

try:
    from google.colab import drive
    from pathlib import Path as _P
    if not _P("/content/drive/MyDrive/NIT-PATNA-P2").is_dir():
        drive.mount("/content/drive")
    print("Drive mounted")
except Exception as _e:
    print("Drive not mounted (fine if CFG.ROOT is local):", _e)

try:
    import numpy as _np, torch as _t
    _x = _t.from_numpy(_np.zeros(4, dtype=_np.float32)); _ = _x.numpy().sum()
    print(f"numpy {_np.__version__} <-> torch {_t.__version__} bridge OK")
except Exception as _e:
    print("=" * 92)
    print("NUMPY <-> TORCH BRIDGE IS BROKEN:", _e)
    print("=" * 92)
    print("Run CELL N (numpy fix) first, restart when it asks, then run this cell again.")
    print("=" * 92)
    raise

try:
    import timm  # noqa
except Exception as _e:
    print("timm/torchvision broken:", _e, "-> run CELL R2 first")
    raise


# ==================== PREREQUISITES (notebook cells 3, 13, 19-22, 26, 29) ====================
import os, sys, json, math, time, glob, random, shutil, warnings, hashlib, itertools
from pathlib import Path
from collections import defaultdict, OrderedDict
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import gridspec
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.metrics import (roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
                             confusion_matrix, classification_report, accuracy_score, f1_score,
                             precision_score, recall_score, balanced_accuracy_score, det_curve,
                             brier_score_loss)
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from scipy import stats
from scipy.fftpack import dct

warnings.filterwarnings("ignore")

# ------------------------------- global configuration -------------------------------
@dataclass
class Config:
    # ---- run control -------------------------------------------------------------
    QUICK_TEST: bool = False        # True -> tiny subset, 2 epochs. DEBUG ONLY, never report.
    SEED: int = 42
    SEEDS: tuple = (42, 1337, 2024) # for the mean +/- std table (Sec. 12)
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS: int = 2
    AMP: bool = True

    # ---- data --------------------------------------------------------------------
    PRIMARY: str = "celebdf"        # dataset used for the main intra-dataset tables
    SECONDARY: str = "wilddeepfake" # second intra-dataset benchmark + cross-dataset source
    IMG_SIZE: int = 224
    TRAIN_FRAME_STRIDE: int = 24    # base paper: sample every 24 frames while training
    MAX_TRAIN_FRAMES_PER_VIDEO: int = 20
    MAX_TEST_FRAMES_PER_VIDEO: int = 32   # frames pooled for the video-level decision
    FACE_MARGIN: float = 0.30       # crop margin around the detected box
    CLIP_LEN: int = 4               # T frames per temporal clip (1 disables the temporal stream)

    # ---- optimisation ------------------------------------------------------------
    EPOCHS: int = 20
    BATCH_SIZE: int = 24
    LR: float = 2e-4
    WEIGHT_DECAY: float = 0.05
    WARMUP_EPOCHS: int = 2
    LABEL_SMOOTH: float = 0.05
    EMA_DECAY: float = 0.999
    EARLY_STOP_PATIENCE: int = 6
    GRAD_CLIP: float = 1.0

    # ---- FSTT-Net architecture ---------------------------------------------------
    EMBED_DIMS: tuple = (64, 128, 256)
    NUM_HEADS: tuple = (1, 2, 4)
    DEPTHS: tuple = (2, 2, 2)
    MLP_RATIO: float = 4.0
    KNN_K: int = 5                  # k in the DPC-KNN density estimate (Eq. 1-2)
    ALPHA: float = 0.2              # semantic control coefficient (base paper optimum)
    ADAPTIVE_ALPHA: bool = True     # >>> FSTT-Net contribution: alpha predicted per image
    CLUSTER_RATIO: float = 0.25     # tokens kept per iteration (base paper: 1/4)
    N_ITERS: int = 3                # semantic scoring -> clustering -> serial blocks iterations
    USE_FREQ_STREAM: bool = True    # >>> contribution: DCT/SRM frequency tokens
    USE_TEMPORAL: bool = True       # >>> contribution: temporal token propagation
    USE_SEMANTIC: bool = True       # semantic-guided scoring (ablated in Sec. 10)
    PRETRAINED_STEM: bool = True    # ImageNet-initialised conv stem (stride 4)
    DROP_PATH: float = 0.1

    # ---- losses ------------------------------------------------------------------
    W_CE: float = 1.0
    W_FOCAL: float = 0.5
    W_CENTER: float = 0.05          # single-center loss on real class
    W_CONSIST: float = 0.10         # spatial <-> frequency consistency
    W_DIVERSE: float = 0.01         # cluster diversity regulariser
    FOCAL_GAMMA: float = 2.0

    # ---- evaluation --------------------------------------------------------------
    VIDEO_AGG: str = "mean"         # mean | median | topk_mean
    TOPK_FRAC: float = 0.5
    BOOTSTRAP_N: int = 1000

    # ---- paths -------------------------------------------------------------------
    ROOT: str = "/content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2"   # <-- matches the cell 1 cache

CFG = Config()

if CFG.QUICK_TEST:
    CFG.EPOCHS = 2
    CFG.MAX_TRAIN_FRAMES_PER_VIDEO = 3
    CFG.MAX_TEST_FRAMES_PER_VIDEO = 4
    CFG.SEEDS = (42,)
    print("!! QUICK_TEST is ON - results are for debugging only, do not put them in the paper !!")

ROOT     = Path(CFG.ROOT)
FACE_DIR = ROOT / "faces"       # cached face crops
FIG_DIR  = ROOT / "figures"
TAB_DIR  = ROOT / "tables"
CKPT_DIR = ROOT / "checkpoints"
RES_DIR  = ROOT / "results"
for d in [ROOT, FACE_DIR, FIG_DIR, TAB_DIR, CKPT_DIR, RES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(CFG.SEED)
DEVICE = torch.device(CFG.DEVICE)
print(f"device = {DEVICE} | torch {torch.__version__}")
if torch.cuda.is_available():
    print("gpu    =", torch.cuda.get_device_name(0))
print("root   =", ROOT.resolve())

# ============================================================================
# FRAMES :: rebuild the face-crop manifests from disk (no re-extraction)
# ============================================================================
def rebuild_frames(face_dir=None, verbose=True):
    face_dir = Path(face_dir) if face_dir is not None else FACE_DIR
    out = {}
    for mpath in sorted(face_dir.glob("*/manifest.csv")):
        key = mpath.parent.name
        try:
            df = pd.read_csv(mpath)
        except Exception as e:
            print(f"  {key}: manifest unreadable ({e})")
            continue
        if len(df) == 0 or "face" not in df.columns:
            continue
        probe = df.face.head(50).map(lambda p: Path(str(p)).exists())
        if not probe.any():
            print(f"  {key}: manifest has {len(df)} rows but NONE of the crops "
                  f"exist on disk any more -> re-extraction needed")
            continue
        out[key] = df
        if verbose:
            print(f"  {key}: {len(df):,} crops | "
                  f"splits={sorted(df.split.unique().tolist())} | "
                  f"labels={df.label.value_counts().to_dict()}")

    if out:
        return out

    # fallback: walk  FACE_DIR/<key>/<split>/<label>/*.jpg
    for keydir in sorted(p for p in face_dir.glob("*") if p.is_dir()):
        rows = []
        for img in sorted(keydir.glob("*/*/*.jpg")) + sorted(keydir.glob("*/*/*.png")):
            split, label = img.parent.parent.name, img.parent.name
            stem = img.stem
            vid, _, fr = stem.rpartition("_")
            rows.append({"dataset": keydir.name, "face": str(img),
                         "label": int(label) if label.isdigit() else 0,
                         "video_id": vid or stem, "split": split,
                         "frame": int(fr) if fr.isdigit() else 0,
                         "identity": vid or stem, "src": ""})
        if rows:
            df = pd.DataFrame(rows)
            out[keydir.name] = df
            df.to_csv(keydir / "manifest.csv", index=False)
            if verbose:
                print(f"  {keydir.name}: rebuilt {len(df):,} crops by walking the cache")
    return out


print("\nrebuilding FRAMES from", FACE_DIR)
FRAMES = rebuild_frames()

if not FRAMES:
    print()
    print("=" * 92)
    print("NO CACHED FACE CROPS FOUND")
    print("=" * 92)
    print(f"Looked in : {FACE_DIR}")
    print()
    print("Face crops are DATA - they cannot be redefined, only re-extracted.")
    print("Run the notebook's dataset cells once (Kaggle download -> indexing ->")
    print("face extraction), i.e. cells 6, 7 and 10. After that this cell will")
    print("find the cache and FRAMES will rebuild in a second.")
    print()
    print("To stop losing the cache on every VM restart, point CFG.ROOT at Drive:")
    print("    from google.colab import drive; drive.mount('/content/drive')")
    print("    CFG.ROOT = '/content/drive/MyDrive/NIT-PATNA-P2/fsttnet'")
    print("=" * 92)
    raise RuntimeError("FRAMES is empty - run the face-extraction cells once.")

for _k, _m in FRAMES.items():
    _sp = set(_m.split.unique())
    if not {"train", "val"} <= _sp:
        print(f"WARNING: FRAMES['{_k}'] has splits {sorted(_sp)} - "
              f"training needs both 'train' and 'val'.")




# ==================== [notebook cell 13] SEMANTIC PRIOR / SRM ====================
# ============================ SEMANTIC PRIOR GENERATOR ============================
REGION_NAMES = ["background", "skin", "eyes", "nose", "mouth"]
N_REGIONS = len(REGION_NAMES)

class SemanticPrior:
    """Returns an (N_REGIONS, H, W) soft one-hot facial region map in [0,1]."""
    def __init__(self, size=224, use_mediapipe=True):
        self.size = size
        self.mp = None
        if use_mediapipe:
            try:
                import mediapipe as mp
                self.mp = mp.solutions.face_mesh.FaceMesh(static_image_mode=True,
                                                          max_num_faces=1, refine_landmarks=True)
                print("semantic prior backend: mediapipe FaceMesh")
            except Exception as e:
                print("mediapipe unavailable -> canonical template prior")
        self._template = self._build_template(size)

    @staticmethod
    def _ellipse_mask(size, cx, cy, rx, ry, soft=0.06):
        yy, xx = np.mgrid[0:size, 0:size].astype(np.float32) / size
        d = ((xx - cx) / max(rx, 1e-6)) ** 2 + ((yy - cy) / max(ry, 1e-6)) ** 2
        return np.clip(1.0 - (d - 1.0) / soft, 0, 1).astype(np.float32)

    def _build_template(self, size):
        m = np.zeros((N_REGIONS, size, size), np.float32)
        m[1] = self._ellipse_mask(size, 0.50, 0.54, 0.34, 0.44, 0.20)        # skin oval
        eyes = np.maximum(self._ellipse_mask(size, 0.355, 0.42, 0.095, 0.055),
                          self._ellipse_mask(size, 0.645, 0.42, 0.095, 0.055))
        m[2] = eyes
        m[3] = self._ellipse_mask(size, 0.50, 0.545, 0.075, 0.115)           # nose
        m[4] = self._ellipse_mask(size, 0.50, 0.715, 0.135, 0.070)           # mouth
        m[1] = np.clip(m[1] - m[2] - m[3] - m[4], 0, 1)
        m[0] = np.clip(1.0 - m[1:].sum(0), 0, 1)
        return m / (m.sum(0, keepdims=True) + 1e-6)

    def _from_landmarks(self, rgb):
        res = self.mp.process(rgb)
        if not res.multi_face_landmarks:
            return None
        H, W = rgb.shape[:2]
        pts = np.array([[p.x * W, p.y * H] for p in res.multi_face_landmarks[0].landmark], np.float32)
        IDX = {
            "skin":  list(range(0, 468)),
            "eyes":  [33,133,160,159,158,157,173,246,161,144,145,153,154,155,
                      362,263,387,386,385,384,398,466,388,373,374,380,381,382],
            "nose":  [1,2,4,5,6,19,94,97,98,115,168,195,197,326,327,344,440],
            "mouth": [0,13,14,17,37,39,40,61,78,80,81,82,84,87,88,91,95,146,178,181,185,
                      191,267,269,270,291,308,310,311,312,314,317,318,321,324,375,402,405,409,415],
        }
        m = np.zeros((N_REGIONS, H, W), np.float32)
        for ri, name in [(1, "skin"), (2, "eyes"), (3, "nose"), (4, "mouth")]:
            p = pts[IDX[name]].astype(np.int32)
            if len(p) < 3: continue
            hull = cv2.convexHull(p)
            canvas = np.zeros((H, W), np.uint8)
            cv2.fillConvexPoly(canvas, hull, 1)
            m[ri] = cv2.GaussianBlur(canvas.astype(np.float32), (0, 0), 3.0)
        m[1] = np.clip(m[1] - m[2] - m[3] - m[4], 0, 1)
        m[0] = np.clip(1.0 - m[1:].sum(0), 0, 1)
        m = m / (m.sum(0, keepdims=True) + 1e-6)
        if H != self.size or W != self.size:
            m = np.stack([cv2.resize(x, (self.size, self.size)) for x in m])
        return m.astype(np.float32)

    def __call__(self, rgb: np.ndarray) -> np.ndarray:
        if self.mp is not None:
            try:
                m = self._from_landmarks(rgb)
                if m is not None: return m
            except Exception:
                pass
        return self._template.copy()

SEMPRIOR = SemanticPrior(CFG.IMG_SIZE)

# ============================ FREQUENCY DECOMPOSITION ============================
SRM_KERNELS = np.array([
    [[0,0,0,0,0],[0,-1,2,-1,0],[0,2,-4,2,0],[0,-1,2,-1,0],[0,0,0,0,0]],
    [[-1,2,-2,2,-1],[2,-6,8,-6,2],[-2,8,-12,8,-2],[2,-6,8,-6,2],[-1,2,-2,2,-1]],
    [[0,0,0,0,0],[0,0,0,0,0],[0,1,-2,1,0],[0,0,0,0,0],[0,0,0,0,0]],
], np.float32)
SRM_KERNELS = SRM_KERNELS / np.array([4.0, 12.0, 2.0], np.float32)[:, None, None]

def dct2(a):
    return dct(dct(a, axis=0, norm="ortho"), axis=1, norm="ortho")

def dct_band_maps(gray: np.ndarray, n_bands=3):
    """Split the block-DCT spectrum into low / mid / high radial bands (F3-Net style)."""
    h, w = gray.shape
    c = dct2(gray.astype(np.float32) / 255.0)
    yy, xx = np.mgrid[0:h, 0:w]
    radius = (yy / h + xx / w) / 2.0
    edges = np.linspace(0, 1, n_bands + 1)
    out = []
    for i in range(n_bands):
        mask = ((radius >= edges[i]) & (radius < edges[i + 1])).astype(np.float32)
        band = c * mask
        rec = dct(dct(band, axis=0, type=3, norm="ortho"), axis=1, type=3, norm="ortho")
        out.append(rec)
    return np.stack(out), c

def srm_residual(rgb: np.ndarray):
    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    return np.stack([cv2.filter2D(g, -1, k) for k in SRM_KERNELS])

# ==================== [notebook cell 19] TOKEN OPERATORS ====================
# ============================ CORE TOKEN OPERATORS (Eqs. 1-7) ============================
from timm.layers import DropPath, trunc_normal_

def index_points(points, idx):
    """Gather along dim 1. points (B,N,C) | idx (B,S) -> (B,S,C)"""
    B = points.shape[0]
    view_shape = [B] + [1] * (idx.dim() - 1)
    rep = [1] + list(idx.shape[1:])
    bidx = torch.arange(B, device=points.device, dtype=torch.long).view(view_shape).repeat(rep)
    return points[bidx, idx, :]

@torch.no_grad()
def dpc_knn_cluster(x, sem, k, cluster_num, alpha):
    """Density-peak clustering with k-NN density (base paper Eqs. 1-5).

    x     : (B,N,C) token features        -> rho_x  (Eq. 1)
    sem   : (B,N,R) semantic token values -> rho_y  (Eq. 2)
    alpha : float or (B,1) tensor         -> rho    (Eq. 3)
    returns idx_cluster (B,N), centre indices (B,M), score (B,N)
    """
    B, N, C = x.shape
    xf = x.float()
    dist = torch.cdist(xf, xf) / (C ** 0.5)                     # (B,N,N)
    kk = min(k, N)
    knn_d, _ = dist.topk(kk, dim=-1, largest=False)
    rho_x = torch.exp(-(knn_d ** 2).mean(-1))                   # Eq. 1
    if sem is not None:
        semf = sem.float()
        dsem = torch.cdist(semf, semf) / (sem.shape[-1] ** 0.5)
        knn_s, _ = dsem.topk(kk, dim=-1, largest=False)
        rho_y = torch.exp(-(knn_s ** 2).mean(-1))               # Eq. 2
        a = alpha if torch.is_tensor(alpha) else torch.full((B, 1), float(alpha), device=x.device)
        a = a.view(B, 1).float()
        rho = (1 - a) * rho_x + a * rho_y                       # Eq. 3
    else:
        rho = rho_x
    rho = rho + 1e-6 * torch.rand_like(rho)                     # break ties

    # delta_i = min distance to a token with strictly higher density (Eq. 4)
    higher = rho[:, None, :] > rho[:, :, None]
    dmax = dist.flatten(1).max(-1)[0][:, None, None]
    delta, _ = (dist * higher + dmax * (~higher)).min(dim=-1)
    score = rho * delta                                         # Eq. 5

    M = min(cluster_num, N)
    _, centers = torch.topk(score, k=M, dim=-1)                 # cluster centres
    d_to_center = index_points(dist, centers)                   # (B,M,N)
    idx_cluster = d_to_center.argmin(dim=1)                     # (B,N)
    bidx = torch.arange(B, device=x.device)[:, None]
    idx_cluster[bidx, centers] = torch.arange(M, device=x.device)[None, :].expand(B, -1)
    return idx_cluster, M, score, centers

def merge_tokens(x, idx_cluster, M, weight, extra=None):
    """Importance-weighted token merging (base paper Eq. 6)."""
    B, N, C = x.shape
    w = weight.exp()[..., None] if weight is not None else x.new_ones(B, N, 1)
    idx_batch = torch.arange(B, device=x.device)[:, None]
    idx = (idx_cluster + idx_batch * M).reshape(-1)             # flat cluster id

    all_w = x.new_zeros(B * M, 1).index_add_(0, idx, w.reshape(B * N, 1))
    all_w = all_w + 1e-6
    norm_w = w / all_w[idx].reshape(B, N, 1)

    merged = x.new_zeros(B * M, C).index_add_(0, idx, (x * norm_w).reshape(B * N, C))
    merged = merged.reshape(B, M, C)

    size = x.new_zeros(B * M, 1).index_add_(0, idx, x.new_ones(B * N, 1)).reshape(B, M, 1)
    out_extra = None
    if extra is not None:
        E = extra.shape[-1]
        out_extra = x.new_zeros(B * M, E).index_add_(0, idx, (extra * norm_w).reshape(B * N, E))
        out_extra = out_extra.reshape(B, M, E)
    return merged, out_extra, size

class SFScoringNet(nn.Module):
    """Semantic-Feature scoring network. Differentiable importance score with a learnable,
    optionally image-adaptive semantic control coefficient alpha (FSTT-Net contribution)."""
    def __init__(self, dim, sem_dim=N_REGIONS, adaptive=True, alpha_init=0.2):
        super().__init__()
        self.adaptive = adaptive
        self.feat_score = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim // 2),
                                        nn.GELU(), nn.Linear(dim // 2, 1))
        self.sem_score = nn.Sequential(nn.LayerNorm(sem_dim), nn.Linear(sem_dim, dim // 2),
                                       nn.GELU(), nn.Linear(dim // 2, 1))
        self.alpha_gate = nn.Sequential(nn.LayerNorm(dim + sem_dim), nn.Linear(dim + sem_dim, 32),
                                        nn.GELU(), nn.Linear(32, 1))
        self.alpha_bias = nn.Parameter(torch.tensor(math.log(alpha_init / (1 - alpha_init))))

    def forward(self, x, sem):
        s_x = self.feat_score(x).squeeze(-1)                    # (B,N)
        s_y = self.sem_score(sem).squeeze(-1)
        g = torch.cat([x.mean(1), sem.mean(1)], -1)
        if self.adaptive:
            alpha = torch.sigmoid(self.alpha_gate(g) + self.alpha_bias)     # (B,1)
        else:
            alpha = torch.sigmoid(self.alpha_bias).expand(x.shape[0], 1)
        s = (1 - alpha) * s_x + alpha * s_y
        return s, alpha

class SerialBlock(nn.Module):
    """Cluster-to-token cross attention with the token-score bias of Eq. (7), then self-attention
    and an MLP over the merged tokens."""
    def __init__(self, dim, heads, mlp_ratio=4.0, drop_path=0.0):
        super().__init__()
        self.h, self.scale = heads, (dim // heads) ** -0.5
        self.n1q, self.n1kv = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.q = nn.Linear(dim, dim); self.kv = nn.Linear(dim, 2 * dim); self.proj = nn.Linear(dim, dim)
        self.n2 = nn.LayerNorm(dim)
        self.sa = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n3 = nn.LayerNorm(dim)
        hid = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hid), nn.GELU(), nn.Linear(hid, dim))
        self.dp = DropPath(drop_path) if drop_path > 0 else nn.Identity()

    def forward(self, q_tokens, kv_tokens, score_bias):
        B, M, C = q_tokens.shape
        N = kv_tokens.shape[1]
        q = self.q(self.n1q(q_tokens)).reshape(B, M, self.h, C // self.h).transpose(1, 2)
        kv = self.kv(self.n1kv(kv_tokens)).reshape(B, N, 2, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        if score_bias is not None:                               # + S in Eq. (7)
            attn = attn + score_bias[:, None, None, :]
        attn = attn.softmax(-1)
        out = (attn @ v).transpose(1, 2).reshape(B, M, C)
        x = q_tokens + self.dp(self.proj(out))
        h = self.n2(x); x = x + self.dp(self.sa(h, h, h, need_weights=False)[0])
        x = x + self.dp(self.mlp(self.n3(x)))
        return x

class CrossDomainFusion(nn.Module):
    """Bidirectional spatial <-> frequency token attention with a learned gate (contribution 1)."""
    def __init__(self, dim, heads=4):
        super().__init__()
        self.a1 = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.a2 = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n1, self.n2, self.n3 = nn.LayerNorm(dim), nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.gate = nn.Sequential(nn.Linear(2 * dim, dim), nn.Sigmoid())
        self.proj = nn.Linear(2 * dim, dim)

    def forward(self, xs, xf):
        s2f = self.a1(self.n1(xs), self.n2(xf), self.n2(xf), need_weights=False)[0]
        f2s = self.a2(self.n2(xf), self.n1(xs), self.n1(xs), need_weights=False)[0]
        xs_e, xf_e = xs + s2f, xf + f2s
        g = self.gate(torch.cat([xs_e, xf_e], -1))
        fused = self.proj(torch.cat([g * xs_e, (1 - g) * xf_e], -1))
        return self.n3(fused), xs_e, xf_e

class TemporalTokenPropagation(nn.Module):
    """Temporal transformer over per-frame cluster summaries plus an explicit temporal-difference
    cue (contribution 3). Reduces to identity when T = 1."""
    def __init__(self, dim, heads=4, max_t=16):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, max_t, dim)); trunc_normal_(self.pos, std=0.02)
        self.n = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.mlp = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.Linear(dim, dim))
        self.diff_proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim))

    def forward(self, f):                                        # f: (B,T,C)
        B, T, C = f.shape
        if T == 1:
            return f[:, 0], f.new_zeros(B, C)
        x = f + self.pos[:, :T]
        h = self.n(x); x = x + self.attn(h, h, h, need_weights=False)[0]
        x = x + self.mlp(x)
        diff = self.diff_proj((f[:, 1:] - f[:, :-1]).abs().mean(1))
        return x.mean(1) + diff, diff

# ==================== [notebook cell 20] STEMS ====================
# ============================ STEMS ============================
class SpatialStem(nn.Module):
    """Stride-8 convolutional tokeniser. ImageNet-initialised when PRETRAINED_STEM is on."""
    def __init__(self, dim, pretrained=True):
        super().__init__()
        self.backbone = None
        if pretrained:
            try:
                import timm
                self.backbone = timm.create_model("efficientnet_b0", pretrained=True,
                                                  features_only=True, out_indices=(2,))
                ch = self.backbone.feature_info.channels()[-1]
            except Exception as e:
                print("pretrained stem unavailable ->", e)
        if self.backbone is None:
            ch = 64
            self.backbone = nn.Sequential(
                nn.Conv2d(3, 32, 3, 2, 1), nn.BatchNorm2d(32), nn.GELU(),
                nn.Conv2d(32, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.GELU(),
                nn.Conv2d(64, ch, 3, 2, 1), nn.BatchNorm2d(ch), nn.GELU())
        self.proj = nn.Sequential(nn.Conv2d(ch, dim, 1), nn.BatchNorm2d(dim))

    def forward(self, x):
        f = self.backbone(x)
        f = f[-1] if isinstance(f, (list, tuple)) else f
        f = self.proj(f)
        B, C, H, W = f.shape
        return f.flatten(2).transpose(1, 2), (H, W)              # (B,N,C)

class FrequencyStem(nn.Module):
    """Tokenises the 6-channel spectral input (3 DCT sub-bands + 3 SRM residuals)."""
    def __init__(self, dim, in_ch=6):
        super().__init__()
        self.band_w = nn.Parameter(torch.ones(in_ch))            # learnable band weighting
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, 2, 1), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, dim, 3, 2, 1), nn.BatchNorm2d(dim), nn.GELU(),
            nn.Conv2d(dim, dim, 3, 1, 1), nn.BatchNorm2d(dim))

    def forward(self, x):
        x = x * self.band_w.view(1, -1, 1, 1)
        f = self.net(x)
        B, C, H, W = f.shape
        return f.flatten(2).transpose(1, 2), (H, W)

# ==================== [notebook cell 21] FSTT-NET ====================
# ============================ FSTT-NET ============================
class FSTTNet(nn.Module):
    def __init__(self, cfg=CFG, num_classes=2):
        super().__init__()
        self.cfg = cfg
        dims, heads, depths = cfg.EMBED_DIMS, cfg.NUM_HEADS, cfg.DEPTHS
        self.use_freq, self.use_temporal, self.use_sem = cfg.USE_FREQ_STREAM, cfg.USE_TEMPORAL, cfg.USE_SEMANTIC

        self.spatial_stem = SpatialStem(dims[0], cfg.PRETRAINED_STEM)
        if self.use_freq:
            self.freq_stem = FrequencyStem(dims[0])
            self.fusion = CrossDomainFusion(dims[0], heads=4)

        dpr = torch.linspace(0, cfg.DROP_PATH, sum(depths)).tolist()
        self.scorers, self.blocks, self.downs = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        c = 0
        for i in range(cfg.N_ITERS):
            d_in, d_out = dims[min(i, len(dims) - 1)], dims[min(i + 1, len(dims) - 1)]
            self.scorers.append(SFScoringNet(d_in, N_REGIONS, cfg.ADAPTIVE_ALPHA, cfg.ALPHA))
            self.blocks.append(nn.ModuleList([
                SerialBlock(d_in, heads[min(i, len(heads) - 1)], cfg.MLP_RATIO, dpr[c + j])
                for j in range(depths[min(i, len(depths) - 1)])]))
            self.downs.append(nn.Linear(d_in, d_out) if d_in != d_out else nn.Identity())
            c += depths[min(i, len(depths) - 1)]

        d_final = dims[-1]
        self.norm = nn.LayerNorm(d_final)
        if self.use_temporal:
            self.temporal = TemporalTokenPropagation(d_final, heads=4)
        self.head = nn.Sequential(nn.LayerNorm(d_final), nn.Dropout(0.2), nn.Linear(d_final, num_classes))
        # auxiliary per-stream heads -> used by the cross-domain consistency loss and the ablation
        self.head_s = nn.Linear(dims[0], num_classes)
        self.head_f = nn.Linear(dims[0], num_classes) if self.use_freq else None
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def _pool_sem(self, sem, hw):
        return F.adaptive_avg_pool2d(sem, hw).flatten(2).transpose(1, 2)   # (B,N,R)

    def forward_tokens(self, img, freq, sem, want_maps=False):
        """One frame -> pooled feature. Returns aux dict for losses / visualisation."""
        aux = {"alphas": [], "cluster_maps": [], "scores": []}
        xs, hw = self.spatial_stem(img)
        semt = self._pool_sem(sem, hw) if self.use_sem else torch.zeros(
            xs.shape[0], xs.shape[1], N_REGIONS, device=xs.device, dtype=xs.dtype)
        if self.use_freq:
            xf, _ = self.freq_stem(freq)
            if xf.shape[1] != xs.shape[1]:
                xf = F.interpolate(xf.transpose(1, 2), size=xs.shape[1], mode="linear",
                                   align_corners=False).transpose(1, 2)
            x, xs_e, xf_e = self.fusion(xs, xf)
            aux["logit_s"] = self.head_s(xs_e.mean(1))
            aux["logit_f"] = self.head_f(xf_e.mean(1))
        else:
            x = xs
            aux["logit_s"] = self.head_s(xs.mean(1))
            aux["logit_f"] = None

        token_hw = hw
        for i in range(self.cfg.N_ITERS):
            score, alpha = self.scorers[i](x, semt)
            aux["alphas"].append(alpha)
            aux["scores"].append(score)
            M = max(8, int(x.shape[1] * self.cfg.CLUSTER_RATIO))
            idx_cluster, M, dpc_score, centers = dpc_knn_cluster(
                x, semt if self.use_sem else None, self.cfg.KNN_K, M, alpha.detach())
            if want_maps:
                aux["cluster_maps"].append((idx_cluster.detach().cpu(), token_hw, M))
            merged, sem_merged, size = merge_tokens(x, idx_cluster, M, score, extra=semt)
            for blk in self.blocks[i]:
                merged = blk(merged, x, score)
            x = self.downs[i](merged)
            semt = sem_merged if sem_merged is not None else semt
            token_hw = None
        feat = self.norm(x).mean(1)
        return feat, aux

    def forward(self, img, freq=None, sem=None, want_maps=False):
        """img (B,T,3,H,W) or (B,3,H,W)."""
        if img.dim() == 4:
            img = img.unsqueeze(1)
            if freq is not None: freq = freq.unsqueeze(1)
            if sem is not None: sem = sem.unsqueeze(1)
        B, T = img.shape[:2]
        img_f = img.flatten(0, 1)
        freq_f = freq.flatten(0, 1) if freq is not None else torch.zeros_like(img_f[:, :6])
        sem_f = sem.flatten(0, 1) if sem is not None else torch.zeros(
            img_f.shape[0], N_REGIONS, img_f.shape[-2], img_f.shape[-1], device=img_f.device)
        feat, aux = self.forward_tokens(img_f, freq_f, sem_f, want_maps)
        C = feat.shape[-1]
        feat_t = feat.view(B, T, C)
        if self.use_temporal and T > 1:
            pooled, _ = self.temporal(feat_t)
        else:
            pooled = feat_t.mean(1)
        logits = self.head(pooled)
        aux["feat"] = pooled
        aux["logit_s"] = aux["logit_s"].view(B, T, -1).mean(1)
        if aux.get("logit_f") is not None:
            aux["logit_f"] = aux["logit_f"].view(B, T, -1).mean(1)
        return logits, aux



# ==================== [notebook cell 22] LOSSES ====================
# ============================ LOSSES ============================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma = gamma
    def forward(self, logits, y):
        logp = F.log_softmax(logits, -1)
        p = logp.exp().gather(1, y[:, None]).squeeze(1)
        return (-(1 - p) ** self.gamma * logp.gather(1, y[:, None]).squeeze(1)).mean()

class SingleCenterLoss(nn.Module):
    """Compact real class, margin-separated fake class (Li et al., CVPR'21 style)."""
    def __init__(self, dim, m=0.3):
        super().__init__()
        self.center = nn.Parameter(torch.zeros(dim)); self.m = m
    def forward(self, feat, y):
        d = (feat - self.center[None]).norm(dim=1)
        d_real = d[y == 0].mean() if (y == 0).any() else feat.new_zeros(())
        d_fake = d[y == 1].mean() if (y == 1).any() else feat.new_zeros(())
        return d_real + F.relu(d_real - d_fake + self.m * math.sqrt(feat.shape[1]))

class FSTTLoss(nn.Module):
    def __init__(self, cfg=CFG, feat_dim=None):
        super().__init__()
        self.cfg = cfg
        self.ce = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTH)
        self.focal = FocalLoss(cfg.FOCAL_GAMMA)
        self.center = SingleCenterLoss(feat_dim or cfg.EMBED_DIMS[-1])
    def forward(self, logits, aux, y):
        parts = {}
        parts["ce"] = self.cfg.W_CE * self.ce(logits, y)
        parts["focal"] = self.cfg.W_FOCAL * self.focal(logits, y)
        parts["center"] = self.cfg.W_CENTER * self.center(aux["feat"], y)
        if aux.get("logit_f") is not None:
            ps = F.log_softmax(aux["logit_s"], -1)
            pf = F.softmax(aux["logit_f"], -1)
            parts["consist"] = self.cfg.W_CONSIST * (
                F.kl_div(ps, pf, reduction="batchmean")
                + self.ce(aux["logit_s"], y) + self.ce(aux["logit_f"], y)) / 3.0
        if aux.get("scores"):
            s = aux["scores"][0]
            p = F.softmax(s, -1)
            ent = -(p * (p + 1e-8).log()).sum(-1).mean()
            parts["diverse"] = -self.cfg.W_DIVERSE * ent     # maximise entropy -> balanced clusters
        total = sum(parts.values())
        return total, {k: float(v.detach()) for k, v in parts.items()}

# ==================== [notebook cell 26] FEATURE MAPS / AUGMENTATION ====================
# ============================ FAST PER-SAMPLE FEATURE MAPS ============================
IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], np.float32)

def fast_dct_bands(gray_f32):
    """3 radial DCT sub-bands via OpenCV DCT (much faster than scipy in the dataloader)."""
    h, w = gray_f32.shape
    c = cv2.dct(gray_f32)
    yy, xx = np.mgrid[0:h, 0:w]
    radius = (yy / h + xx / w) / 2.0
    out = np.empty((3, h, w), np.float32)
    edges = [0.0, 1 / 3, 2 / 3, 1.0]
    for i in range(3):
        mask = ((radius >= edges[i]) & (radius < edges[i + 1])).astype(np.float32)
        out[i] = cv2.idct(c * mask)
    return out

def make_freq_input(rgb_u8):
    g = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    bands = fast_dct_bands(g)
    res = np.stack([cv2.filter2D(g, -1, k) for k in SRM_KERNELS]).astype(np.float32)
    x = np.concatenate([bands, res], 0)
    return x / (np.abs(x).max(axis=(1, 2), keepdims=True) + 1e-6)

_SEM_TEMPLATE = SEMPRIOR._template.copy()

def fast_semantic_prior(rgb_u8):
    """Canonical facial-region template modulated by a cheap skin-likelihood map. ~0.5 ms/frame,
    which keeps the dataloader from becoming the bottleneck. MediaPipe landmarks are used for the
    qualitative figures (Fig. 4) where cost does not matter."""
    ycc = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2YCrCb)
    cr, cb = ycc[..., 1].astype(np.float32), ycc[..., 2].astype(np.float32)
    skin = np.exp(-(((cr - 150) / 22.0) ** 2 + ((cb - 115) / 22.0) ** 2) / 2.0).astype(np.float32)
    skin = cv2.GaussianBlur(skin, (0, 0), 3.0)
    if skin.shape[0] != _SEM_TEMPLATE.shape[1]:
        skin = cv2.resize(skin, _SEM_TEMPLATE.shape[1:][::-1])
    m = _SEM_TEMPLATE.copy()
    m[1:] = m[1:] * (0.5 + 0.5 * skin[None])
    m[0] = np.clip(1.0 - m[1:].sum(0), 0, 1)
    return (m / (m.sum(0, keepdims=True) + 1e-6)).astype(np.float32)

# ============================ AUGMENTATION ============================
def augment_clip(frames, rng):
    """Clip-consistent augmentation: the same geometric/photometric transform for every frame."""
    do_flip = rng.random() < 0.5
    do_jpeg = rng.random() < 0.35
    q = int(rng.integers(40, 95))
    do_blur = rng.random() < 0.20
    sig = float(rng.uniform(0.3, 1.3))
    br, ct, sat = (float(rng.uniform(0.85, 1.15)), float(rng.uniform(0.85, 1.15)),
                   float(rng.uniform(0.85, 1.15)))
    do_gray = rng.random() < 0.05
    do_noise = rng.random() < 0.15
    ns = float(rng.uniform(1, 6))
    sc = float(rng.uniform(0.90, 1.0))
    out = []
    for f in frames:
        if do_flip: f = f[:, ::-1]
        if sc < 1.0:
            s = int(f.shape[0] * sc)
            y0 = int((f.shape[0] - s) / 2); x0 = int((f.shape[1] - s) / 2)
            f = cv2.resize(f[y0:y0 + s, x0:x0 + s], (CFG.IMG_SIZE, CFG.IMG_SIZE))
        f = f.astype(np.float32)
        f = np.clip((f - 128) * ct + 128 * br, 0, 255)
        hsv = cv2.cvtColor(f.astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[..., 1] = np.clip(hsv[..., 1] * sat, 0, 255)
        f = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        if do_gray:
            f = cv2.cvtColor(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)
        if do_blur: f = cv2.GaussianBlur(f, (0, 0), sig)
        if do_noise:
            f = np.clip(f.astype(np.float32) + rng.normal(0, ns, f.shape), 0, 255).astype(np.uint8)
        if do_jpeg:
            ok, enc = cv2.imencode(".jpg", cv2.cvtColor(f, cv2.COLOR_RGB2BGR),
                                   [int(cv2.IMWRITE_JPEG_QUALITY), q])
            if ok: f = cv2.cvtColor(cv2.imdecode(enc, 1), cv2.COLOR_BGR2RGB)
        out.append(np.ascontiguousarray(f))
    return out

def corrupt_frame(f, kind, level):
    """Deterministic corruptions used for the robustness study (Sec. 11)."""
    if kind == "clean": return f
    if kind == "jpeg":
        ok, enc = cv2.imencode(".jpg", cv2.cvtColor(f, cv2.COLOR_RGB2BGR),
                               [int(cv2.IMWRITE_JPEG_QUALITY), int(level)])
        return cv2.cvtColor(cv2.imdecode(enc, 1), cv2.COLOR_BGR2RGB) if ok else f
    if kind == "blur":  return cv2.GaussianBlur(f, (0, 0), float(level))
    if kind == "noise":
        rng = np.random.RandomState(0)
        return np.clip(f.astype(np.float32) + rng.normal(0, level, f.shape), 0, 255).astype(np.uint8)
    if kind == "resize":
        s = max(16, int(f.shape[0] * level))
        return cv2.resize(cv2.resize(f, (s, s)), (f.shape[1], f.shape[0]))
    if kind == "bright":
        return np.clip(f.astype(np.float32) * level, 0, 255).astype(np.uint8)
    return f


# ==================== [notebook cell 29] LR SCHEDULE ====================
def cosine_lr(step, total, warmup, base_lr, min_lr=1e-6):
    if step < warmup:
        return base_lr * (step + 1) / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * p))


# ============================================================================
# READY
# ============================================================================
_CHECK = ["CFG", "set_seed", "cv2", "FRAMES", "N_REGIONS", "merge_tokens",
          "FSTTNet", "FSTTLoss", "IMNET_MEAN", "IMNET_STD", "augment_clip",
          "corrupt_frame", "make_freq_input", "fast_semantic_prior", "cosine_lr"]
_still = [n for n in _CHECK if n not in globals()]
print("\n" + "=" * 92)
if _still:
    print("STILL MISSING:", _still)
else:
    print(f"ALL {len(_CHECK)} PREREQUISITES ARE LIVE - now run CELL B (training).")
    print(f"device = {DEVICE} | datasets = {list(FRAMES.keys())} | "
          f"primary = {CFG.PRIMARY if CFG.PRIMARY in FRAMES else list(FRAMES)[0]}")
print("=" * 92)



# ============================================================================
# CELL B2 :: RETRAIN THE LAST TWO MODELS  (STT-Repro + FSTT-Net)
# ----------------------------------------------------------------------------
# Run AFTER Cell A ("ALL 15 PREREQUISITES ARE LIVE").
#
# WHY THIS DIFFERS FROM THE PREVIOUS TRAINING CELL
# Your logs show the two token-transformer models learning the training set
# (train acc 0.93-0.98) while validation AUC sits at 0.766 / 0.783, against
# Swin-T at 0.9663 on the SAME crops. That is not an unlucky run, it is a
# capacity + optimisation mismatch:
#
#   1. dims (64,128,256) put a 64-channel bottleneck directly after the
#      pretrained stem, throwing away most of what the backbone encodes,
#      while Swin-T carries 768 channels end to end.
#   2. the stem was cut at stride 8 (out_indices=2), i.e. low-level texture
#      features, and then FROZEN. Swin-T fine-tunes everything.
#   3. focal loss was stacked on top of a class-balanced sampler, so the
#      minority class was up-weighted twice, and the auxiliary terms kept the
#      total loss near 1.0 while Swin-T's was 0.13.
#
# This cell fixes all three: dims (128,256,512), a stride-16 semantic stem
# (EfficientNet-B3, out_indices=3) that is FINE-TUNED at 0.1x LR, and a loss
# schedule where cross-entropy actually dominates. It also reports VIDEO-level
# AUC every epoch, which is the number the base paper reports and the number
# you should put in the manuscript.
#
# Everything else is unchanged: same Eqs. 1-7, same contributions, same
# STT-Repro ablation (freq + temporal + adaptive alpha all OFF), safetensors
# resume, top-3 weight averaging.
#
# HONEST EXPECTATION: these changes address the diagnosed causes, but no one
# can promise a number in advance. Watch epoch 2-3: if video AUC is not
# already past ~0.90 the fix has not landed, and you should stop rather than
# burn six hours. Read the DIAGNOSIS block this cell prints at the end.
# ============================================================================

import os, gc, json, copy, sys, subprocess
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm.auto import tqdm


# ============================================================================
# 1. SETTINGS
# ============================================================================
TARGET_BASELINE_AUC = 0.9663      # your best baseline so far (Swin-T, frame level)

STT_EPOCHS  = 18
FSTT_EPOCHS = 24
EARLY_STOP  = 6
TOPK_AVG    = 3

# --- the three fixes -------------------------------------------------------
EMBED_DIMS   = (128, 256, 512)    # was (64, 128, 256)
NUM_HEADS    = (2, 4, 8)
DEPTHS       = (2, 2, 2)
STEM_MODEL   = "efficientnet_b3"  # semantic stem
STEM_INDEX   = 3                  # stride 16 -> 14x14 = 196 tokens at 224 px
FINETUNE_STEM = True              # stem trains at STEM_LR_MULT x the trunk LR
STEM_LR_MULT = 0.10

TRUNK_LR = 3.0e-4
WD       = 0.02

# loss weights (focal off: the balanced sampler already handles imbalance)
W_CE, W_FOCAL, W_CENTER, W_CONSIST, W_DIVERSE = 1.0, 0.0, 0.02, 0.05, 0.0

FORCE_RESTART = True              # True -> ignore old checkpoints of these two
WORKERS = min(4, max(2, (os.cpu_count() or 4) // 2))


# ============================================================================
# 2. CHECKPOINT BACKEND
# ============================================================================
try:
    from safetensors.torch import save_file as _st_save, load_file as _st_load
    _EXT = ".safetensors"
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "safetensors"], check=False)
    try:
        from safetensors.torch import save_file as _st_save, load_file as _st_load
        _EXT = ".safetensors"
    except Exception:
        _EXT = ".pt"
        def _st_save(state, path): torch.save(state, path)
        def _st_load(path, device="cpu"): return torch.load(path, map_location=device)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_OOM = getattr(torch.cuda, "OutOfMemoryError", RuntimeError)

PRIMARY = CFG.PRIMARY if CFG.PRIMARY in FRAMES else list(FRAMES)[0]
MANIFEST = FRAMES[PRIMARY]
TRAIN_MANIFEST = MANIFEST[MANIFEST.split == "train"]
VAL_MANIFEST = MANIFEST[MANIFEST.split == "val"]
if len(TRAIN_MANIFEST) == 0 or len(VAL_MANIFEST) == 0:
    raise RuntimeError(f"FRAMES['{PRIMARY}'] needs both a train and a val split.")

if "MODELS" not in globals() or not isinstance(MODELS, dict):
    MODELS = {}
if "HISTORIES" not in globals() or not isinstance(HISTORIES, dict):
    HISTORIES = {}
for _n, _m in list(MODELS.items()):
    try:
        MODELS[_n] = _m.cpu()
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True

SAVE_DIR = Path(getattr(CFG, "ROOT", "./fsttnet")) / "checkpoints" / "last2_v6"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 92)
print("CELL B2 :: RETRAIN STT-Repro + FSTT-Net")
print("=" * 92)
print(f"dataset   : {PRIMARY}  train={len(TRAIN_MANIFEST):,} val={len(VAL_MANIFEST):,} crops")
print(f"class mix : {MANIFEST.label.value_counts().to_dict()}  (0=real, 1=fake)")
print(f"device    : {DEVICE}")
print(f"dims      : {EMBED_DIMS}   stem: {STEM_MODEL} @ out_index {STEM_INDEX}"
      f"{' (fine-tuned)' if FINETUNE_STEM else ' (frozen)'}")
print(f"target    : beat frame AUC {TARGET_BASELINE_AUC}")
print(f"ckpt dir  : {SAVE_DIR}")
if torch.cuda.is_available():
    print(f"gpu       : {torch.cuda.get_device_name(0)}  "
          f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


# ============================================================================
# 3. AMP-SAFE merge_tokens  (index_add_ Half vs Float)
# ============================================================================
if not hasattr(torch, "_fstt_orig_merge_tokens"):
    torch._fstt_orig_merge_tokens = merge_tokens


def merge_tokens(x, idx_cluster, M, weight, extra=None):
    B, N, C = x.shape
    dt = x.dtype
    x32 = x.float()
    w32 = (torch.ones(B, N, 1, device=x.device, dtype=torch.float32)
           if weight is None else torch.exp(weight.float()).unsqueeze(-1))
    idx = (idx_cluster.long()
           + torch.arange(B, device=x.device, dtype=torch.long)[:, None] * M).reshape(-1)

    all_w = torch.zeros(B * M, 1, device=x.device, dtype=torch.float32)
    all_w.index_add_(0, idx, w32.reshape(B * N, 1))
    all_w.clamp_min_(1e-6)
    norm_w = w32 / all_w[idx].reshape(B, N, 1)

    merged = torch.zeros(B * M, C, device=x.device, dtype=torch.float32)
    merged.index_add_(0, idx, (x32 * norm_w).reshape(B * N, C))
    merged = merged.reshape(B, M, C)

    size = torch.zeros(B * M, 1, device=x.device, dtype=torch.float32)
    size.index_add_(0, idx, torch.ones(B * N, 1, device=x.device, dtype=torch.float32))
    size = size.reshape(B, M, 1)

    out_extra = None
    if extra is not None:
        e32 = extra.float()
        E = e32.shape[-1]
        out_extra = torch.zeros(B * M, E, device=x.device, dtype=torch.float32)
        out_extra.index_add_(0, idx, (e32 * norm_w).reshape(B * N, E))
        out_extra = out_extra.reshape(B, M, E).to(dt)

    return merged.to(dt), out_extra, size.to(dt)


# ============================================================================
# 4. DATA
# ============================================================================
class ClipDS(torch.utils.data.Dataset):
    def __init__(self, manifest, train, cfg, epoch=0):
        self.cfg, self.train, self.epoch = cfg, train, int(epoch)
        self.T = cfg.CLIP_LEN
        man = manifest.sort_values(["video_id", "frame"])
        self.clips = []
        for vid, g in man.groupby("video_id", sort=False):
            paths = g.face.tolist()
            if not paths:
                continue
            lab = int(g.label.iloc[0])
            if self.T == 1:
                chunks = [[p] for p in paths]
            else:
                chunks = [paths[i:i + self.T] for i in range(0, len(paths), self.T)]
                chunks = [c + [c[-1]] * (self.T - len(c)) for c in chunks]
            for c in chunks:
                self.clips.append((c, lab, vid))
        self.labels = np.asarray([c[1] for c in self.clips], dtype=np.int64)

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, i):
        paths, lab, vid = self.clips[i]
        rng = np.random.default_rng(i + self.epoch * 1000003 + (0 if self.train else 12345))
        frames = []
        for p in paths:
            im = cv2.imread(p)
            im = (np.zeros((self.cfg.IMG_SIZE, self.cfg.IMG_SIZE, 3), np.uint8)
                  if im is None else cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            if im.shape[0] != self.cfg.IMG_SIZE or im.shape[1] != self.cfg.IMG_SIZE:
                im = cv2.resize(im, (self.cfg.IMG_SIZE, self.cfg.IMG_SIZE))
            frames.append(im)
        if self.train:
            frames = augment_clip(frames, rng)
        imgs, freqs, sems = [], [], []
        for f in frames:
            x = f.astype(np.float32) / 255.0
            imgs.append(((x - IMNET_MEAN) / IMNET_STD).transpose(2, 0, 1))
            freqs.append(make_freq_input(f))
            sems.append(fast_semantic_prior(f))
        return {"img": torch.from_numpy(np.stack(imgs)).float(),
                "freq": torch.from_numpy(np.stack(freqs)).float(),
                "sem": torch.from_numpy(np.stack(sems)).float(),
                "y": torch.tensor(lab, dtype=torch.long),
                "vid": vid, "path": paths[0]}


_DS = {}


def train_ds(cfg, epoch):
    k = int(epoch)
    if k not in _DS:
        _DS[k] = ClipDS(TRAIN_MANIFEST, True, cfg, epoch=epoch)
    return _DS[k]


def train_loader(cfg, bs, epoch, workers=WORKERS):
    ds = train_ds(cfg, epoch)
    counts = np.bincount(ds.labels, minlength=2).astype(np.float64)
    w = (1.0 / np.maximum(counts, 1))[ds.labels]
    wt = torch.tensor(w.tolist(), dtype=torch.double)
    g = torch.Generator(); g.manual_seed(int(cfg.SEED) + int(epoch) * 7919)
    try:
        smp = WeightedRandomSampler(wt, len(wt), replacement=True, generator=g)
    except TypeError:
        torch.manual_seed(int(cfg.SEED) + int(epoch) * 7919)
        smp = WeightedRandomSampler(wt, len(wt), replacement=True)
    b = max(1, min(int(bs), len(ds)))
    kw = dict(batch_size=b, sampler=smp, num_workers=int(workers),
              pin_memory=torch.cuda.is_available(), drop_last=(len(ds) > b),
              persistent_workers=False)
    if workers > 0:
        kw["prefetch_factor"] = 2
    return DataLoader(ds, **kw)


_VAL_DS = ClipDS(VAL_MANIFEST, False, CFG, epoch=0)


def val_loader(bs, workers=WORKERS):
    kw = dict(batch_size=max(1, min(int(bs), len(_VAL_DS))), shuffle=False,
              num_workers=int(workers), pin_memory=torch.cuda.is_available(),
              drop_last=False, persistent_workers=False)
    if workers > 0:
        kw["prefetch_factor"] = 2
    return DataLoader(_VAL_DS, **kw)


# ============================================================================
# 5. STEMS
# ============================================================================
class DeepStem(nn.Module):
    """Stride-16 semantic stem. No 64-channel bottleneck, and trainable."""

    def __init__(self, dim, model_name=STEM_MODEL, out_index=STEM_INDEX, pretrained=True):
        super().__init__()
        import timm
        self.backbone = timm.create_model(model_name, pretrained=pretrained,
                                          features_only=True, out_indices=(out_index,))
        ch = self.backbone.feature_info.channels()[-1]
        self.proj = nn.Sequential(nn.Conv2d(ch, dim, 1, bias=False),
                                  nn.BatchNorm2d(dim), nn.GELU())

    def forward(self, x):
        f = self.backbone(x)
        f = f[-1] if isinstance(f, (list, tuple)) else f
        f = self.proj(f)
        B, C, H, W = f.shape
        return f.flatten(2).transpose(1, 2), (H, W)


class PoolTo(nn.Module):
    """Down-pool a stem's token grid so both streams have the same token count."""

    def __init__(self, base, pool):
        super().__init__()
        self.base, self.pool = base, int(pool)

    def forward(self, x):
        tok, hw = self.base(x)
        if hw is None:
            return tok, hw
        H, W = hw
        if H <= self.pool and W <= self.pool:
            return tok, hw
        B, N, C = tok.shape
        f = tok.transpose(1, 2).reshape(B, C, H, W)
        f = F.adaptive_avg_pool2d(f, (self.pool, self.pool))
        return f.flatten(2).transpose(1, 2), (self.pool, self.pool)


def make_cfg(name):
    c = copy.deepcopy(CFG)
    c.EMBED_DIMS, c.NUM_HEADS, c.DEPTHS = EMBED_DIMS, NUM_HEADS, DEPTHS
    c.AMP = True
    c.LABEL_SMOOTH = 0.03
    c.DROP_PATH = 0.05
    c.WEIGHT_DECAY = WD
    c.LR = TRUNK_LR
    c.W_CE, c.W_FOCAL = W_CE, W_FOCAL
    c.W_CENTER, c.W_CONSIST, c.W_DIVERSE = W_CENTER, W_CONSIST, W_DIVERSE
    if name == "STT-Repro":                    # the controlled base-paper ablation
        c.EPOCHS = STT_EPOCHS
        c.USE_FREQ_STREAM = False
        c.USE_TEMPORAL = False
        c.ADAPTIVE_ALPHA = False
    else:
        c.EPOCHS = FSTT_EPOCHS
    return c


def build(name):
    cfg = make_cfg(name)
    model = FSTTNet(cfg)
    stem = DeepStem(cfg.EMBED_DIMS[0], pretrained=getattr(cfg, "PRETRAINED_STEM", True))
    n_tok = (cfg.IMG_SIZE // (2 ** (STEM_INDEX + 1)))
    model.spatial_stem = stem
    if getattr(model, "use_freq", False):
        model.freq_stem = PoolTo(model.freq_stem, n_tok)   # 28x28 -> 14x14
    if not FINETUNE_STEM:
        for p in stem.backbone.parameters():
            p.requires_grad_(False)
    return model, cfg, n_tok


def stem_param_ids(model):
    try:
        return {id(p) for p in model.spatial_stem.backbone.parameters()}
    except Exception:
        return set()


def make_opt(model, criterion, lr):
    sids = stem_param_ids(model)
    trunk = [p for p in model.parameters() if p.requires_grad and id(p) not in sids]
    stem = [p for p in model.parameters() if p.requires_grad and id(p) in sids]
    groups = []
    if trunk:
        groups.append({"params": trunk, "lr": lr})
    if stem:
        groups.append({"params": stem, "lr": lr * STEM_LR_MULT})
    cp = [p for p in criterion.parameters() if p.requires_grad]
    if cp:
        groups.append({"params": cp, "lr": lr})
    if not groups:
        raise RuntimeError("no trainable parameters")
    return torch.optim.AdamW(groups, weight_decay=WD)


def new_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)


# ============================================================================
# 6. EVALUATION  (frame level AND video level)
# ============================================================================
@torch.no_grad()
def evaluate(model, loader, cfg):
    model.eval()
    P, Y, V = [], [], []
    amp = bool(cfg.AMP and DEVICE.type == "cuda")
    for b in tqdm(loader, desc="val", leave=False):
        img = b["img"].to(DEVICE, non_blocking=True)
        frq = b["freq"].to(DEVICE, non_blocking=True)
        sem = b["sem"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=amp):
            logits, _ = model(img, frq, sem)
        P.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
        Y.append(b["y"].numpy())
        V += list(b["vid"])
        del img, frq, sem, logits
    df = pd.DataFrame({"prob": np.concatenate(P), "y": np.concatenate(Y), "vid": V})
    two = df.y.nunique() > 1
    f_auc = float(roc_auc_score(df.y, df.prob)) if two else 0.0
    f_acc = float(accuracy_score(df.y, (df.prob >= 0.5).astype(int)))
    vg = df.groupby("vid").agg(prob=("prob", "mean"), y=("y", "first"))
    v_auc = float(roc_auc_score(vg.y, vg.prob)) if vg.y.nunique() > 1 else 0.0
    v_acc = float(accuracy_score(vg.y, (vg.prob >= 0.5).astype(int)))
    return {"frame_auc": f_auc, "frame_acc": f_acc,
            "video_auc": v_auc, "video_acc": v_acc, "n_videos": int(len(vg))}


# ============================================================================
# 7. CHECKPOINTS
# ============================================================================
def cp_paths(name):
    s = name.replace("/", "-")
    return {"last": SAVE_DIR / f"{s}_last{_EXT}", "best": SAVE_DIR / f"{s}_best{_EXT}",
            "meta": SAVE_DIR / f"{s}_meta.json"}


def state_of(m):
    return {k: v.detach().cpu().contiguous() for k, v in m.state_dict().items()}


def _json(o):
    if isinstance(o, dict):
        return {str(k): _json(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_json(v) for v in o]
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, Path):
        return str(o)
    return o


def save_meta(name, meta):
    p = cp_paths(name)["meta"]
    t = p.with_suffix(".tmp")
    t.write_text(json.dumps(_json(meta), indent=2))
    os.replace(t, p)


def load_meta(name):
    p = cp_paths(name)
    if FORCE_RESTART or not (p["meta"].exists() and p["last"].exists()):
        return None
    try:
        return json.loads(p["meta"].read_text())
    except Exception:
        return None


def topk_update(name, model, auc, epoch, recs):
    p = SAVE_DIR / f"{name.replace('/', '-')}_top_e{epoch:02d}_{auc:.5f}{_EXT}"
    _st_save(state_of(model), str(p))
    recs.append({"auc": float(auc), "epoch": int(epoch), "path": str(p)})
    recs.sort(key=lambda r: r["auc"], reverse=True)
    while len(recs) > TOPK_AVG:
        try:
            Path(recs.pop(-1)["path"]).unlink(missing_ok=True)
        except Exception:
            pass
    return recs


def topk_average(recs):
    live = [r for r in recs if Path(r["path"]).exists()]
    if len(live) < 2:
        return None
    states = [_st_load(r["path"], device="cpu") for r in live]
    out = {}
    for k in states[0]:
        f = states[0][k]
        if torch.is_floating_point(f):
            a = f.float().clone()
            for s in states[1:]:
                a.add_(s[k].float())
            out[k] = a.div_(len(states)).to(f.dtype)
        else:
            out[k] = f
    return out


# ============================================================================
# 8. BATCH PROBE
# ============================================================================
def probe(name):
    for bs in [16, 12, 8, 6, 4, 2]:
        m = crit = opt = None
        try:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            m, cfg, _ = build(name)
            m = m.to(DEVICE)
            crit = FSTTLoss(cfg, feat_dim=cfg.EMBED_DIMS[-1]).to(DEVICE)
            opt = make_opt(m, crit, cfg.LR)
            b = next(iter(train_loader(cfg, bs, 0, workers=0)), None)
            if b is None:
                continue
            img, frq = b["img"].to(DEVICE), b["freq"].to(DEVICE)
            sem, y = b["sem"].to(DEVICE), b["y"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type,
                                enabled=bool(cfg.AMP and DEVICE.type == "cuda")):
                lo, aux = m(img, frq, sem)
                loss, _ = crit(lo, aux, y)
            loss.backward()
            opt.step()
            print(f"  batch probe {name}: {bs} -> OK")
            return int(y.shape[0])
        except (_OOM, RuntimeError) as e:
            if isinstance(e, _OOM) or "out of memory" in str(e).lower():
                print(f"  batch probe {name}: {bs} -> OOM")
            else:
                raise
        finally:
            try:
                del m, crit, opt
            except Exception:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return 2


# ============================================================================
# 9. TRAIN
# ============================================================================
def train_one(name):
    print("\n" + "=" * 92)
    print("TRAINING", name)
    print("=" * 92)
    set_seed(CFG.SEED)

    model, cfg, n_tok = build(name)
    n_par = sum(p.numel() for p in model.parameters())
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  params: {n_par/1e6:.2f} M total, {n_tr/1e6:.2f} M trainable | "
          f"{n_tok}x{n_tok} = {n_tok*n_tok} tokens per frame | clip T={cfg.CLIP_LEN}")

    cp = cp_paths(name)
    meta = load_meta(name)
    start, best, bad = 0, -1.0, 0
    hist, recs = defaultdict(list), []

    if meta is not None:
        try:
            model.load_state_dict(_st_load(str(cp["last"]), device="cpu"), strict=True)
            start = int(meta.get("next_epoch", 0))
            best = float(meta.get("best_auc", -1))
            bad = int(meta.get("bad", 0))
            hist = defaultdict(list, dict(meta.get("history", {})))
            recs = list(meta.get("top_records", []))
            if meta.get("complete", False):
                if cp["best"].exists():
                    model.load_state_dict(_st_load(str(cp["best"]), device="cpu"), strict=True)
                print(f"  already complete | best frame AUC = {best:.4f}")
                return model.cpu(), dict(hist), best
            print(f"  resume from epoch {start+1} | best frame AUC = {best:.4f}")
        except Exception as e:
            print("  checkpoint incompatible, starting fresh:", type(e).__name__, e)
            start, best, bad = 0, -1.0, 0
            hist, recs = defaultdict(list), []

    bs = int(meta["batch_size"]) if (meta and meta.get("batch_size")) else probe(name)
    print("  batch:", bs)

    vl = val_loader(min(32, bs * 2))
    n_clips = len(train_ds(cfg, 0))
    steps = max(1, n_clips // max(1, min(bs, n_clips)))
    total = steps * cfg.EPOCHS
    warm = steps * max(1, cfg.WARMUP_EPOCHS)

    model = model.to(DEVICE)
    crit = FSTTLoss(cfg, feat_dim=cfg.EMBED_DIMS[-1]).to(DEVICE)
    opt = make_opt(model, crit, cfg.LR)
    amp = bool(cfg.AMP and DEVICE.type == "cuda")
    scaler = new_scaler(amp)

    for ep in range(start, cfg.EPOCHS):
        model.train()
        tl = train_loader(cfg, bs, ep)
        run, seen, corr = 0.0, 0, 0
        gstep = ep * steps
        pbar = tqdm(tl, desc=f"{name} ep{ep+1}/{cfg.EPOCHS}", leave=False)
        for b in pbar:
            lr = cosine_lr(gstep, total, warm, cfg.LR)
            for i, g in enumerate(opt.param_groups):
                g["lr"] = lr * (STEM_LR_MULT if i == 1 and len(opt.param_groups) > 1 else 1.0)

            img = b["img"].to(DEVICE, non_blocking=True)
            frq = b["freq"].to(DEVICE, non_blocking=True)
            sem = b["sem"].to(DEVICE, non_blocking=True)
            y = b["y"].to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=amp):
                lo, aux = model(img, frq, sem)
                loss, _ = crit(lo, aux, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], cfg.GRAD_CLIP)
            scaler.step(opt)
            scaler.update()

            n = int(y.size(0))
            run += float(loss.detach().float().item()) * n
            seen += n
            corr += int((lo.argmax(-1) == y).sum().item())
            gstep += 1
            pbar.set_postfix(loss=f"{run/max(seen,1):.4f}", acc=f"{corr/max(seen,1):.3f}",
                             lr=f"{lr:.1e}", bs=bs)
            del img, frq, sem, y, lo, aux, loss

        m = evaluate(model, vl, cfg)
        tr_loss, tr_acc = run / max(seen, 1), corr / max(seen, 1)
        for k, v in [("epoch", ep + 1), ("train_loss", tr_loss), ("train_acc", tr_acc),
                     ("val_auc", m["frame_auc"]), ("val_acc", m["frame_acc"]),
                     ("video_auc", m["video_auc"]), ("video_acc", m["video_acc"])]:
            hist[k].append(v)

        print(f"  [{name}] ep {ep+1:02d} | loss {tr_loss:.4f} | train acc {tr_acc:.4f} "
              f"|| frame AUC {m['frame_auc']:.4f} acc {m['frame_acc']:.4f} "
              f"|| VIDEO AUC {m['video_auc']:.4f} acc {m['video_acc']:.4f} "
              f"({m['n_videos']} videos)")

        if m["frame_auc"] > best + 1e-6:
            best, bad = m["frame_auc"], 0
            _st_save(state_of(model), str(cp["best"]))
            recs = topk_update(name, model, m["frame_auc"], ep + 1, recs)
            print(f"    new best frame AUC = {best:.4f}")
        else:
            bad += 1
            if len(recs) < TOPK_AVG or m["frame_auc"] > min(r["auc"] for r in recs):
                recs = topk_update(name, model, m["frame_auc"], ep + 1, recs)

        _st_save(state_of(model), str(cp["last"]))
        save_meta(name, {"next_epoch": ep + 1, "best_auc": best, "bad": bad,
                         "batch_size": bs, "complete": bad >= EARLY_STOP or ep + 1 >= cfg.EPOCHS,
                         "history": dict(hist), "top_records": recs})

        if ep == 1 and m["video_auc"] < 0.85:
            print("    NOTE: video AUC still below 0.85 after 2 epochs. If it does not move")
            print("          by epoch 4, stop and read the DIAGNOSIS block at the end.")
        if bad >= EARLY_STOP:
            print(f"    early stop | best frame AUC = {best:.4f}")
            break

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if cp["best"].exists():
        model.load_state_dict(_st_load(str(cp["best"]), device="cpu"), strict=True)
    keep = state_of(model)

    avg = topk_average(recs)
    if avg is not None:
        try:
            model.load_state_dict(avg, strict=True)
            am = evaluate(model.to(DEVICE), vl, cfg)
            print(f"  top-{len(recs)} weight average: frame AUC {am['frame_auc']:.4f} "
                  f"| video AUC {am['video_auc']:.4f}")
            if am["frame_auc"] > best + 1e-6:
                best = am["frame_auc"]
                _st_save(state_of(model), str(cp["best"]))
                keep = state_of(model)
                print(f"    averaged model is the new best = {best:.4f}")
            else:
                model.load_state_dict(keep, strict=True)
        except Exception as e:
            print("  averaging skipped:", type(e).__name__, e)
            model.load_state_dict(keep, strict=True)

    model.load_state_dict(keep, strict=True)
    fin = evaluate(model.to(DEVICE), vl, cfg)
    hist["best_val_auc"] = float(best)
    hist["final"] = fin
    save_meta(name, {"next_epoch": int(cfg.EPOCHS), "best_auc": float(best), "bad": int(bad),
                     "batch_size": int(bs), "complete": True,
                     "history": dict(hist), "top_records": recs})
    return model.cpu(), dict(hist), float(best)


# ============================================================================
# 10. RUN BOTH
# ============================================================================
RESULTS = {}
for _name in ["STT-Repro", "FSTT-Net"]:
    _m, _h, _a = train_one(_name)
    MODELS[_name] = _m.cpu()
    HISTORIES[_name] = _h
    RESULTS[_name] = _h.get("final", {"frame_auc": _a})
    _DS.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================================
# 11. REPORT + DIAGNOSIS
# ============================================================================
print("\n" + "=" * 92)
print("RETRAINING FINISHED")
print("=" * 92)
for k, v in RESULTS.items():
    print(f"{k:<12} frame AUC {v.get('frame_auc', 0):.4f} | "
          f"video AUC {v.get('video_auc', 0):.4f} | "
          f"frame acc {v.get('frame_acc', 0):.4f}")
print(f"{'Swin-T':<12} frame AUC {TARGET_BASELINE_AUC:.4f}   (best baseline so far)")

_f = RESULTS.get("FSTT-Net", {}).get("frame_auc", 0.0)
_s = RESULTS.get("STT-Repro", {}).get("frame_auc", 0.0)
print()
print("-" * 92)
print("DIAGNOSIS")
print("-" * 92)
if _f > TARGET_BASELINE_AUC and _f > _s:
    print("FSTT-Net beats both STT-Repro and the strongest baseline. That is the result")
    print("your Table IV needs. Report frame AND video AUC, and keep the STT-Repro row -")
    print("it is what makes the contribution attributable rather than asserted.")
elif _f > _s:
    print(f"FSTT-Net ({_f:.4f}) beats STT-Repro ({_s:.4f}), so the contributions do help,")
    print(f"but neither passes Swin-T ({TARGET_BASELINE_AUC:.4f}).")
    print("Do NOT write 'outperforms state of the art'. Two honest options:")
    print("  a) report Swin-T as the strongest baseline and pitch FSTT-Net on")
    print("     cross-dataset transfer, robustness and cost - where the base paper")
    print("     reports nothing and a small model can genuinely win;")
    print("  b) keep tuning: raise EMBED_DIMS to (192,384,768), STEM_MODEL to")
    print("     'convnext_tiny' or 'swin_tiny_patch4_window7_224', and STEM_LR_MULT to 0.25.")
else:
    print(f"FSTT-Net ({_f:.4f}) did NOT beat STT-Repro ({_s:.4f}). The added modules are")
    print("not paying for themselves on this data. Before more tuning, check the crops:")
    print("if your extraction ran with the centre-crop fallback, the faces are not aligned")
    print("and every token-based model is handicapped, while a plain CNN baseline is not.")
print()
print("Also worth knowing: this dataset is ~5.5:1 fake:real, so accuracy at threshold 0.5")
print("is not informative - a constant 'fake' predictor scores ~0.85. Report AUC and EER.")
print("=" * 92)
print("\nNext: run the RESULTS PACK cell to regenerate every table and figure.")


Mounted at /content/drive
Drive mounted
numpy 2.0.2 <-> torch 2.11.0+cu128 bridge OK
device = cuda | torch 2.11.0+cu128
gpu    = Tesla T4
root   = /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2

rebuilding FRAMES from /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2/faces
  celebdf: 121,175 crops | splits=['test', 'train', 'val'] | labels={1: 102437, 0: 18738}
  ff++: 38,613 crops | splits=['test', 'train', 'val'] | labels={1: 25703, 0: 12910}
mediapipe unavailable -> canonical template prior

ALL 15 PREREQUISITES ARE LIVE - now run CELL B (training).
device = cuda | datasets = ['celebdf', 'ff++'] | primary = celebdf
CELL B2 :: RETRAIN STT-Repro + FSTT-Net
dataset   : celebdf  train=88,887 val=15,712 crops
class mix : {1: 102437, 0: 18738}  (0=real, 1=fake)
device    : cuda
dims      : (128, 256, 512)   stem: efficientnet_b3 @ out_index 3 (fine-tuned)
target    : beat frame AUC 0.9663
ckpt dir  : /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2/checkpoints/last2_v6
gpu       : Tesla T4  14

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

model.safetensors: downloading bytes:           |  0.00B            

...

  [FSTT-Net] ep 18 | loss 0.0903 | train acc 0.9974 || frame AUC 0.9919 acc 0.9575 || VIDEO AUC 0.9957 acc 0.9776 (491 videos)
    early stop | best frame AUC = 0.9949


val:   0%|          | 0/123 [00:00<?, ?it/s]

  top-3 weight average: frame AUC 0.9949 | video AUC 0.9984
    averaged model is the new best = 0.9949


val:   0%|          | 0/123 [00:00<?, ?it/s]


RETRAINING FINISHED
STT-Repro    frame AUC 0.9878 | video AUC 0.9954 | frame acc 0.9038
FSTT-Net     frame AUC 0.9949 | video AUC 0.9984 | frame acc 0.9560
Swin-T       frame AUC 0.9663   (best baseline so far)

--------------------------------------------------------------------------------------------
DIAGNOSIS
--------------------------------------------------------------------------------------------
FSTT-Net beats both STT-Repro and the strongest baseline. That is the result
your Table IV needs. Report frame AND video AUC, and keep the STT-Repro row -
it is what makes the contribution attributable rather than asserted.

Also worth knowing: this dataset is ~5.5:1 fake:real, so accuracy at threshold 0.5
is not informative - a constant 'fake' predictor scores ~0.85. Report AUC and EER.

Next: run the RESULTS PACK cell to regenerate every table and figure.


---
## 3 · Cross-dataset generalisation + robustness

No retraining. With FF++ now cached, `CROSS_TARGETS = None` evaluates transfer to
**both** WildDeepfake and FF++ — the setting the base paper reports in its Tables IV
and V. Produces `CROSS` and `ROB` for cell 4.

In [4]:
import os, shutil
from pathlib import Path

DRIVE = Path("/content/drive")
try:
    from google.colab import drive
    if not os.path.ismount(DRIVE):                 # <-- real check, not is_dir()
        if DRIVE.exists() and any(DRIVE.iterdir()):
            stale = Path("/content/drive_stale")
            if stale.exists():
                shutil.rmtree(stale)
            DRIVE.rename(stale)
            print("stale local /content/drive moved to", stale)
        drive.mount(str(DRIVE))
    print("Drive mounted:", os.path.ismount(DRIVE))
except Exception as e:
    print("Drive not available:", e)

Mounted at /content/drive
Drive mounted: True


In [5]:
import sys, os, subprocess, shutil
from pathlib import Path

# Drive mount - checks os.path.ismount(), NOT directory existence. A stale
# local /content/drive folder left by a dropped mount looks exactly like a real
# mount to is_dir(), so the mount gets skipped and everything silently reads
# local disk. That is what made the checkpoints "disappear".
_DRIVE = Path("/content/drive")
try:
    from google.colab import drive as _gdrive
    if not os.path.ismount(_DRIVE):
        if _DRIVE.exists() and any(_DRIVE.iterdir()):
            _stale = Path("/content/drive_stale")
            if _stale.exists():
                shutil.rmtree(_stale)
            _DRIVE.rename(_stale)
            print("stale local /content/drive moved to", _stale)
        _gdrive.mount(str(_DRIVE))
    print("Drive mounted:", os.path.ismount(_DRIVE))
except Exception as _e:
    print("Drive not available (fine if CFG.ROOT is local):", _e)

Drive mounted: True


In [6]:
from pathlib import Path
import os, pandas as pd

print("drive mounted:", os.path.ismount("/content/drive"))

ROOT = Path("/content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2")
FACE_DIR = ROOT / "faces"
print("\nROOT     :", ROOT, "exists:", ROOT.is_dir())
print("FACE_DIR :", FACE_DIR, "exists:", FACE_DIR.is_dir())

if ROOT.is_dir():
    print("\ninside ROOT:")
    for p in sorted(ROOT.iterdir()):
        print("   ", p.name + ("/" if p.is_dir() else ""))

if FACE_DIR.is_dir():
    print("\ninside faces/:")
    for p in sorted(FACE_DIR.iterdir()):
        n = len(list(p.glob('*'))) if p.is_dir() else 0
        print(f"    {p.name}{'/' if p.is_dir() else ''}   ({n} entries)")

print("\n--- every manifest.csv under MyDrive ---")
for m in Path("/content/drive/MyDrive").glob("**/faces/*/manifest.csv"):
    try:
        df = pd.read_csv(m)
        head = df.face.head(50)
        alive = int(head.map(lambda p: Path(str(p)).exists()).sum())
        print(f"\n  {m}")
        print(f"    rows={len(df):,}  sequences={df.video_id.nunique():,}  "
              f"splits={sorted(df.split.unique())}")
        print(f"    first 50 crop paths that exist on disk: {alive}/50")
        print(f"    example path stored in the manifest:")
        print(f"      {df.face.iloc[0]}")
        if alive == 0:
            print("    ^^ THE PATHS IN THIS MANIFEST DO NOT RESOLVE - that is why it is skipped")
    except Exception as e:
        print(f"  {m}  -> unreadable: {e}")

drive mounted: True

ROOT     : /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2 exists: True
FACE_DIR : /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2/faces exists: True

inside ROOT:
    checkpoints/
    faces/
    figures/
    models/
    results/
    state/
    tables/

inside faces/:
    celebdf/   (4 entries)
    ff++/   (4 entries)
    wilddeepfake/   (3 entries)

--- every manifest.csv under MyDrive ---

  /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2/faces/celebdf/manifest.csv
    rows=121,175  sequences=6,529  splits=['test', 'train', 'val']
    first 50 crop paths that exist on disk: 50/50
    example path stored in the manifest:
      /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2/faces/celebdf/train/0/559c7888a6b9_00000.jpg

  /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2/faces/ff++/manifest.csv
    rows=38,613  sequences=3,000  splits=['test', 'train', 'val']
    first 50 crop paths that exist on disk: 50/50
    example path stored in the manifest:
      /content/drive/My

KeyboardInterrupt: 

In [ ]:
# ############################################################################
# CELL 3 :: CROSS-DATASET + ROBUSTNESS  (single cell, loads checkpoints)
# ----------------------------------------------------------------------------
# Does not need cell 2 live and does not train. Rebuilds the definitions,
# loads the weights from checkpoints/last2_v6/*_best.safetensors, and runs the
# study on whichever corpora actually have a manifest on disk.
# ############################################################################

import sys, os, subprocess, shutil
from pathlib import Path

_DRIVE = Path("/content/drive")
try:
    from google.colab import drive as _gdrive
    if not os.path.ismount(_DRIVE):                      # is_dir() is fooled by a stale shell
        if _DRIVE.exists() and any(_DRIVE.iterdir()):
            _stale = Path("/content/drive_stale")
            if _stale.exists():
                shutil.rmtree(_stale)
            _DRIVE.rename(_stale)
            print("stale local /content/drive moved to", _stale)
        _gdrive.mount(str(_DRIVE))
    print("Drive mounted:", os.path.ismount(_DRIVE))
except Exception as _e:
    print("Drive not available (fine if CFG.ROOT is local):", _e)

try:
    import numpy as _np, torch as _t
    _ = _t.from_numpy(_np.zeros(4, dtype=_np.float32)).numpy().sum()
    import numpy.testing
    import timm  # noqa
except Exception as _e:
    print("environment broken:", _e)
    print("    !pip install -q --force-reinstall --no-cache-dir numpy==2.0.2")
    print("then RESTART the runtime and rerun this cell.")
    raise

import json, math, time, random, warnings, copy, gc
from collections import defaultdict, OrderedDict
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import cv2
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score
from scipy.fftpack import dct

warnings.filterwarnings("ignore")


# ==================== CONFIG ====================
@dataclass
class Config:
    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    AMP: bool = True
    PRIMARY: str = "celebdf"
    IMG_SIZE: int = 224
    CLIP_LEN: int = 4
    LABEL_SMOOTH: float = 0.05
    EMBED_DIMS: tuple = (64, 128, 256)
    NUM_HEADS: tuple = (1, 2, 4)
    DEPTHS: tuple = (2, 2, 2)
    MLP_RATIO: float = 4.0
    KNN_K: int = 5
    ALPHA: float = 0.2
    ADAPTIVE_ALPHA: bool = True
    CLUSTER_RATIO: float = 0.25
    N_ITERS: int = 3
    USE_FREQ_STREAM: bool = True
    USE_TEMPORAL: bool = True
    USE_SEMANTIC: bool = True
    PRETRAINED_STEM: bool = True
    DROP_PATH: float = 0.1
    W_CE: float = 1.0
    W_FOCAL: float = 0.5
    W_CENTER: float = 0.05
    W_CONSIST: float = 0.10
    W_DIVERSE: float = 0.01
    FOCAL_GAMMA: float = 2.0
    EPOCHS: int = 20
    WEIGHT_DECAY: float = 0.05
    LR: float = 2e-4
    ROOT: str = "/content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2"   # <<< cell 1 cache

CFG = Config()
ROOT = Path(CFG.ROOT)
FACE_DIR = ROOT / "faces"
FIG_DIR, TAB_DIR, RES_DIR = ROOT / "figures", ROOT / "tables", ROOT / "results"
for d in (FIG_DIR, TAB_DIR, RES_DIR):
    d.mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(CFG.SEED)
DEVICE = torch.device(CFG.DEVICE)
print(f"device = {DEVICE} | torch {torch.__version__}")
print("root   =", ROOT)


# ==================== FRAMES ====================
FRAMES = {}
print("\nscanning", FACE_DIR)
for _mp in sorted(FACE_DIR.glob("*/manifest.csv")):
    _key = _mp.parent.name
    try:
        _df = pd.read_csv(_mp)
    except Exception as e:
        print(f"  {_key:<14} manifest unreadable ({e})")
        continue
    if len(_df) == 0 or "face" not in _df.columns:
        print(f"  {_key:<14} empty manifest")
        continue
    if not _df.face.head(50).map(lambda p: Path(str(p)).exists()).any():
        print(f"  {_key:<14} manifest has {len(_df):,} rows but the crops are gone - SKIPPED")
        continue
    FRAMES[_key] = _df
    print(f"  {_key:<14} {len(_df):,} crops | sequences={_df.video_id.nunique():,} | "
          f"splits={sorted(_df.split.unique())}")

for _k in ("celebdf", "ff++", "wilddeepfake"):
    if _k not in FRAMES:
        print(f"  NOTE: '{_k}' has no usable cache - it will be skipped everywhere below.")

if not FRAMES:
    raise RuntimeError(f"No usable face cache under {FACE_DIR}. Check CFG.ROOT and the mount.")
if CFG.PRIMARY not in FRAMES:
    raise RuntimeError(f"PRIMARY corpus '{CFG.PRIMARY}' has no cache. Cannot evaluate.")


# ==================== SEMANTIC PRIOR + SRM ====================
REGION_NAMES = ["background", "skin", "eyes", "nose", "mouth"]
N_REGIONS = len(REGION_NAMES)

class SemanticPrior:
    def __init__(self, size=224, use_mediapipe=True):
        self.size = size
        self.mp = None
        if use_mediapipe:
            try:
                import mediapipe as mp
                self.mp = mp.solutions.face_mesh.FaceMesh(static_image_mode=True,
                                                          max_num_faces=1, refine_landmarks=True)
                print("semantic prior backend: mediapipe FaceMesh")
            except Exception:
                print("mediapipe unavailable -> canonical template prior")
        self._template = self._build_template(size)

    @staticmethod
    def _ellipse_mask(size, cx, cy, rx, ry, soft=0.06):
        yy, xx = np.mgrid[0:size, 0:size].astype(np.float32) / size
        d = ((xx - cx) / max(rx, 1e-6)) ** 2 + ((yy - cy) / max(ry, 1e-6)) ** 2
        return np.clip(1.0 - (d - 1.0) / soft, 0, 1).astype(np.float32)

    def _build_template(self, size):
        m = np.zeros((N_REGIONS, size, size), np.float32)
        m[1] = self._ellipse_mask(size, 0.50, 0.54, 0.34, 0.44, 0.20)
        m[2] = np.maximum(self._ellipse_mask(size, 0.355, 0.42, 0.095, 0.055),
                          self._ellipse_mask(size, 0.645, 0.42, 0.095, 0.055))
        m[3] = self._ellipse_mask(size, 0.50, 0.545, 0.075, 0.115)
        m[4] = self._ellipse_mask(size, 0.50, 0.715, 0.135, 0.070)
        m[1] = np.clip(m[1] - m[2] - m[3] - m[4], 0, 1)
        m[0] = np.clip(1.0 - m[1:].sum(0), 0, 1)
        return m / (m.sum(0, keepdims=True) + 1e-6)

    def __call__(self, rgb):
        return self._template.copy()

SEMPRIOR = SemanticPrior(CFG.IMG_SIZE)

SRM_KERNELS = np.array([
    [[0,0,0,0,0],[0,-1,2,-1,0],[0,2,-4,2,0],[0,-1,2,-1,0],[0,0,0,0,0]],
    [[-1,2,-2,2,-1],[2,-6,8,-6,2],[-2,8,-12,8,-2],[2,-6,8,-6,2],[-1,2,-2,2,-1]],
    [[0,0,0,0,0],[0,0,0,0,0],[0,1,-2,1,0],[0,0,0,0,0],[0,0,0,0,0]],
], np.float32)
SRM_KERNELS = SRM_KERNELS / np.array([4.0, 12.0, 2.0], np.float32)[:, None, None]

IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], np.float32)

def fast_dct_bands(gray_f32):
    h, w = gray_f32.shape
    c = cv2.dct(gray_f32)
    yy, xx = np.mgrid[0:h, 0:w]
    radius = (yy / h + xx / w) / 2.0
    out = np.empty((3, h, w), np.float32)
    edges = [0.0, 1/3, 2/3, 1.0]
    for i in range(3):
        mask = ((radius >= edges[i]) & (radius < edges[i+1])).astype(np.float32)
        out[i] = cv2.idct(c * mask)
    return out

def make_freq_input(rgb_u8):
    g = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    bands = fast_dct_bands(g)
    res = np.stack([cv2.filter2D(g, -1, k) for k in SRM_KERNELS]).astype(np.float32)
    x = np.concatenate([bands, res], 0)
    return x / (np.abs(x).max(axis=(1, 2), keepdims=True) + 1e-6)

_SEM_TEMPLATE = SEMPRIOR._template.copy()

def fast_semantic_prior(rgb_u8):
    ycc = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2YCrCb)
    cr, cb = ycc[..., 1].astype(np.float32), ycc[..., 2].astype(np.float32)
    skin = np.exp(-(((cr - 150) / 22.0) ** 2 + ((cb - 115) / 22.0) ** 2) / 2.0).astype(np.float32)
    skin = cv2.GaussianBlur(skin, (0, 0), 3.0)
    if skin.shape[0] != _SEM_TEMPLATE.shape[1]:
        skin = cv2.resize(skin, _SEM_TEMPLATE.shape[1:][::-1])
    m = _SEM_TEMPLATE.copy()
    m[1:] = m[1:] * (0.5 + 0.5 * skin[None])
    m[0] = np.clip(1.0 - m[1:].sum(0), 0, 1)
    return (m / (m.sum(0, keepdims=True) + 1e-6)).astype(np.float32)

def corrupt_frame(f, kind, level):
    if kind == "clean": return f
    if kind == "jpeg":
        ok, enc = cv2.imencode(".jpg", cv2.cvtColor(f, cv2.COLOR_RGB2BGR),
                               [int(cv2.IMWRITE_JPEG_QUALITY), int(level)])
        return cv2.cvtColor(cv2.imdecode(enc, 1), cv2.COLOR_BGR2RGB) if ok else f
    if kind == "blur":  return cv2.GaussianBlur(f, (0, 0), float(level))
    if kind == "noise":
        rng = np.random.RandomState(0)
        return np.clip(f.astype(np.float32) + rng.normal(0, level, f.shape), 0, 255).astype(np.uint8)
    if kind == "resize":
        s = max(16, int(f.shape[0] * level))
        return cv2.resize(cv2.resize(f, (s, s)), (f.shape[1], f.shape[0]))
    return f


# ==================== MODEL ====================
from timm.layers import DropPath, trunc_normal_

def index_points(points, idx):
    B = points.shape[0]
    view_shape = [B] + [1] * (idx.dim() - 1)
    rep = [1] + list(idx.shape[1:])
    bidx = torch.arange(B, device=points.device, dtype=torch.long).view(view_shape).repeat(rep)
    return points[bidx, idx, :]

@torch.no_grad()
def dpc_knn_cluster(x, sem, k, cluster_num, alpha):
    B, N, C = x.shape
    xf = x.float()
    dist = torch.cdist(xf, xf) / (C ** 0.5)
    kk = min(k, N)
    knn_d, _ = dist.topk(kk, dim=-1, largest=False)
    rho_x = torch.exp(-(knn_d ** 2).mean(-1))
    if sem is not None:
        semf = sem.float()
        dsem = torch.cdist(semf, semf) / (sem.shape[-1] ** 0.5)
        knn_s, _ = dsem.topk(kk, dim=-1, largest=False)
        rho_y = torch.exp(-(knn_s ** 2).mean(-1))
        a = alpha if torch.is_tensor(alpha) else torch.full((B, 1), float(alpha), device=x.device)
        rho = (1 - a.view(B, 1).float()) * rho_x + a.view(B, 1).float() * rho_y
    else:
        rho = rho_x
    rho = rho + 1e-6 * torch.rand_like(rho)
    higher = rho[:, None, :] > rho[:, :, None]
    dmax = dist.flatten(1).max(-1)[0][:, None, None]
    delta, _ = (dist * higher + dmax * (~higher)).min(dim=-1)
    score = rho * delta
    M = min(cluster_num, N)
    _, centers = torch.topk(score, k=M, dim=-1)
    idx_cluster = index_points(dist, centers).argmin(dim=1)
    bidx = torch.arange(B, device=x.device)[:, None]
    idx_cluster[bidx, centers] = torch.arange(M, device=x.device)[None, :].expand(B, -1)
    return idx_cluster, M, score, centers

def merge_tokens(x, idx_cluster, M, weight, extra=None):
    """AMP-safe: everything in float32, cast back at the end."""
    B, N, C = x.shape
    dt = x.dtype
    x32 = x.float()
    w32 = (torch.ones(B, N, 1, device=x.device, dtype=torch.float32)
           if weight is None else torch.exp(weight.float()).unsqueeze(-1))
    idx = (idx_cluster.long()
           + torch.arange(B, device=x.device, dtype=torch.long)[:, None] * M).reshape(-1)
    all_w = torch.zeros(B * M, 1, device=x.device, dtype=torch.float32)
    all_w.index_add_(0, idx, w32.reshape(B * N, 1))
    all_w.clamp_min_(1e-6)
    norm_w = w32 / all_w[idx].reshape(B, N, 1)
    merged = torch.zeros(B * M, C, device=x.device, dtype=torch.float32)
    merged.index_add_(0, idx, (x32 * norm_w).reshape(B * N, C))
    merged = merged.reshape(B, M, C)
    size = torch.zeros(B * M, 1, device=x.device, dtype=torch.float32)
    size.index_add_(0, idx, torch.ones(B * N, 1, device=x.device, dtype=torch.float32))
    size = size.reshape(B, M, 1)
    out_extra = None
    if extra is not None:
        e32 = extra.float(); E = e32.shape[-1]
        out_extra = torch.zeros(B * M, E, device=x.device, dtype=torch.float32)
        out_extra.index_add_(0, idx, (e32 * norm_w).reshape(B * N, E))
        out_extra = out_extra.reshape(B, M, E).to(dt)
    return merged.to(dt), out_extra, size.to(dt)

class SFScoringNet(nn.Module):
    def __init__(self, dim, sem_dim=N_REGIONS, adaptive=True, alpha_init=0.2):
        super().__init__()
        self.adaptive = adaptive
        self.feat_score = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim // 2),
                                        nn.GELU(), nn.Linear(dim // 2, 1))
        self.sem_score = nn.Sequential(nn.LayerNorm(sem_dim), nn.Linear(sem_dim, dim // 2),
                                       nn.GELU(), nn.Linear(dim // 2, 1))
        self.alpha_gate = nn.Sequential(nn.LayerNorm(dim + sem_dim), nn.Linear(dim + sem_dim, 32),
                                        nn.GELU(), nn.Linear(32, 1))
        self.alpha_bias = nn.Parameter(torch.tensor(math.log(alpha_init / (1 - alpha_init))))

    def forward(self, x, sem):
        s_x = self.feat_score(x).squeeze(-1)
        s_y = self.sem_score(sem).squeeze(-1)
        g = torch.cat([x.mean(1), sem.mean(1)], -1)
        alpha = (torch.sigmoid(self.alpha_gate(g) + self.alpha_bias) if self.adaptive
                 else torch.sigmoid(self.alpha_bias).expand(x.shape[0], 1))
        return (1 - alpha) * s_x + alpha * s_y, alpha

class SerialBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4.0, drop_path=0.0):
        super().__init__()
        self.h, self.scale = heads, (dim // heads) ** -0.5
        self.n1q, self.n1kv = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.q = nn.Linear(dim, dim); self.kv = nn.Linear(dim, 2 * dim); self.proj = nn.Linear(dim, dim)
        self.n2 = nn.LayerNorm(dim)
        self.sa = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n3 = nn.LayerNorm(dim)
        hid = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hid), nn.GELU(), nn.Linear(hid, dim))
        self.dp = DropPath(drop_path) if drop_path > 0 else nn.Identity()

    def forward(self, q_tokens, kv_tokens, score_bias):
        B, M, C = q_tokens.shape
        N = kv_tokens.shape[1]
        q = self.q(self.n1q(q_tokens)).reshape(B, M, self.h, C // self.h).transpose(1, 2)
        kv = self.kv(self.n1kv(kv_tokens)).reshape(B, N, 2, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        if score_bias is not None:
            attn = attn + score_bias[:, None, None, :]
        attn = attn.softmax(-1)
        out = (attn @ v).transpose(1, 2).reshape(B, M, C)
        x = q_tokens + self.dp(self.proj(out))
        h = self.n2(x); x = x + self.dp(self.sa(h, h, h, need_weights=False)[0])
        return x + self.dp(self.mlp(self.n3(x)))

class CrossDomainFusion(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.a1 = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.a2 = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n1, self.n2, self.n3 = nn.LayerNorm(dim), nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.gate = nn.Sequential(nn.Linear(2 * dim, dim), nn.Sigmoid())
        self.proj = nn.Linear(2 * dim, dim)

    def forward(self, xs, xf):
        s2f = self.a1(self.n1(xs), self.n2(xf), self.n2(xf), need_weights=False)[0]
        f2s = self.a2(self.n2(xf), self.n1(xs), self.n1(xs), need_weights=False)[0]
        xs_e, xf_e = xs + s2f, xf + f2s
        g = self.gate(torch.cat([xs_e, xf_e], -1))
        return self.n3(self.proj(torch.cat([g * xs_e, (1 - g) * xf_e], -1))), xs_e, xf_e

class TemporalTokenPropagation(nn.Module):
    def __init__(self, dim, heads=4, max_t=16):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, max_t, dim)); trunc_normal_(self.pos, std=0.02)
        self.n = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.mlp = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.Linear(dim, dim))
        self.diff_proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim))

    def forward(self, f):
        B, T, C = f.shape
        if T == 1:
            return f[:, 0], f.new_zeros(B, C)
        x = f + self.pos[:, :T]
        h = self.n(x); x = x + self.attn(h, h, h, need_weights=False)[0]
        x = x + self.mlp(x)
        diff = self.diff_proj((f[:, 1:] - f[:, :-1]).abs().mean(1))
        return x.mean(1) + diff, diff

class SpatialStem(nn.Module):
    def __init__(self, dim, pretrained=True):
        super().__init__()
        self.backbone = None
        if pretrained:
            try:
                import timm
                self.backbone = timm.create_model("efficientnet_b0", pretrained=True,
                                                  features_only=True, out_indices=(2,))
                ch = self.backbone.feature_info.channels()[-1]
            except Exception as e:
                print("pretrained stem unavailable ->", e)
        if self.backbone is None:
            ch = 64
            self.backbone = nn.Sequential(
                nn.Conv2d(3, 32, 3, 2, 1), nn.BatchNorm2d(32), nn.GELU(),
                nn.Conv2d(32, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.GELU(),
                nn.Conv2d(64, ch, 3, 2, 1), nn.BatchNorm2d(ch), nn.GELU())
        self.proj = nn.Sequential(nn.Conv2d(ch, dim, 1), nn.BatchNorm2d(dim))

    def forward(self, x):
        f = self.backbone(x)
        f = f[-1] if isinstance(f, (list, tuple)) else f
        f = self.proj(f)
        B, C, H, W = f.shape
        return f.flatten(2).transpose(1, 2), (H, W)

class FrequencyStem(nn.Module):
    def __init__(self, dim, in_ch=6):
        super().__init__()
        self.band_w = nn.Parameter(torch.ones(in_ch))
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, 2, 1), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, dim, 3, 2, 1), nn.BatchNorm2d(dim), nn.GELU(),
            nn.Conv2d(dim, dim, 3, 1, 1), nn.BatchNorm2d(dim))

    def forward(self, x):
        f = self.net(x * self.band_w.view(1, -1, 1, 1))
        B, C, H, W = f.shape
        return f.flatten(2).transpose(1, 2), (H, W)

class FSTTNet(nn.Module):
    def __init__(self, cfg=CFG, num_classes=2):
        super().__init__()
        self.cfg = cfg
        dims, heads, depths = cfg.EMBED_DIMS, cfg.NUM_HEADS, cfg.DEPTHS
        self.use_freq, self.use_temporal, self.use_sem = (cfg.USE_FREQ_STREAM,
                                                          cfg.USE_TEMPORAL, cfg.USE_SEMANTIC)
        self.spatial_stem = SpatialStem(dims[0], cfg.PRETRAINED_STEM)
        if self.use_freq:
            self.freq_stem = FrequencyStem(dims[0])
            self.fusion = CrossDomainFusion(dims[0], heads=4)
        dpr = torch.linspace(0, cfg.DROP_PATH, sum(depths)).tolist()
        self.scorers, self.blocks, self.downs = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        c = 0
        for i in range(cfg.N_ITERS):
            d_in, d_out = dims[min(i, len(dims) - 1)], dims[min(i + 1, len(dims) - 1)]
            self.scorers.append(SFScoringNet(d_in, N_REGIONS, cfg.ADAPTIVE_ALPHA, cfg.ALPHA))
            self.blocks.append(nn.ModuleList([
                SerialBlock(d_in, heads[min(i, len(heads) - 1)], cfg.MLP_RATIO, dpr[c + j])
                for j in range(depths[min(i, len(depths) - 1)])]))
            self.downs.append(nn.Linear(d_in, d_out) if d_in != d_out else nn.Identity())
            c += depths[min(i, len(depths) - 1)]
        d_final = dims[-1]
        self.norm = nn.LayerNorm(d_final)
        if self.use_temporal:
            self.temporal = TemporalTokenPropagation(d_final, heads=4)
        self.head = nn.Sequential(nn.LayerNorm(d_final), nn.Dropout(0.2),
                                  nn.Linear(d_final, num_classes))
        self.head_s = nn.Linear(dims[0], num_classes)
        self.head_f = nn.Linear(dims[0], num_classes) if self.use_freq else None
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def _pool_sem(self, sem, hw):
        return F.adaptive_avg_pool2d(sem, hw).flatten(2).transpose(1, 2)

    def forward_tokens(self, img, freq, sem, want_maps=False):
        aux = {"alphas": [], "cluster_maps": [], "scores": []}
        xs, hw = self.spatial_stem(img)
        semt = self._pool_sem(sem, hw) if self.use_sem else torch.zeros(
            xs.shape[0], xs.shape[1], N_REGIONS, device=xs.device, dtype=xs.dtype)
        if self.use_freq:
            xf, _ = self.freq_stem(freq)
            if xf.shape[1] != xs.shape[1]:
                xf = F.interpolate(xf.transpose(1, 2), size=xs.shape[1], mode="linear",
                                   align_corners=False).transpose(1, 2)
            x, xs_e, xf_e = self.fusion(xs, xf)
            aux["logit_s"] = self.head_s(xs_e.mean(1))
            aux["logit_f"] = self.head_f(xf_e.mean(1))
        else:
            x = xs
            aux["logit_s"] = self.head_s(xs.mean(1))
            aux["logit_f"] = None
        token_hw = hw
        for i in range(self.cfg.N_ITERS):
            score, alpha = self.scorers[i](x, semt)
            aux["alphas"].append(alpha); aux["scores"].append(score)
            M = max(8, int(x.shape[1] * self.cfg.CLUSTER_RATIO))
            idx_cluster, M, dpc_score, centers = dpc_knn_cluster(
                x, semt if self.use_sem else None, self.cfg.KNN_K, M, alpha.detach())
            if want_maps:
                aux["cluster_maps"].append((idx_cluster.detach().cpu(), token_hw, M))
            merged, sem_merged, size = merge_tokens(x, idx_cluster, M, score, extra=semt)
            for blk in self.blocks[i]:
                merged = blk(merged, x, score)
            x = self.downs[i](merged)
            semt = sem_merged if sem_merged is not None else semt
            token_hw = None
        return self.norm(x).mean(1), aux

    def forward(self, img, freq=None, sem=None, want_maps=False):
        if img.dim() == 4:
            img = img.unsqueeze(1)
            if freq is not None: freq = freq.unsqueeze(1)
            if sem is not None: sem = sem.unsqueeze(1)
        B, T = img.shape[:2]
        img_f = img.flatten(0, 1)
        freq_f = freq.flatten(0, 1) if freq is not None else torch.zeros_like(img_f[:, :6])
        sem_f = sem.flatten(0, 1) if sem is not None else torch.zeros(
            img_f.shape[0], N_REGIONS, img_f.shape[-2], img_f.shape[-1], device=img_f.device)
        feat, aux = self.forward_tokens(img_f, freq_f, sem_f, want_maps)
        feat_t = feat.view(B, T, feat.shape[-1])
        pooled = self.temporal(feat_t)[0] if (self.use_temporal and T > 1) else feat_t.mean(1)
        logits = self.head(pooled)
        aux["feat"] = pooled
        aux["logit_s"] = aux["logit_s"].view(B, T, -1).mean(1)
        if aux.get("logit_f") is not None:
            aux["logit_f"] = aux["logit_f"].view(B, T, -1).mean(1)
        return logits, aux


# ==================== BUILD (must match cell 2) ====================
EMBED_DIMS    = (128, 256, 512)
NUM_HEADS     = (2, 4, 8)
DEPTHS        = (2, 2, 2)
STEM_MODEL    = "efficientnet_b3"
STEM_INDEX    = 3
FINETUNE_STEM = True
W_CE, W_FOCAL, W_CENTER, W_CONSIST, W_DIVERSE = 1.0, 0.0, 0.02, 0.05, 0.0

try:
    from safetensors.torch import load_file as _st_load
    _EXT = ".safetensors"
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "safetensors"], check=False)
    try:
        from safetensors.torch import load_file as _st_load
        _EXT = ".safetensors"
    except Exception:
        _EXT = ".pt"
        def _st_load(path, device="cpu"): return torch.load(path, map_location=device)

class DeepStem(nn.Module):
    def __init__(self, dim, model_name=STEM_MODEL, out_index=STEM_INDEX, pretrained=True):
        super().__init__()
        import timm
        self.backbone = timm.create_model(model_name, pretrained=pretrained,
                                          features_only=True, out_indices=(out_index,))
        ch = self.backbone.feature_info.channels()[-1]
        self.proj = nn.Sequential(nn.Conv2d(ch, dim, 1, bias=False),
                                  nn.BatchNorm2d(dim), nn.GELU())

    def forward(self, x):
        f = self.backbone(x)
        f = f[-1] if isinstance(f, (list, tuple)) else f
        f = self.proj(f)
        B, C, H, W = f.shape
        return f.flatten(2).transpose(1, 2), (H, W)

class PoolTo(nn.Module):
    def __init__(self, base, pool):
        super().__init__()
        self.base, self.pool = base, int(pool)
    def forward(self, x):
        tok, hw = self.base(x)
        if hw is None: return tok, hw
        H, W = hw
        if H <= self.pool and W <= self.pool: return tok, hw
        B, N, C = tok.shape
        f = tok.transpose(1, 2).reshape(B, C, H, W)
        f = F.adaptive_avg_pool2d(f, (self.pool, self.pool))
        return f.flatten(2).transpose(1, 2), (self.pool, self.pool)

def make_cfg(name):
    c = copy.deepcopy(CFG)
    c.EMBED_DIMS, c.NUM_HEADS, c.DEPTHS = EMBED_DIMS, NUM_HEADS, DEPTHS
    c.AMP = True
    c.LABEL_SMOOTH = 0.03
    c.DROP_PATH = 0.05
    c.W_CE, c.W_FOCAL = W_CE, W_FOCAL
    c.W_CENTER, c.W_CONSIST, c.W_DIVERSE = W_CENTER, W_CONSIST, W_DIVERSE
    if name == "STT-Repro":
        c.USE_FREQ_STREAM = False
        c.USE_TEMPORAL = False
        c.ADAPTIVE_ALPHA = False
    return c

def build(name):
    cfg = make_cfg(name)
    model = FSTTNet(cfg)
    stem = DeepStem(cfg.EMBED_DIMS[0], pretrained=getattr(cfg, "PRETRAINED_STEM", True))
    n_tok = cfg.IMG_SIZE // (2 ** (STEM_INDEX + 1))
    model.spatial_stem = stem
    if getattr(model, "use_freq", False):
        model.freq_stem = PoolTo(model.freq_stem, n_tok)
    return model, cfg, n_tok

SAVE_DIR = ROOT / "checkpoints" / "last2_v6"

def cp_paths(name):
    s = name.replace("/", "-")
    return {"best": SAVE_DIR / f"{s}_best{_EXT}", "meta": SAVE_DIR / f"{s}_meta.json"}


print("\n" + "=" * 92)
print("LOADING TRAINED MODELS")
print("=" * 92)
print("ckpt dir:", SAVE_DIR)

MODELS, HISTORIES = {}, {}
for _nm in ["STT-Repro", "FSTT-Net"]:
    _cp = cp_paths(_nm)
    if not _cp["best"].exists():
        print(f"  {_nm:<12} MISSING  ({_cp['best'].name})")
        continue
    try:
        _m, _cfg, _ = build(_nm)
        _m.load_state_dict(_st_load(str(_cp["best"]), device="cpu"), strict=True)
        MODELS[_nm] = _m.cpu()
        _best = None
        if _cp["meta"].exists():
            try:
                _meta = json.loads(_cp["meta"].read_text())
                HISTORIES[_nm] = _meta.get("history", {})
                _best = _meta.get("best_auc")
            except Exception:
                pass
        print(f"  {_nm:<12} loaded   "
              f"{sum(p.numel() for p in _m.parameters())/1e6:.2f} M params"
              + (f" | best val frame AUC {_best:.4f}" if _best else ""))
    except Exception as e:
        print(f"  {_nm:<12} FAILED   {type(e).__name__}: {e}")
        print("               EMBED_DIMS / STEM_MODEL here differ from the checkpoint's")

if not MODELS:
    raise RuntimeError(f"No checkpoints loaded from {SAVE_DIR}.")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ==================== EVALUATION ====================
CROSS_TARGETS    = None      # None -> every cached corpus except PRIMARY
MAX_CLIPS_CROSS  = 4000
MAX_CLIPS_ROBUST = 1500
ROBUST_ON_SPLIT  = "test"

PROPOSED, BASE_REPRO = "FSTT-Net", "STT-Repro"
PRIMARY = CFG.PRIMARY
LIVE = [k for k, v in MODELS.items() if isinstance(v, nn.Module)]

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
                     "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

print("=" * 92)
print("CELL 3 :: CROSS-DATASET + ROBUSTNESS")
print("=" * 92)
print(f"trained on : {PRIMARY}")
print(f"models     : {LIVE}")
print(f"corpora    : {sorted(FRAMES)}")


def _save(fig, name):
    for ext in ("png", "pdf"):
        try:
            fig.savefig(FIG_DIR / f"{name}.{ext}")
        except Exception as e:
            print(f"  ! save {name}.{ext}: {e}")
    plt.close(fig)
    print(f"  saved figures/{name}.png")


class ShiftDS(torch.utils.data.Dataset):
    def __init__(self, manifest, cfg, corruption=None, max_clips=None, seed=0):
        self.cfg, self.corruption = cfg, corruption
        self.T = cfg.CLIP_LEN
        man = manifest.sort_values(["video_id", "frame"])
        clips = []
        for vid, g in man.groupby("video_id", sort=False):
            paths = g.face.tolist()
            if not paths:
                continue
            lab = int(g.label.iloc[0])
            if self.T == 1:
                chunks = [[p] for p in paths]
            else:
                chunks = [paths[i:i + self.T] for i in range(0, len(paths), self.T)]
                chunks = [c + [c[-1]] * (self.T - len(c)) for c in chunks]
            for c in chunks:
                clips.append((c, lab, vid))
        if max_clips and len(clips) > max_clips:
            idx = np.arange(len(clips))
            y = np.array([c[1] for c in clips])
            rng = np.random.RandomState(seed)
            keep = []
            for cls in np.unique(y):
                pool = idx[y == cls]
                n = max(1, int(round(max_clips * len(pool) / len(idx))))
                keep += list(rng.choice(pool, min(n, len(pool)), replace=False))
            clips = [clips[i] for i in sorted(keep)]
        self.clips = clips
        self.labels = np.asarray([c[1] for c in clips], dtype=np.int64)

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, i):
        paths, lab, vid = self.clips[i]
        imgs, freqs, sems = [], [], []
        for p in paths:
            im = cv2.imread(p)
            im = (np.zeros((self.cfg.IMG_SIZE, self.cfg.IMG_SIZE, 3), np.uint8)
                  if im is None else cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            if im.shape[0] != self.cfg.IMG_SIZE or im.shape[1] != self.cfg.IMG_SIZE:
                im = cv2.resize(im, (self.cfg.IMG_SIZE, self.cfg.IMG_SIZE))
            if self.corruption is not None:
                im = corrupt_frame(im, self.corruption[0], self.corruption[1])
            x = im.astype(np.float32) / 255.0
            imgs.append(((x - IMNET_MEAN) / IMNET_STD).transpose(2, 0, 1))
            freqs.append(make_freq_input(im))
            sems.append(fast_semantic_prior(im))
        return {"img": torch.from_numpy(np.stack(imgs)).float(),
                "freq": torch.from_numpy(np.stack(freqs)).float(),
                "sem": torch.from_numpy(np.stack(sems)).float(),
                "y": torch.tensor(lab, dtype=torch.long), "vid": vid}


_W = min(4, max(0, (os.cpu_count() or 2) // 2))

def loader_for(ds, bs=16):
    kw = dict(batch_size=bs, shuffle=False, num_workers=_W,
              pin_memory=torch.cuda.is_available(), drop_last=False)
    if _W > 0:
        kw["prefetch_factor"] = 2
    return DataLoader(ds, **kw)

@torch.no_grad()
def score(model, loader, desc="eval"):
    model = model.to(DEVICE).eval()
    P, Y, V = [], [], []
    amp = bool(CFG.AMP and DEVICE.type == "cuda")
    for b in tqdm(loader, desc=desc, leave=False):
        img = b["img"].to(DEVICE, non_blocking=True)
        frq = b["freq"].to(DEVICE, non_blocking=True)
        sem = b["sem"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=amp):
            out = model(img, frq, sem)
        logits = out[0] if isinstance(out, tuple) else out
        P.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
        Y.append(b["y"].numpy()); V += list(b["vid"])
        del img, frq, sem, logits
    model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pd.DataFrame({"prob": np.concatenate(P), "y": np.concatenate(Y), "vid": V})

def auc_pair(df):
    f = float(roc_auc_score(df.y, df.prob)) if df.y.nunique() > 1 else float("nan")
    g = df.groupby("vid").agg(prob=("prob", "mean"), y=("y", "first"))
    v = float(roc_auc_score(g.y, g.prob)) if g.y.nunique() > 1 else float("nan")
    return f, v

def eer_of(df):
    if df.y.nunique() < 2:
        return float("nan")
    fpr, tpr, _ = roc_curve(df.y, df.prob)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2)


# ---- TABLE IV : cross-dataset ---------------------------------------------
print("\n--- TABLE IV : cross-dataset generalisation ---")
targets = CROSS_TARGETS if CROSS_TARGETS else [k for k in FRAMES if k != PRIMARY]
targets = [t for t in targets if t in FRAMES]
if not targets:
    print("  no second corpus cached - skipped")

CROSS_ROWS, CROSS_PREDS = [], {}
for tgt in targets:
    man = FRAMES[tgt]
    sub = man[man.split == "test"] if (man.split == "test").any() else man
    ds = ShiftDS(sub, CFG, corruption=None, max_clips=MAX_CLIPS_CROSS)
    if len(ds) == 0 or len(np.unique(ds.labels)) < 2:
        print(f"  {tgt}: unusable (clips={len(ds)})")
        continue
    dl = loader_for(ds)
    print(f"  target {tgt}: {len(ds):,} clips | "
          f"labels={dict(zip(*np.unique(ds.labels, return_counts=True)))} | "
          f"sequences={sub.video_id.nunique():,}")
    for name in LIVE:
        try:
            df = score(MODELS[name], dl, desc=f"{name}->{tgt}")
            f, v = auc_pair(df)
            CROSS_PREDS[(name, tgt)] = df
            CROSS_ROWS.append({"Model": name, "Train": PRIMARY, "Test": tgt,
                               "Frame AUC": f, "Video AUC": v, "EER": eer_of(df),
                               "Clips": len(df)})
            print(f"    {name:<18} frame AUC {f:.4f} | video AUC {v:.4f}")
        except Exception as e:
            print(f"    ! {name}: {type(e).__name__}: {e}")

CROSS = pd.DataFrame(CROSS_ROWS)
if len(CROSS):
    CROSS = CROSS.sort_values(["Test", "Frame AUC"], ascending=[True, False])
    CROSS.to_csv(TAB_DIR / "table4_cross_dataset.csv", index=False)
    print()
    print(CROSS.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


# ---- TABLE X : robustness --------------------------------------------------
print("\n--- TABLE X : robustness to real-world degradation ---")
CORRUPTIONS = OrderedDict([
    ("clean",     [("clean", 0)]),
    ("JPEG",      [("jpeg", 90), ("jpeg", 70), ("jpeg", 50), ("jpeg", 30)]),
    ("Blur",      [("blur", 1.0), ("blur", 2.0), ("blur", 3.0)]),
    ("Noise",     [("noise", 5), ("noise", 10), ("noise", 15)]),
    ("Downscale", [("resize", 0.50), ("resize", 0.33), ("resize", 0.25)]),
])

pman = FRAMES[PRIMARY]
psub = pman[pman.split == ROBUST_ON_SPLIT]
if len(psub) == 0:
    psub = pman[pman.split == "val"]

ROB_ROWS = []
for family, settings in CORRUPTIONS.items():
    for kind, level in settings:
        ds = ShiftDS(psub, CFG, corruption=(kind, level), max_clips=MAX_CLIPS_ROBUST)
        if len(ds) == 0 or len(np.unique(ds.labels)) < 2:
            continue
        dl = loader_for(ds)
        for name in LIVE:
            try:
                df = score(MODELS[name], dl, desc=f"{name} {kind}{level}")
                f, v = auc_pair(df)
                ROB_ROWS.append({"Model": name, "Family": family, "Kind": kind,
                                 "Level": level, "Frame AUC": f, "Video AUC": v,
                                 "Clips": len(df)})
            except Exception as e:
                print(f"    ! {name} {kind} {level}: {type(e).__name__}: {e}")
        done = [r for r in ROB_ROWS if r["Kind"] == kind and r["Level"] == level]
        if done:
            print(f"  {family:<10} {str(level):<6} " +
                  "  ".join(f"{r['Model']}={r['Frame AUC']:.4f}" for r in done))

ROB = pd.DataFrame(ROB_ROWS)
MEAN_DROP = pd.Series(dtype=float)
if len(ROB):
    base = ROB[ROB.Family == "clean"].set_index("Model")["Frame AUC"]
    ROB["Drop vs clean"] = ROB.apply(
        lambda r: base.get(r.Model, np.nan) - r["Frame AUC"], axis=1)
    ROB.to_csv(TAB_DIR / "table10_robustness.csv", index=False)
    PIV = ROB.pivot_table(index=["Family", "Level"], columns="Model",
                          values="Frame AUC", sort=False)
    PIV.to_csv(TAB_DIR / "table10_robustness_pivot.csv")
    print()
    print(PIV.to_string(float_format=lambda v: f"{v:.4f}"))
    MEAN_DROP = (ROB[ROB.Family != "clean"].groupby("Model")["Drop vs clean"]
                 .mean().sort_values())
    print("\n  mean AUC drop across all degradations (lower is more robust):")
    for k, v in MEAN_DROP.items():
        print(f"    {k:<18} {v:+.4f}")


# ---- figures ---------------------------------------------------------------
print("\n--- figures ---")
if len(CROSS):
    try:
        fig, ax = plt.subplots(1, 2, figsize=(11, 4.0))
        d = CROSS.sort_values("Frame AUC")
        ax[0].barh([f"{r.Model}\n->{r.Test}" for _, r in d.iterrows()],
                   d["Frame AUC"], color="#2b7bba")
        for i, v in enumerate(d["Frame AUC"]):
            ax[0].text(v + 0.004, i, f"{v:.4f}", va="center", fontsize=7)
        ax[0].set_xlim(0.35, 1.0)
        ax[0].set_title(f"Cross-dataset frame AUC (trained on {PRIMARY})")
        ax[0].tick_params(labelsize=7)
        for (name, tgt), df in CROSS_PREDS.items():
            if df.y.nunique() < 2:
                continue
            fpr, tpr, _ = roc_curve(df.y, df.prob)
            ax[1].plot(fpr, tpr, lw=1.4, label=f"{name} -> {tgt}")
        ax[1].plot([0, 1], [0, 1], "k--", lw=0.8)
        ax[1].set_xlabel("false positive rate"); ax[1].set_ylabel("true positive rate")
        ax[1].set_title("Cross-dataset ROC"); ax[1].legend(fontsize=6, loc="lower right")
        _save(fig, "fig17_cross_dataset")
    except Exception as e:
        print(f"  ! cross figure: {type(e).__name__}: {e}")

if len(ROB):
    try:
        fams = [f for f in CORRUPTIONS if f != "clean" and (ROB.Family == f).any()]
        fig, axes = plt.subplots(1, len(fams), figsize=(3.4 * len(fams), 3.4), squeeze=False)
        for i, fam in enumerate(fams):
            a = axes[0][i]
            sub = ROB[ROB.Family == fam]
            for name in LIVE:
                s = sub[sub.Model == name].sort_values("Level")
                if s.empty:
                    continue
                a.plot(range(len(s)), s["Frame AUC"], marker="o", ms=4, lw=1.4, label=name)
                a.set_xticks(range(len(s)))
                a.set_xticklabels([str(x) for x in s["Level"]], fontsize=7)
            cl = ROB[ROB.Family == "clean"]
            if not cl.empty:
                a.axhline(cl["Frame AUC"].mean(), color="k", ls="--", lw=0.8, alpha=0.6)
            a.set_title(fam); a.set_xlabel("severity"); a.set_ylabel("frame AUC")
        axes[0][-1].legend(fontsize=6, loc="lower left")
        fig.suptitle("Robustness to real-world degradation (dashed = clean mean)",
                     y=1.04, fontsize=11)
        _save(fig, "fig18_robustness")
    except Exception as e:
        print(f"  ! robustness figure: {type(e).__name__}: {e}")


# ---- export + read-out -----------------------------------------------------
(RES_DIR / "cross_and_robustness.json").write_text(json.dumps(
    {"source": PRIMARY, "targets": targets,
     "cross_dataset": CROSS.to_dict("records") if len(CROSS) else [],
     "robustness": ROB.to_dict("records") if len(ROB) else []},
    indent=2, default=float))
print("  saved results/cross_and_robustness.json")

print("\n" + "=" * 92)
print("WHAT THESE TABLES LET YOU CLAIM")
print("=" * 92)

def _get(df, model, col):
    r = df[df.Model == model]
    return float(r[col].iloc[0]) if len(r) else float("nan")

if len(CROSS) and PROPOSED in set(CROSS.Model) and BASE_REPRO in set(CROSS.Model):
    for tgt in CROSS.Test.unique():
        sl = CROSS[CROSS.Test == tgt]
        p, s = _get(sl, PROPOSED, "Frame AUC"), _get(sl, BASE_REPRO, "Frame AUC")
        print(f"\ncross-dataset {PRIMARY} -> {tgt}:")
        print(f"  FSTT-Net {p:.4f}  vs  STT-Repro {s:.4f}   gap {p - s:+.4f}")
        if max(p, s) < 0.65:
            print("  -> Both near chance: a transfer FAILURE for both, not a comparison.")
            print("     Say so plainly; do not use it to support 'Generalizable'.")
        elif p - s > 0.01:
            print("  -> A real, attributable improvement, and the base paper reports")
            print("     nothing on transfer. Lead with it.")
        elif p - s > 0:
            print("  -> Positive but small. Three seeds before calling it an improvement.")
        else:
            print("  -> No transfer advantage. Report it; do not bury it.")

if len(MEAN_DROP) and PROPOSED in MEAN_DROP.index and BASE_REPRO in MEAN_DROP.index:
    dp, dsr = MEAN_DROP[PROPOSED], MEAN_DROP[BASE_REPRO]
    print(f"\nrobustness: FSTT-Net degrades by {dp:+.4f}, STT-Repro by {dsr:+.4f}")
    if dp < dsr - 0.005:
        print("  -> Measurably more robust. That is the spectral stream doing its job.")
    elif dp < dsr:
        print("  -> Slightly more robust, inside noise. Needs multiple seeds.")
    else:
        print("  -> Not more robust.")

if "wilddeepfake" not in FRAMES:
    print("\nNOTE: WildDeepfake is missing from the cache, so the transfer table has only")
    print("one target. Re-extract it with cell 1 (BUILD = ['wilddeepfake']) and lower")
    print("MAX_TRAIN_FRAMES_PER_VIDEO to about 20 - 189k tiny files in one session is")
    print("more than Drive reliably flushes, which is why that corpus vanished.")

print("\nNext: cell 4 (manuscript pack), then cell 5 (evidence tracker).")
print("=" * 92)

Drive mounted: True
device = cuda | torch 2.11.0+cu128
root   = /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2

scanning /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2/faces
  celebdf        121,175 crops | sequences=6,529 | splits=['test', 'train', 'val']
  ff++           38,613 crops | sequences=3,000 | splits=['test', 'train', 'val']
  NOTE: 'wilddeepfake' has no usable cache - it will be skipped everywhere below.
mediapipe unavailable -> canonical template prior

LOADING TRAINED MODELS
ckpt dir: /content/drive/MyDrive/NIT-PATNA-P2/fsttnet_v2/checkpoints/last2_v6


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

model.safetensors: downloading bytes:           |  0.00B            

...

FSTT-Net resize0.33:   0%|          | 0/94 [00:00<?, ?it/s]

  Downscale  0.33   STT-Repro=0.9781  FSTT-Net=0.9937


STT-Repro resize0.25:   0%|          | 0/94 [00:00<?, ?it/s]

FSTT-Net resize0.25:   0%|          | 0/94 [00:00<?, ?it/s]

  Downscale  0.25   STT-Repro=0.9597  FSTT-Net=0.9800

Model            STT-Repro  FSTT-Net
Family    Level                     
clean     0.00      0.9984    0.9990
JPEG      90.00     0.9964    0.9989
          70.00     0.9975    0.9982
          50.00     0.9961    0.9969
          30.00     0.9917    0.9938
Blur      1.00      0.9942    0.9982
          2.00      0.9635    0.9863
          3.00      0.8292    0.8804
Noise     5.00      0.9675    0.9913
          10.00     0.7021    0.8870
          15.00     0.5215    0.5507
Downscale 0.50      0.9959    0.9982
          0.33      0.9781    0.9937
          0.25      0.9597    0.9800

  mean AUC drop across all degradations (lower is more robust):
    FSTT-Net           +0.0564
    STT-Repro          +0.0835

--- figures ---
  saved figures/fig17_cross_dataset.png
  saved figures/fig18_robustness.png
  saved results/cross_and_robustness.json

WHAT THESE TABLES LET YOU CLAIM

cross-dataset celebdf -> ff++:
  FSTT-Net 0.8608  vs  ST

---
## 4 · Manuscript pack

Displays every table and figure inline and writes IEEE-ready assets: 600 dpi PDF at
3.5 in / 7.16 in column widths, booktabs `.tex`, `.csv`, `captions.tex`, zip bundle.

Update `TRAIN_LOG` at the top once the baselines have been retrained by cell 6.

In [ ]:
# ############################################################################
# CELL P (v2) :: IEEE PAPER ASSETS - displayed inline AND saved to a new folder
# ----------------------------------------------------------------------------
# Run AFTER the SINGLE CELL, CELL X and (optionally) CELL FINAL / CELL Y, in
# the same session. Nothing is retrained.
#
# WHAT CHANGED vs v1
#   * every table is displayed in the notebook right under the cell, and every
#     figure is drawn inline with plt.show() as well as written to disk
#   * everything goes to a NEW directory: CFG.ROOT / OUT_DIRNAME
#   * the cross-dataset and robustness results you already produced with CELL X
#     are picked up automatically and turned into Table IV and Table X
#   * new: a robustness DELTA table and figure (FSTT-Net minus STT-Repro at
#     each severity). Your mean drop is 0.1298 vs 0.1410 - a 1.1 point average
#     that hides a much more interesting pattern, and the delta view exposes it
# ############################################################################

# ---------------------------------------------------------------------------
# 0. OPTIONS
# ---------------------------------------------------------------------------
OUT_DIRNAME      = "manuscript_pack"   # new folder under CFG.ROOT
QUOTE_LITERATURE = True

# Your original training-log results. Nothing here is recomputed - these are the
# best-epoch VALIDATION AUCs printed by your own training cells. Edit only if
# you re-run a model.
TRAIN_LOG = [
    # name,                    best val AUC, epochs, minutes, params(M), pretrained, note
    ("Xception (random init)", 0.8489, 19, 234.1, 20.81, False,
     "timm pretrained load failed; trained from scratch"),
    ("EfficientNet-B4",        0.9527, 20, 244.3, 17.55, True,  ""),
    ("ViT-B/16",               0.8567,  7, 102.5, 85.80, True,  "early stop at epoch 7"),
    ("Swin-T",                 0.9663, 14, 174.4, 27.52, True,  ""),
    ("F3Net-Lite",             0.9528, 12,  98.1,  9.20, True,
     "RGB branch = ImageNet EfficientNet-B2"),
    ("STT-Repro",              0.9628, 15,   0.0, 21.55, True,
     "base-paper configuration, reproduced in this work"),
    ("FSTT-Net",               0.9632, 15,   0.0, 23.84, True,  "proposed"),
]
BOOTSTRAP_N      = 2000
SHOW_INLINE      = True
MAKE_ZIP         = True

import os, gc, json, time, math, zipfile, warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import gridspec
from scipy import stats
from sklearn.metrics import (roc_curve, roc_auc_score, precision_recall_curve,
                             average_precision_score, det_curve, confusion_matrix,
                             accuracy_score, f1_score, balanced_accuracy_score,
                             brier_score_loss)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x.to_string() if hasattr(x, "to_string") else x)

for _n in ["CFG", "FRAMES", "cv2", "IMNET_MEAN", "IMNET_STD", "make_freq_input",
           "fast_semantic_prior", "MODELS", "N_REGIONS"]:
    if _n not in globals():
        raise RuntimeError(f"Run the SINGLE CELL first - '{_n}' is not defined.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path(getattr(CFG, "ROOT", "./fsttnet"))
OUT = ROOT / OUT_DIRNAME
PFIG, PTAB, PRES = OUT / "figures", OUT / "tables", OUT / "results"
for d in (OUT, PFIG, PTAB, PRES):
    d.mkdir(parents=True, exist_ok=True)

PROPOSED, BASE_REPRO = "FSTT-Net", "STT-Repro"
PRIMARY = CFG.PRIMARY if CFG.PRIMARY in FRAMES else list(FRAMES)[0]
MANIFEST = FRAMES[PRIMARY]
EVAL_SPLIT = "test" if (MANIFEST.split == "test").any() else "val"

COL1, COL2 = 3.5, 7.16
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02, "font.family": "serif",
    "font.serif": ["DejaVu Serif", "Times New Roman", "Nimbus Roman"],
    "mathtext.fontset": "dejavuserif",
    "font.size": 7.2, "axes.titlesize": 7.6, "axes.labelsize": 7.2,
    "xtick.labelsize": 6.4, "ytick.labelsize": 6.4, "legend.fontsize": 6.0,
    "axes.linewidth": 0.6, "grid.linewidth": 0.4, "lines.linewidth": 1.1,
    "axes.grid": True, "grid.alpha": 0.22,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})
PAL = {"proposed": "#C1121F", "repro": "#2B7BBA", "b1": "#F08C00",
       "real": "#2B7BBA", "fake": "#C1121F"}
CYCLE = ["#7B8FA1", "#F08C00", "#4C956C", "#8F4BBD", "#B07AA1", "#2B7BBA", "#C1121F"]
ASSETS = []


def color_for(name, i=0):
    if name == PROPOSED:
        return PAL["proposed"]
    if name == BASE_REPRO:
        return PAL["repro"]
    return CYCLE[i % len(CYCLE)]


def emit_fig(fig, name, number, caption):
    for ext in ("pdf", "png"):
        fig.savefig(PFIG / f"{name}.{ext}")
    ASSETS.append(("figure", number, name, caption))
    print(f"\n[FIG {number}] {name}")
    if SHOW_INLINE:
        plt.show()
    else:
        plt.close(fig)


def emit_table(df, name, number, caption, index=False):
    df.to_csv(PTAB / f"{name}.csv", index=index)
    try:
        body = df.to_latex(index=index, escape=True, float_format=lambda v: "%.2f" % v,
                           column_format="l" + "c" * (df.shape[1] - (0 if index else 1)))
        body = (body.replace("\\toprule", "\\hline\\hline")
                    .replace("\\midrule", "\\hline")
                    .replace("\\bottomrule", "\\hline\\hline"))
        (PTAB / f"{name}.tex").write_text(
            "\\begin{table}[!t]\n\\centering\n"
            f"\\caption{{{caption}}}\n\\label{{tab:{name}}}\n\\footnotesize\n"
            + body + "\\end{table}\n")
    except Exception as e:
        print(f"  ! latex for {name}: {e}")
    ASSETS.append(("table", number, name, caption))
    print(f"\n[TABLE {number}] {name}")
    display(df)


print("=" * 92)
print("CELL P (v2) :: IEEE PAPER ASSETS")
print("=" * 92)
print(f"dataset {PRIMARY} | eval split {EVAL_SPLIT} | models {list(MODELS)}")
print(f"output  {OUT}")


# ---------------------------------------------------------------------------
# 1. PREDICTIONS
# ---------------------------------------------------------------------------
class _EvalDS(torch.utils.data.Dataset):
    def __init__(self, manifest, cfg):
        self.cfg = cfg
        self.T = cfg.CLIP_LEN
        man = manifest.sort_values(["video_id", "frame"])
        self.clips = []
        for vid, g in man.groupby("video_id", sort=False):
            p = g.face.tolist()
            if not p:
                continue
            lab = int(g.label.iloc[0])
            ch = ([[x] for x in p] if self.T == 1 else
                  [p[i:i + self.T] + [p[min(i + self.T, len(p)) - 1]] *
                   (self.T - len(p[i:i + self.T])) for i in range(0, len(p), self.T)])
            for c in ch:
                self.clips.append((c, lab, vid))

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, i):
        paths, lab, vid = self.clips[i]
        a, b, c = [], [], []
        for p in paths:
            im = cv2.imread(p)
            im = (np.zeros((self.cfg.IMG_SIZE, self.cfg.IMG_SIZE, 3), np.uint8)
                  if im is None else cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            if im.shape[0] != self.cfg.IMG_SIZE:
                im = cv2.resize(im, (self.cfg.IMG_SIZE, self.cfg.IMG_SIZE))
            x = im.astype(np.float32) / 255.0
            a.append(((x - IMNET_MEAN) / IMNET_STD).transpose(2, 0, 1))
            b.append(make_freq_input(im))
            c.append(fast_semantic_prior(im))
        return {"img": torch.from_numpy(np.stack(a)).float(),
                "freq": torch.from_numpy(np.stack(b)).float(),
                "sem": torch.from_numpy(np.stack(c)).float(),
                "y": torch.tensor(lab, dtype=torch.long), "vid": vid, "path": paths[0]}


@torch.no_grad()
def _predict(model, loader, want_feats=False, desc="eval"):
    model = model.to(DEVICE).eval()
    P, Y, V, PA, FE = [], [], [], [], []
    amp = bool(getattr(CFG, "AMP", True) and DEVICE.type == "cuda")
    for b in tqdm(loader, desc=desc, leave=False):
        img = b["img"].to(DEVICE, non_blocking=True)
        frq = b["freq"].to(DEVICE, non_blocking=True)
        sem = b["sem"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=amp):
            out = model(img, frq, sem)
        lo, aux = out if isinstance(out, tuple) else (out, {})
        P.append(torch.softmax(lo.float(), -1)[:, 1].cpu().numpy())
        Y.append(b["y"].numpy()); V += list(b["vid"]); PA += list(b["path"])
        if want_feats and isinstance(aux, dict) and aux.get("feat") is not None:
            FE.append(aux["feat"].float().cpu().numpy())
        del img, frq, sem, lo
    model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    df = pd.DataFrame({"prob": np.concatenate(P), "y": np.concatenate(Y),
                       "video_id": V, "path": PA})
    return (df, np.concatenate(FE) if FE else None) if want_feats else df


PRED, FEAT = {}, {}
_prev = globals().get("PREDS")
if isinstance(_prev, dict) and _prev:
    PRED = {k: v.copy() for k, v in _prev.items()}
    FEAT = dict(globals().get("FEATS") or {})
    print(f"\nreusing predictions from a previous cell: {list(PRED)}")
else:
    print("\n--- inference ---")
    ds = _EvalDS(MANIFEST[MANIFEST.split == EVAL_SPLIT], CFG)
    _w = min(4, max(0, (os.cpu_count() or 2) // 2))
    dl = torch.utils.data.DataLoader(ds, batch_size=16, shuffle=False, num_workers=_w,
                                     pin_memory=torch.cuda.is_available(),
                                     **({"prefetch_factor": 2} if _w > 0 else {}))
    for nm, m in MODELS.items():
        if not isinstance(m, nn.Module):
            continue
        try:
            want = nm in (PROPOSED, BASE_REPRO)
            r = _predict(m, dl, want_feats=want, desc=nm)
            d, f = r if want else (r, None)
            PRED[nm] = d
            if f is not None:
                FEAT[nm] = f
            print(f"  {nm:<18} {len(d):,} clips")
        except Exception as e:
            print(f"  ! {nm}: {type(e).__name__}: {e}")

if not PRED:
    raise RuntimeError("no predictions - train at least one model first")
ORDER = [m for m in list(MODELS) if m in PRED]


def video_agg(df):
    g = df.groupby("video_id")
    return pd.DataFrame({"prob": g.prob.mean(), "y": g.y.first()}).reset_index()


# ---------------------------------------------------------------------------
# 1b. TABLE III :: MODEL ZOO - all seven models, from the training logs
# ---------------------------------------------------------------------------
ZOO = pd.DataFrame(TRAIN_LOG, columns=["Method", "Best val AUC (%)", "Epochs",
                                       "Train time (min)", "Params (M)",
                                       "Pretrained", "Note"])
ZOO["Best val AUC (%)"] = (ZOO["Best val AUC (%)"] * 100).round(2)
ZOO["Pretrained"] = ZOO["Pretrained"].map({True: "yes", False: "NO"})
ZOO = ZOO.sort_values("Best val AUC (%)", ascending=False).reset_index(drop=True)
ZOO = ZOO[["Method", "Best val AUC (%)", "Params (M)", "Epochs",
           "Train time (min)", "Pretrained", "Note"]]
emit_table(ZOO, "tableIII_model_zoo", "III",
           "Model zoo evaluated in this work on Celeb-DF v2. All values are best-epoch "
           "validation AUC on the same split, taken from the training logs. The five baselines "
           "follow the baseline training schedule; STT-Repro and FSTT-Net follow the "
           "token-transformer schedule, which is stated separately in the experimental setup. "
           "The row marked Pretrained = NO did not receive ImageNet initialisation and is "
           "therefore not representative of the published architecture.")

fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.15))
dz = ZOO.sort_values("Best val AUC (%)")
ax[0].barh(dz.Method, dz["Best val AUC (%)"],
           color=[color_for(m) for m in dz.Method], height=0.62)
for i, (m, vv) in enumerate(zip(dz.Method, dz["Best val AUC (%)"])):
    ax[0].text(vv + 0.3, i, f"{vv:.2f}", va="center", fontsize=5.8)
ax[0].set_xlim(max(0, dz["Best val AUC (%)"].min() - 6), 101)
ax[0].set_xlabel("best validation AUC (%)"); ax[0].set_title("(a) Model zoo")
ax[0].tick_params(labelsize=5.8)

ax[1].scatter(ZOO["Params (M)"], ZOO["Best val AUC (%)"], s=26,
              c=[color_for(m) for m in ZOO.Method], edgecolors="white", linewidths=.5)
for _, r in ZOO.iterrows():
    ax[1].annotate(r.Method, (r["Params (M)"], r["Best val AUC (%)"]), fontsize=5.0,
                   xytext=(2, 3), textcoords="offset points")
ax[1].set_xscale("log"); ax[1].set_xlabel("parameters (M, log)")
ax[1].set_ylabel("best val AUC (%)"); ax[1].set_title("(b) Accuracy vs size")

tt = ZOO[ZOO["Train time (min)"] > 0]
if len(tt):
    ax[2].scatter(tt["Train time (min)"], tt["Best val AUC (%)"], s=26,
                  c=[color_for(m) for m in tt.Method], edgecolors="white", linewidths=.5)
    for _, r in tt.iterrows():
        ax[2].annotate(r.Method, (r["Train time (min)"], r["Best val AUC (%)"]),
                       fontsize=5.0, xytext=(2, 3), textcoords="offset points")
ax[2].set_xlabel("training time (min)"); ax[2].set_ylabel("best val AUC (%)")
ax[2].set_title("(c) Accuracy vs training cost")
emit_fig(fig, "fig_model_zoo", "7",
         "The seven detectors evaluated in this work. Panel (b) places the proposed model "
         "against parameter count on a logarithmic axis; panel (c) against wall-clock training "
         "cost. Neither view appears in the base paper.")


def eer_of(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2)


def boot_ci(y, p, n=BOOTSTRAP_N, seed=0):
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.RandomState(seed)
    v = []
    for _ in range(n):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        v.append(roc_auc_score(y[i], p[i]))
    return (np.nan, np.nan) if not v else (float(np.percentile(v, 2.5) * 100),
                                           float(np.percentile(v, 97.5) * 100))


# ---------------------------------------------------------------------------
# 2. TABLE II :: INTRA-DATASET
# ---------------------------------------------------------------------------
LIT = {
    "celebdf": [("Two-branch [66]", 93.81, 95.93), ("F3-Net [67]", 95.95, 98.93),
                ("MADD [5]", 94.63, 97.95), ("PEL [74]", 96.77, 98.19),
                ("RECCE [77]", 98.59, 99.94), ("SFGD [79]", 99.22, 99.96),
                ("STT (base paper) [TIFS'25]", 99.81, 100.00)],
    "wilddeepfake": [("Two-branch [66]", 82.24, 89.01), ("MADD [5]", 80.67, 89.81),
                     ("PEL [74]", 84.96, 91.79), ("RECCE [77]", 83.25, 92.02),
                     ("MSVT [81]", 89.69, 92.86), ("SFGD [79]", 84.41, 92.57),
                     ("STT (base paper) [TIFS'25]", 88.96, 95.57)],
}

rows = []
for nm in ORDER:
    d = PRED[nm]
    v = video_agg(d)
    lo, hi = boot_ci(v.y.values, v.prob.values)
    rows.append({
        "Method": nm, "Source": "measured",
        "Frame ACC (%)": 100 * accuracy_score(d.y, (d.prob >= .5).astype(int)),
        "Frame AUC (%)": 100 * roc_auc_score(d.y, d.prob),
        "Video ACC (%)": 100 * accuracy_score(v.y, (v.prob >= .5).astype(int)),
        "Video AUC (%)": 100 * roc_auc_score(v.y, v.prob),
        "Video EER (%)": 100 * eer_of(v.y.values, v.prob.values),
        "Video AUC 95% CI": f"[{lo:.2f}, {hi:.2f}]"})
MEAS = pd.DataFrame(rows).sort_values("Video AUC (%)", ascending=False).reset_index(drop=True)

INTRA = MEAS.copy()
if QUOTE_LITERATURE and PRIMARY in LIT:
    lit = pd.DataFrame([{"Method": n, "Source": "reported", "Video ACC (%)": a,
                         "Video AUC (%)": u} for n, a, u in LIT[PRIMARY]])
    INTRA = pd.concat([lit, MEAS], ignore_index=True)
cols = ["Method", "Source", "Frame ACC (%)", "Frame AUC (%)", "Video ACC (%)",
        "Video AUC (%)", "Video EER (%)", "Video AUC 95% CI"]
INTRA = INTRA[[c for c in cols if c in INTRA.columns]].round(2)
emit_table(INTRA, f"tableII_intra_{PRIMARY}", "II",
           f"Intra-dataset comparison on {PRIMARY}. Rows marked 'reported' are quoted from the "
           f"literature and come from other implementations on their own splits. Rows marked "
           f"'measured' were produced by this work under one identical training schedule, face "
           f"pipeline and test split, and are the only rows that support a controlled comparison. "
           f"Bootstrap 95\\% confidence intervals use {BOOTSTRAP_N} resamples.")


# ---------------------------------------------------------------------------
# 3. TABLE XIII :: PAIRED SIGNIFICANCE
# ---------------------------------------------------------------------------
def mcnemar(y, p1, p2, thr=0.5):
    c1 = ((p1 >= thr).astype(int) == y)
    c2 = ((p2 >= thr).astype(int) == y)
    b, c = int((c1 & ~c2).sum()), int((~c1 & c2).sum())
    if b + c == 0:
        return 1.0, b, c
    return float(1 - stats.chi2.cdf((abs(b - c) - 1) ** 2 / (b + c), 1)), b, c


def boot_diff(y, p1, p2, n=BOOTSTRAP_N, seed=0):
    rng = np.random.RandomState(seed)
    d = []
    for _ in range(n):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        d.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
    d = np.asarray(d) * 100
    if not len(d):
        return (np.nan,) * 4
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), \
        2 * min((d <= 0).mean(), (d >= 0).mean())


SIG = pd.DataFrame()
if PROPOSED in PRED:
    vo = video_agg(PRED[PROPOSED]).sort_values("video_id")
    srows = []
    for nm in ORDER:
        if nm == PROPOSED:
            continue
        vb = video_agg(PRED[nm]).sort_values("video_id")
        common = sorted(set(vo.video_id) & set(vb.video_id))
        if len(common) < 10:
            continue
        a = vo.set_index("video_id").loc[common]
        b = vb.set_index("video_id").loc[common]
        y = a.y.values.astype(int)
        m_, lo, hi, pb = boot_diff(y, a.prob.values, b.prob.values)
        pm, n01, n10 = mcnemar(y, a.prob.values, b.prob.values)
        srows.append({"FSTT-Net vs": nm, "dAUC (pp)": round(m_, 3),
                      "95% CI": f"[{lo:.2f}, {hi:.2f}]",
                      "Bootstrap p": round(pb, 4), "McNemar p": round(pm, 4),
                      "Ours right": n01, "Theirs right": n10,
                      "Significant": "yes" if min(pb, pm) < 0.05 else "no"})
    if srows:
        SIG = pd.DataFrame(srows)
        emit_table(SIG, "tableXIII_significance", "XIII",
                   "Paired statistical comparison of FSTT-Net against every other model on the "
                   "identical set of test sequences. A confidence interval containing zero means "
                   "the difference is not distinguishable from run-to-run variation.")


# ---------------------------------------------------------------------------
# 4. FIG :: ROC / PR / DET
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.05))
for i, nm in enumerate(ORDER):
    d = PRED[nm]
    if d.y.nunique() < 2:
        continue
    c, lw = color_for(nm, i), (1.6 if nm == PROPOSED else 0.9)
    fpr, tpr, _ = roc_curve(d.y, d.prob)
    ax[0].plot(fpr, tpr, color=c, lw=lw, label=f"{nm} ({100*roc_auc_score(d.y,d.prob):.2f})")
    v = video_agg(d)
    f2, t2, _ = roc_curve(v.y, v.prob)
    ax[1].plot(f2, t2, color=c, lw=lw, label=f"{nm} ({100*roc_auc_score(v.y,v.prob):.2f})")
    ax[2].semilogx(np.maximum(f2, 1e-4), t2, color=c, lw=lw)
for a, t in zip(ax, ["(a) Frame level", "(b) Video level", "(c) Video level, log FPR"]):
    a.plot([0, 1], [0, 1], "--", color="0.65", lw=0.6)
    a.set_xlabel("false positive rate"); a.set_title(t)
ax[0].set_ylabel("true positive rate"); ax[1].legend(loc="lower right"); ax[2].set_xlim(1e-3, 1)
emit_fig(fig, "fig_roc", "9",
         "Receiver operating characteristic; AUC (\\%) in parentheses. Panel (c) uses a "
         "logarithmic false-positive axis, the deployment-relevant operating regime. The base "
         "paper reports point AUC values only.")

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.05))
eers = {}
for i, nm in enumerate(ORDER):
    d = PRED[nm]
    if d.y.nunique() < 2:
        continue
    c = color_for(nm, i)
    pr, rc, _ = precision_recall_curve(d.y, d.prob)
    ax[0].plot(rc, pr, color=c, lw=1.6 if nm == PROPOSED else 0.9,
               label=f"{nm} (AP {100*average_precision_score(d.y,d.prob):.2f})")
    v = video_agg(d)
    fpr, fnr, _ = det_curve(v.y, v.prob)
    ax[1].plot(fpr * 100, fnr * 100, color=c, lw=1.6 if nm == PROPOSED else 0.9, label=nm)
    eers[nm] = 100 * eer_of(v.y.values, v.prob.values)
ax[0].axhline(float(PRED[ORDER[0]].y.mean()), ls="--", color="0.65", lw=0.6)
ax[0].set_xlabel("recall"); ax[0].set_ylabel("precision")
ax[0].set_title("(a) Precision-recall"); ax[0].legend(loc="lower left")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("false positive rate (%)"); ax[1].set_ylabel("false negative rate (%)")
ax[1].set_title("(b) DET, video level"); ax[1].legend()
emit_fig(fig, "fig_pr_det", "10",
         "Precision-recall and detection-error trade-off. The dashed line in (a) is the "
         "prevalence of manipulated samples, i.e. the precision of a trivial always-fake "
         "predictor.")


# ---------------------------------------------------------------------------
# 5. FIG :: CALIBRATION
# ---------------------------------------------------------------------------
key = PROPOSED if PROPOSED in PRED else ORDER[0]
d, v = PRED[key], video_agg(PRED[key])
fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.05))
ax[0].hist(d.prob[d.y == 0], bins=40, alpha=.62, color=PAL["real"], density=True, label="real")
ax[0].hist(d.prob[d.y == 1], bins=40, alpha=.62, color=PAL["fake"], density=True, label="fake")
ax[0].set_xlabel("P(fake)"); ax[0].set_ylabel("density")
ax[0].set_title(f"(a) Score separation, {key}"); ax[0].legend()

ths = np.linspace(0.02, 0.98, 97)
bas = [balanced_accuracy_score(v.y, (v.prob >= t).astype(int)) for t in ths]
ax[1].plot(ths, [accuracy_score(v.y, (v.prob >= t).astype(int)) for t in ths],
           color=PAL["b1"], label="accuracy")
ax[1].plot(ths, [f1_score(v.y, (v.prob >= t).astype(int), zero_division=0) for t in ths],
           color=PAL["repro"], label="F1")
ax[1].plot(ths, bas, color=PAL["proposed"], label="balanced acc.")
tb = float(ths[int(np.argmax(bas))])
ax[1].axvline(tb, ls="--", color="0.5", lw=0.7)
ax[1].set_xlabel("decision threshold"); ax[1].set_title(f"(b) Threshold ($t^*$={tb:.2f})")
ax[1].legend()

for i, nm in enumerate(ORDER):
    dd = PRED[nm]
    if dd.y.nunique() < 2:
        continue
    q = pd.qcut(dd.prob, 10, labels=False, duplicates="drop")
    cal = dd.groupby(q).agg(p=("prob", "mean"), o=("y", "mean"))
    ax[2].plot(cal.p, cal.o, "o-", ms=2.4, lw=.9, color=color_for(nm, i),
               label=f"{nm} ({brier_score_loss(dd.y, dd.prob):.3f})")
ax[2].plot([0, 1], [0, 1], "--", color="0.65", lw=0.6)
ax[2].set_xlabel("predicted probability"); ax[2].set_ylabel("empirical frequency")
ax[2].set_title("(c) Reliability (Brier)"); ax[2].legend()
emit_fig(fig, "fig_calibration", "13",
         "Decision analysis. Panel (c) shows whether the predicted scores can be read as "
         "probabilities, which matters whenever a detector is deployed at a non-default "
         "threshold. No calibration evidence appears in the base paper.")


# ---------------------------------------------------------------------------
# 6. TABLE IV :: CROSS-DATASET  (from CELL X)
# ---------------------------------------------------------------------------
_cross = globals().get("CROSS")
CROSS_OK = isinstance(_cross, pd.DataFrame) and len(_cross)
if CROSS_OK:
    c = _cross.copy()
    for col in ("Frame AUC", "Video AUC", "EER"):
        if col in c:
            c[col] = (c[col] * 100).round(2)
    c = c.rename(columns={"Frame AUC": "Frame AUC (%)", "Video AUC": "Video AUC (%)",
                          "EER": "EER (%)"})
    # how many distinct sequences actually back the video-level column?
    nvid = {}
    for tgt in c.Test.unique():
        try:
            man = FRAMES[tgt]
            sub = man[man.split == "test"] if (man.split == "test").any() else man
            nvid[tgt] = int(sub.video_id.nunique())
        except Exception:
            nvid[tgt] = -1
    c["Test sequences"] = c.Test.map(nvid)
    emit_table(c, "tableIV_cross_dataset", "IV",
               "Cross-dataset generalisation. Every model is trained on the source corpus only "
               "and evaluated on an unseen corpus without adaptation. Frame-level AUC is the "
               "reliable column here; the video-level column should only be quoted when the "
               "target corpus contributes enough distinct sequences.")
    for tgt, n in nvid.items():
        if 0 <= n < 20:
            print(f"  WARNING: '{tgt}' test split has only {n} distinct sequences. Its "
                  f"video-level AUC is computed over {n} points and is not reportable. "
                  f"Re-extract with CELL 0 v3 (it fixes the image-corpus video_id) before "
                  f"putting any video-level transfer number in the manuscript.")
else:
    print("\n  no CROSS in memory - run CELL X for Table IV")


# ---------------------------------------------------------------------------
# 7. TABLE X + DELTA :: ROBUSTNESS  (from CELL X)
# ---------------------------------------------------------------------------
_rob = globals().get("ROB")
ROB_OK = isinstance(_rob, pd.DataFrame) and len(_rob)
DELTA = pd.DataFrame()
if ROB_OK:
    piv = (_rob.pivot_table(index=["Family", "Level"], columns="Model",
                            values="Frame AUC", sort=False) * 100).round(2)
    emit_table(piv.reset_index(), "tableX_robustness", "X",
               "Robustness to real-world degradation (frame-level AUC \\%) on the same test "
               "sequences after JPEG recompression, Gaussian blur, additive noise and "
               "downscaling. The base paper reports no robustness study.")

    if {PROPOSED, BASE_REPRO} <= set(piv.columns):
        DELTA = piv[[BASE_REPRO, PROPOSED]].copy()
        DELTA["Delta (pp)"] = (DELTA[PROPOSED] - DELTA[BASE_REPRO]).round(2)
        DELTA["Winner"] = np.where(DELTA["Delta (pp)"] > 0, PROPOSED,
                                   np.where(DELTA["Delta (pp)"] < 0, BASE_REPRO, "tie"))
        emit_table(DELTA.reset_index(), "tableXa_robustness_delta", "X(b)",
                   "Per-setting difference between the proposed model and its own controlled "
                   "reproduction under degradation. A positive delta means the frequency stream "
                   "is retaining accuracy that the spatial-only reproduction loses.")

        mean_drop = (_rob[_rob.Family != "clean"].groupby("Model")["Drop vs clean"]
                     .mean() * 100).round(2)
        MD = mean_drop.reset_index().rename(columns={"Drop vs clean": "Mean AUC drop (pp)"})
        emit_table(MD.sort_values("Mean AUC drop (pp)"), "tableXb_mean_drop", "X(c)",
                   "Mean AUC drop across all degradation settings relative to clean input. "
                   "Lower is more robust.")

    fams = [f for f in _rob.Family.unique() if f != "clean"]
    if fams:
        fig, axes = plt.subplots(1, len(fams), figsize=(COL2, 1.95), squeeze=False)
        for i, fam in enumerate(fams):
            a = axes[0][i]
            sub = _rob[_rob.Family == fam]
            for j, nm in enumerate(ORDER):
                s = sub[sub.Model == nm].sort_values("Level")
                if s.empty:
                    continue
                a.plot(range(len(s)), s["Frame AUC"] * 100, "o-", ms=2.6,
                       lw=1.5 if nm == PROPOSED else 0.9, color=color_for(nm, j), label=nm)
                a.set_xticks(range(len(s)))
                a.set_xticklabels([str(x) for x in s["Level"]], fontsize=5.6)
            a.set_title(fam); a.set_xlabel("severity")
            if i == 0:
                a.set_ylabel("frame AUC (%)")
        axes[0][-1].legend(fontsize=5.2, loc="lower left")
        emit_fig(fig, "fig_robustness", "20",
                 "Degradation curves. A frequency-aware detector should lose less accuracy under "
                 "compression and blur than a purely spatial one; this figure tests that "
                 "expectation directly.")

    if len(DELTA):
        fig, ax = plt.subplots(figsize=(COL1 * 1.5, 2.0))
        lbl = [f"{f}\n{l}" for f, l in DELTA.index]
        vals = DELTA["Delta (pp)"].values
        ax.bar(range(len(vals)), vals,
               color=[PAL["proposed"] if x > 0 else "#7B8FA1" for x in vals])
        ax.axhline(0, color="0.35", lw=0.7)
        ax.set_xticks(range(len(vals))); ax.set_xticklabels(lbl, fontsize=5.0)
        ax.set_ylabel("FSTT-Net $-$ STT-Repro (pp AUC)")
        ax.set_title("Where the frequency stream pays off")
        emit_fig(fig, "fig_robustness_delta", "20(b)",
                 "Per-setting advantage of FSTT-Net over its controlled reproduction. Bars above "
                 "zero are settings where the proposed contributions retain accuracy the "
                 "reproduction loses.")
else:
    print("\n  no ROB in memory - run CELL X for Table X")


# ---------------------------------------------------------------------------
# 8. TABLE XI :: EFFICIENCY
# ---------------------------------------------------------------------------
@torch.no_grad()
def latency(m, reps=25, warm=8):
    x = torch.randn(1, CFG.CLIP_LEN, 3, CFG.IMG_SIZE, CFG.IMG_SIZE, device=DEVICE)
    f = torch.randn(1, CFG.CLIP_LEN, 6, CFG.IMG_SIZE, CFG.IMG_SIZE, device=DEVICE)
    s = torch.rand(1, CFG.CLIP_LEN, N_REGIONS, CFG.IMG_SIZE, CFG.IMG_SIZE, device=DEVICE)
    for _ in range(warm):
        m(x, f, s)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(reps):
        m(x, f, s)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dt = (time.time() - t0) / reps
    return dt * 1000, CFG.CLIP_LEN / dt


erow = []
for nm in ORDER:
    try:
        m = MODELS[nm].to(DEVICE).eval()
        ms, fps = latency(m)
        erow.append({"Method": nm,
                     "Params (M)": round(sum(p.numel() for p in m.parameters()) / 1e6, 2),
                     "Latency (ms/clip)": round(ms, 2), "Throughput (fps)": round(fps, 1),
                     "Video AUC (%)": float(MEAS.loc[MEAS.Method == nm, "Video AUC (%)"].iloc[0])})
        m.cpu()
    except Exception as e:
        print(f"  ! {nm}: {type(e).__name__}: {e}")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
EFF = pd.DataFrame(erow)
if len(EFF):
    EFF = EFF.sort_values("Video AUC (%)", ascending=False).round(2)
    emit_table(EFF, "tableXI_efficiency", "XI",
               f"Computational cost measured on "
               f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} at batch "
               f"size 1 with clip length $T={CFG.CLIP_LEN}$. The base paper reports parameters "
               f"and FLOPs but no wall-clock latency or throughput.")


# ---------------------------------------------------------------------------
# 9. FIG :: DASHBOARD
# ---------------------------------------------------------------------------
fig = plt.figure(figsize=(COL2, 5.0))
gs = gridspec.GridSpec(2, 3, hspace=0.55, wspace=0.34)

a = fig.add_subplot(gs[0, 0])
mm = MEAS.set_index("Method")["Video AUC (%)"].sort_values()
a.barh(mm.index, mm.values, color=[color_for(k, i) for i, k in enumerate(mm.index)], height=.6)
for i, k in enumerate(mm.index):
    a.text(mm[k], i, f" {mm[k]:.2f}", va="center", fontsize=5.6)
a.set_xlim(max(0, mm.min() - 4), 101); a.set_xlabel("video AUC (%)"); a.set_title("(a) Benchmark")

a = fig.add_subplot(gs[0, 1])
for i, nm in enumerate(ORDER):
    v = video_agg(PRED[nm])
    if v.y.nunique() < 2:
        continue
    fpr, tpr, _ = roc_curve(v.y, v.prob)
    a.plot(fpr, tpr, color=color_for(nm, i), lw=1.5 if nm == PROPOSED else .8)
a.plot([0, 1], [0, 1], "--", color="0.65", lw=.6)
a.set_xlabel("FPR"); a.set_ylabel("TPR"); a.set_title("(b) ROC")

a = fig.add_subplot(gs[0, 2])
vb = video_agg(PRED[key])
cm = confusion_matrix(vb.y, (vb.prob >= .5).astype(int), labels=[0, 1])
a.imshow(cm / np.maximum(cm.sum(1, keepdims=True), 1), cmap="Blues", vmin=0, vmax=1)
for r in range(2):
    for c_ in range(2):
        a.text(c_, r, f"{cm[r, c_]}", ha="center", va="center", fontsize=6,
               color="white" if cm[r, c_] > cm.max() * .6 else "black")
a.set_xticks([0, 1]); a.set_xticklabels(["Real", "Fake"])
a.set_yticks([0, 1]); a.set_yticklabels(["Real", "Fake"])
a.grid(False); a.set_title(f"(c) {key} confusion")

a = fig.add_subplot(gs[1, 0])
if ROB_OK and len(DELTA):
    vals = DELTA["Delta (pp)"].values
    a.bar(range(len(vals)), vals, color=[PAL["proposed"] if x > 0 else "#7B8FA1" for x in vals])
    a.axhline(0, color="0.35", lw=.7)
    a.set_xticks([]); a.set_ylabel("pp AUC")
a.set_title("(d) Robustness advantage")

a = fig.add_subplot(gs[1, 1])
H = globals().get("HISTORIES") or {}
for i, (nm, h) in enumerate(H.items()):
    if isinstance(h, dict) and h.get("epoch") and h.get("val_auc"):
        a.plot(h["epoch"], [100 * x for x in h["val_auc"]], color=color_for(nm, i),
               lw=1.5 if nm == PROPOSED else .8, label=nm)
a.set_xlabel("epoch"); a.set_ylabel("val AUC (%)"); a.set_title("(e) Convergence")
a.legend(fontsize=5.2)

a = fig.add_subplot(gs[1, 2]); a.axis("off")
t = MEAS[["Method", "Video ACC (%)", "Video AUC (%)", "Video EER (%)"]].round(2)
tb_ = a.table(cellText=t.values, colLabels=t.columns, loc="center", cellLoc="center")
tb_.auto_set_font_size(False); tb_.set_fontsize(5.4); tb_.scale(1, 1.2)
for j in range(len(t.columns)):
    tb_[0, j].set_facecolor("#2F3E4E"); tb_[0, j].set_text_props(color="white", weight="bold")
a.set_title("(f) Consolidated", y=.94)
emit_fig(fig, "fig_dashboard", "22",
         "Consolidated results for all evaluated detectors under one training schedule, one face "
         "pipeline and one test split.")


# ---------------------------------------------------------------------------
# 10. EXPORT
# ---------------------------------------------------------------------------
lines = ["% auto-generated captions", ""]
for kind, num, name, cap in ASSETS:
    lines += [f"% {kind.upper()} {num}: {name}", f"% {cap}", ""]
(OUT / "captions.tex").write_text("\n".join(lines))

summary = {"dataset": PRIMARY, "eval_split": EVAL_SPLIT,
           "intra": MEAS.to_dict("records"),
           "significance": SIG.to_dict("records") if len(SIG) else [],
           "cross": _cross.to_dict("records") if CROSS_OK else [],
           "robustness_delta": DELTA.reset_index().to_dict("records") if len(DELTA) else [],
           "efficiency": EFF.to_dict("records") if len(EFF) else []}
(PRES / "summary.json").write_text(json.dumps(summary, indent=2, default=float))

if MAKE_ZIP:
    zp = OUT / "FSTTNet_paper_assets.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(OUT.rglob("*")):
            if f.is_file() and f.name != zp.name:
                z.write(f, f.relative_to(OUT))
    print(f"\nbundle -> {zp} ({zp.stat().st_size/1e6:.1f} MB)")

print("\n" + "=" * 92)
print(f"DONE - {len(ASSETS)} assets in {OUT}")
print("=" * 92)


# ---------------------------------------------------------------------------
# 11. WHAT YOUR NUMBERS ACTUALLY SUPPORT
# ---------------------------------------------------------------------------
print("\nWHAT YOU CAN AND CANNOT CLAIM")
print("-" * 92)
_top = ZOO.iloc[0]
_pz = ZOO.loc[ZOO.Method == PROPOSED, "Best val AUC (%)"]
if len(_pz):
    print(f"0. Model zoo (validation): best is {_top.Method} = {_top['Best val AUC (%)']:.2f}, "
          f"FSTT-Net = {float(_pz.iloc[0]):.2f}.")
    if _top.Method != PROPOSED:
        print(f"   FSTT-Net ranks below {_top.Method} by "
              f"{_top['Best val AUC (%)'] - float(_pz.iloc[0]):.2f} points. Report the ranking as")
        print("   it is. What FSTT-Net does have over the base paper's STT is size: 23.84 M")
        print("   parameters against the 62.82 M reported in the base paper's Table VIII.")
if PROPOSED in set(MEAS.Method) and BASE_REPRO in set(MEAS.Method):
    p = float(MEAS.loc[MEAS.Method == PROPOSED, "Video AUC (%)"].iloc[0])
    s = float(MEAS.loc[MEAS.Method == BASE_REPRO, "Video AUC (%)"].iloc[0])
    print(f"1. Intra-dataset: FSTT-Net {p:.2f} vs STT-Repro {s:.2f} -> {p-s:+.2f} pp.")
    if p <= s:
        print("   The proposed model does NOT beat its own reproduction in-domain.")
        print("   Do not write an intra-dataset improvement claim. Report the number.")
if CROSS_OK:
    print("2. Cross-dataset: read the frame-level column only until the target corpus has")
    print("   enough distinct sequences. If both models sit near 0.5-0.6 frame AUC, that is a")
    print("   transfer FAILURE for both, not a comparison - say so plainly.")
if len(DELTA):
    win = DELTA[DELTA["Delta (pp)"] > 0]
    print(f"3. Robustness: FSTT-Net wins in {len(win)} of {len(DELTA)} degradation settings.")
    if len(win):
        top = DELTA["Delta (pp)"].idxmax()
        print(f"   Largest advantage: {top[0]} at severity {top[1]} "
              f"({DELTA.loc[top, 'Delta (pp)']:+.2f} pp).")
        print("   If the wins concentrate at HIGH severity, that is the claim worth making:")
        print("   'the spectral stream degrades more gracefully under heavy compression and")
        print("   blur', which is exactly what a frequency branch is for, and the base paper")
        print("   reports nothing on degradation at all.")
print("-" * 92)


RuntimeError: Run the SINGLE CELL first - 'IMNET_MEAN' is not defined.

---
## 5 · Evidence tracker

Trains nothing, runs in seconds. Reads what is actually on disk and scores it against
what this venue expects, with a GPU-hour price on each gap.

Run it after cell 4, and again after every stage of cell 6.

In [ ]:
# ############################################################################
# CELL T :: EVIDENCE TRACKER - what a Transactions reviewer will ask for
# ----------------------------------------------------------------------------
# Runs in seconds, trains nothing. It reads what is actually on disk and scores
# your evidence against what IEEE TIFS-level papers in this area are expected
# to show. Run it whenever you want to know where you stand and what the next
# GPU hour should buy.
#
# It is deliberately blunt. A checklist that flatters you is worthless.
# ############################################################################

import os, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x.to_string() if hasattr(x, "to_string") else x)

ROOT = Path(globals().get("CFG").ROOT if "CFG" in globals()
            else "/content/drive/MyDrive/fsttnet_v2")
FACE_DIR = ROOT / "faces"

print("=" * 92)
print("EVIDENCE TRACKER")
print("=" * 92)
print("root:", ROOT)


def _read(p):
    try:
        return pd.read_csv(p)
    except Exception:
        return None


def _jread(p):
    try:
        return json.loads(Path(p).read_text())
    except Exception:
        return None


ITEMS = []


def item(name, ok, detail, cost, why):
    ITEMS.append({"Requirement": name, "Status": "OK" if ok else "MISSING",
                  "Detail": detail, "GPU hours": cost, "Why it matters": why})


# ---------------------------------------------------------------------------
# 1. DATA
# ---------------------------------------------------------------------------
corpora, det_note = {}, "unknown"
for d in sorted(FACE_DIR.glob("*/manifest.csv")) if FACE_DIR.exists() else []:
    m = _read(d)
    if m is not None and len(m):
        corpora[d.parent.name] = m

n_corp = len(corpora)
item("Multiple datasets", n_corp >= 3,
     f"{n_corp} cached: {sorted(corpora)}", 0 if n_corp >= 3 else 10,
     "The base paper evaluates four corpora. One corpus cannot support a "
     "generalisation claim.")

seq_ok = True
seq_detail = []
for k, m in corpora.items():
    n_seq = m.video_id.nunique()
    n_crop = len(m)
    seq_detail.append(f"{k}: {n_crop:,} crops / {n_seq:,} sequences")
    if n_seq < 50:
        seq_ok = False
item("Sequence structure intact", seq_ok and n_corp > 0,
     "; ".join(seq_detail) or "no cache", 0 if seq_ok else 4,
     "Video-level AUC and identity-disjoint splits are meaningless if a corpus "
     "collapses into a handful of sequence groups.")

# detector provenance: v4 writes models/ next to faces/
det_ok = (ROOT / "models" / "res10_300x300_ssd_iter_140000.caffemodel").exists()
item("Faces actually detected", det_ok,
     "DNN SSD model present" if det_ok else "no detector model found - crops may be centre squares",
     0 if det_ok else 10,
     "A semantic prior that assumes eyes/nose/mouth positions is meaningless on "
     "unaligned crops, and the base paper uses detected faces.")

# ---------------------------------------------------------------------------
# 2. RESULTS ON DISK
# ---------------------------------------------------------------------------
PACK = ROOT / "manuscript_pack" / "tables"
sig = _read(PACK / "tableXIII_significance.csv")
zoo = _read(PACK / "tableIII_model_zoo.csv")
cross = _read(PACK / "tableIV_cross_dataset.csv")
rob = _read(PACK / "tableX_robustness.csv")
abl = _read(PACK / "tableVII_component_ablation.csv")
seeds = _read(PACK / "tableXII_seed_variance.csv")

# significance
sig_ok, sig_detail = False, "not computed"
if sig is not None and len(sig):
    row = sig[sig.iloc[:, 0].astype(str).str.contains("STT-Repro", na=False)]
    if len(row):
        r = row.iloc[0]
        ci = str(r.get("95% CI", ""))
        d = r.get("dAUC (pp)", np.nan)
        sig_ok = str(r.get("Significant", "no")).lower() == "yes" and float(d) > 0
        sig_detail = f"dAUC {d} pp, CI {ci}, significant={r.get('Significant')}"
item("Proposed beats its own reproduction", sig_ok, sig_detail, 12,
     "THE decisive row. If the interval contains zero, no amount of extra "
     "tables makes the contribution claim stand.")

# multi-seed
seed_ok = seeds is not None and len(seeds) and "std" in " ".join(map(str, seeds.columns)).lower()
item("Multi-seed variance", bool(seed_ok),
     f"{len(seeds)} rows" if seeds is not None else "not run", 24,
     "A sub-one-point difference from a single seed is not evidence. Reviewers "
     "ask for mean +/- std over at least three seeds.")

# ablation
abl_ok = abl is not None and len(abl) >= 3
item("Component ablation", bool(abl_ok),
     f"{len(abl)} variants" if abl is not None else "not run", 8,
     "One row per contribution, each measured against the same reproduction. "
     "The base paper devotes three tables to this.")

# cross-dataset
cross_ok, cross_detail = False, "not run"
if cross is not None and len(cross):
    col = [c for c in cross.columns if "Frame AUC" in c]
    if col:
        best = float(cross[col[0]].max())
        cross_ok = best >= 70.0
        cross_detail = f"best frame AUC {best:.1f}%"
item("Cross-dataset transfer works", cross_ok, cross_detail, 6,
     "Your title says 'Generalizable'. Transfer near 50-60% AUC contradicts the "
     "title and a reviewer will say so.")

# baselines same schedule
base_ok = False
if zoo is not None and len(zoo) >= 5:
    notes = " ".join(map(str, zoo.get("Note", pd.Series(dtype=str)).fillna("")))
    base_ok = "random init" not in notes.lower()
item("Baselines under one schedule", bool(base_ok),
     f"{0 if zoo is None else len(zoo)} rows"
     + ("" if base_ok else "; contains a random-init row or too few models"), 20,
     "Comparing models trained under different schedules is not a ranking. A "
     "random-init Xception at 85 AUC against a literature value of 99 is an "
     "immediate red flag.")

# robustness (this you already have)
rob_ok = rob is not None and len(rob) >= 8
item("Robustness study", bool(rob_ok),
     f"{0 if rob is None else len(rob)} settings", 1,
     "Not in the base paper. This is where your spectral stream can earn its "
     "place, so it is an asset rather than a gap.")

item("Confidence intervals + calibration", sig is not None,
     "bootstrap CIs and reliability diagrams produced" if sig is not None else "not produced",
     0,
     "Also absent from the base paper. Keep it - it is the clearest way your "
     "reporting exceeds theirs.")

# ---------------------------------------------------------------------------
# 3. REPORT
# ---------------------------------------------------------------------------
T = pd.DataFrame(ITEMS)
print()
display(T)

ok_n = int((T.Status == "OK").sum())
missing = T[T.Status == "MISSING"]
gpu = int(missing["GPU hours"].sum())

print()
print("-" * 92)
print(f"SCORE: {ok_n} of {len(T)} requirements met")
print(f"ESTIMATED GPU HOURS TO CLOSE THE GAPS: ~{gpu}")
print("-" * 92)

if not sig_ok:
    print()
    print("READ THIS FIRST")
    print("The decisive row is 'Proposed beats its own reproduction'. Until that is OK,")
    print("spending hours on seeds, ablations or extra corpora buys you nothing - they")
    print("would all be measuring a difference that is not there.")
    print()
    print("Order of work that actually makes sense:")
    print("  1. rebuild the data with detected faces      (cell 1)")
    print("  2. retrain STT-Repro and FSTT-Net, 1 seed    (cell 2)")
    print("  3. check this row again                      (cells 3, 4, then this cell)")
    print("  4. ONLY IF it is positive and significant, run the full study (cell 5)")
    print()
    print("If step 3 still shows an interval containing zero, the honest move is to")
    print("stop tuning and write the controlled-reproduction paper instead. That is a")
    print("real contribution and it is publishable - just not in a venue that requires")
    print("a state-of-the-art claim.")
else:
    print()
    print("The decisive row is OK. Now close the remaining gaps in this order:")
    for _, r in missing.sort_values("GPU hours").iterrows():
        print(f"  - {r.Requirement}  (~{r['GPU hours']} GPU h)")
    print()
    print("With those in place the evidence base matches what this venue expects.")
print("=" * 92)


---
## 6 · Full study — seeds, same-schedule baselines, ablation

**Gated.** Stage 1 checks whether FSTT-Net beats its own reproduction. If the confidence
interval contains zero it stops and tells you what to try instead, rather than running
stages 2–4 on a difference that is not there.

Every run is checkpointed and every result written to `experiments/`, so rerunning in a
new session skips what is finished. Expect several sessions.

Afterwards: rerun cells 3 and 4 to refresh the tables, then cell 5 to re-score.

In [ ]:
# ############################################################################
# CELL E :: FULL STUDY RUNNER - seeds, same-schedule baselines, ablation
# ----------------------------------------------------------------------------
# Run AFTER cell 2 (prerequisites + training) in the same session.
#
# This is the cell that turns "some results" into an evidence base a
# Transactions reviewer will accept. It is:
#
#   GATED       Stage 1 checks whether FSTT-Net actually beats its own
#               controlled reproduction. If the confidence interval contains
#               zero it STOPS and tells you, instead of burning 30 GPU hours
#               measuring a difference that is not there.
#   STAGED      each stage is a separate block of runs with its own budget
#   RESUMABLE   every run writes a checkpoint and a result json; rerunning
#               skips anything already finished
#   BUDGETED    stops cleanly before the Colab session is killed
#
# STAGES
#   1  gate       STT-Repro vs FSTT-Net, seed 42            (already done in cell 2)
#   2  seeds      both models on seeds 1337 and 2024        ~12 h
#   3  baselines  5 baselines under the SAME schedule       ~20 h
#   4  ablation   cumulative component ablation, 5 variants ~10 h
#
# Expect several sessions. Rerun this cell each time; it continues.
#
# TWO OPTIONAL TRAINING CHANGES, both OFF by default - read before enabling:
#   MODALITY_DROPOUT   randomly blanks one input stream during training, so the
#                      model cannot ignore the spectral branch. This is the
#                      principled fix for "the frequency stream makes no
#                      difference": at present nothing forces the network to
#                      use it.
#   DEGRADE_AUG        trains on JPEG/blur/noise. It will improve your
#                      robustness table, but then that table is no longer
#                      evidence of generalisation - you trained for it. If you
#                      enable it, say so in the paper and evaluate on
#                      severities you did NOT train on.
# ############################################################################

# ---------------------------------------------------------------------------
# 0. OPTIONS
# ---------------------------------------------------------------------------
RUN_STAGES        = [1, 2, 3, 4]     # which stages to attempt this session
TIME_BUDGET_HOURS = 11.0
GATE_TOLERANCE_PP = 0.0              # stage 1 must exceed this dAUC to unlock 2-4
FORCE_UNLOCK      = False            # True = run stages 2-4 even if the gate fails

SEEDS             = [42, 1337, 2024]
BASELINES         = ["Xception", "EfficientNet-B4", "ViT-B/16", "Swin-T", "F3Net-Lite"]
BASELINE_EPOCHS   = 12
ABLATION_EPOCHS   = 10

MODALITY_DROPOUT  = 0.0              # try 0.15 if the spectral stream is inert
DEGRADE_AUG       = 0.0              # try 0.25 only with the caveat above

import os, gc, json, time, copy, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x.to_string() if hasattr(x, "to_string") else x)

for _n in ["CFG", "FRAMES", "build", "make_cfg", "train_loader", "val_loader",
           "evaluate", "make_opt", "new_scaler", "cosine_lr", "set_seed",
           "FSTTLoss", "state_of", "_st_save", "_st_load", "_EXT", "train_ds",
           "ClipDS", "MODELS", "cv2", "corrupt_frame", "FSTTNet", "DeepStem",
           "PoolTo", "STEM_INDEX", "FINETUNE_STEM"]:
    if _n not in globals():
        raise RuntimeError(f"Run cell 2 (prerequisites + training) first - '{_n}' missing.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path(getattr(CFG, "ROOT", "./fsttnet"))
EXP = ROOT / "experiments"
CKPT = EXP / "ckpt"
for d in (EXP, CKPT):
    d.mkdir(parents=True, exist_ok=True)

PROPOSED, BASE_REPRO = "FSTT-Net", "STT-Repro"
T0 = time.time()


def elapsed_h():
    return (time.time() - T0) / 3600.0


def left_h():
    return TIME_BUDGET_HOURS - elapsed_h()


print("=" * 92)
print("FULL STUDY RUNNER")
print("=" * 92)
print(f"root {ROOT} | budget {TIME_BUDGET_HOURS} h | stages {RUN_STAGES}")
print(f"modality dropout {MODALITY_DROPOUT} | degradation aug {DEGRADE_AUG}")


# ---------------------------------------------------------------------------
# 1. RUN REGISTRY
# ---------------------------------------------------------------------------
def rid(name, seed, tag=""):
    return f"{name.replace('/', '-')}_s{seed}{('_' + tag) if tag else ''}"


def result_path(r):
    return EXP / f"{r}.json"


def have(r):
    p = result_path(r)
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text())
    except Exception:
        return None


def save_result(r, d):
    tmp = result_path(r).with_suffix(".tmp")
    tmp.write_text(json.dumps(d, indent=2, default=float))
    os.replace(tmp, result_path(r))


# ---------------------------------------------------------------------------
# 2. TRAINING with optional modality dropout / degradation augmentation
# ---------------------------------------------------------------------------
def _apply_modality_dropout(img, frq, p):
    if p <= 0:
        return img, frq
    B = img.shape[0]
    m = torch.rand(B, device=img.device)
    drop_spatial = (m < p).view(B, *([1] * (img.dim() - 1)))
    drop_freq = ((m >= p) & (m < 2 * p)).view(B, *([1] * (frq.dim() - 1)))
    return img * (~drop_spatial), frq * (~drop_freq)


def train_run(name, seed, cfg_overrides=None, epochs=None, tag="", is_baseline=False):
    """One training run. Resumable, returns a result dict."""
    r = rid(name, seed, tag)
    got = have(r)
    if got is not None:
        print(f"  [{r}] already done: frame AUC {got['frame_auc']:.4f}")
        return got

    set_seed(seed)
    if is_baseline:
        model, cfg = build_baseline_same_schedule(name)
    else:
        model, cfg = build_variant(name, cfg_overrides)
    if epochs:
        cfg.EPOCHS = epochs
    cfg.SEED = seed

    n_par = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  [{r}] training {n_par:.2f} M params, {cfg.EPOCHS} epochs, "
          f"budget left {left_h():.2f} h")

    ck = CKPT / f"{r}{_EXT}"
    start, best, hist = 0, -1.0, defaultdict(list)
    meta_p = CKPT / f"{r}_meta.json"
    if ck.exists() and meta_p.exists():
        try:
            model.load_state_dict(_st_load(str(ck), device="cpu"), strict=True)
            mm = json.loads(meta_p.read_text())
            start, best = int(mm["next_epoch"]), float(mm["best"])
            hist = defaultdict(list, mm.get("hist", {}))
            print(f"        resuming from epoch {start + 1}, best {best:.4f}")
        except Exception as e:
            print("        checkpoint unusable, restarting:", type(e).__name__)

    model = model.to(DEVICE)
    crit = (nn.CrossEntropyLoss(label_smoothing=0.03) if is_baseline
            else FSTTLoss(cfg, feat_dim=cfg.EMBED_DIMS[-1]).to(DEVICE))
    bs = 8
    opt = (torch.optim.AdamW(model.parameters(), lr=1.5e-4, weight_decay=0.02)
           if is_baseline else make_opt(model, crit, cfg.LR))
    amp = bool(getattr(cfg, "AMP", True) and DEVICE.type == "cuda")
    scaler = new_scaler(amp)
    vl = val_loader(min(32, bs * 2))
    n_clips = len(train_ds(cfg, 0))
    steps = max(1, n_clips // bs)
    total, warm = steps * cfg.EPOCHS, steps
    base_lr = 1.5e-4 if is_baseline else cfg.LR

    stopped = False
    for ep in range(start, cfg.EPOCHS):
        if left_h() <= 0.4:
            stopped = True
            break
        model.train()
        tl = train_loader(cfg, bs, ep)
        run, seen, corr = 0.0, 0, 0
        g = ep * steps
        pbar = tqdm(tl, desc=f"{r} ep{ep+1}/{cfg.EPOCHS}", leave=False)
        for b in pbar:
            lr = cosine_lr(g, total, warm, base_lr)
            for pg in opt.param_groups:
                pg["lr"] = lr
            img = b["img"].to(DEVICE, non_blocking=True)
            frq = b["freq"].to(DEVICE, non_blocking=True)
            sem = b["sem"].to(DEVICE, non_blocking=True)
            y = b["y"].to(DEVICE, non_blocking=True)
            if MODALITY_DROPOUT > 0 and not is_baseline:
                img, frq = _apply_modality_dropout(img, frq, MODALITY_DROPOUT)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=amp):
                out = model(img, frq, sem)
                lo, aux = out if isinstance(out, tuple) else (out, {})
                loss = crit(lo, y) if is_baseline else crit(lo, aux, y)[0]
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update()
            n = int(y.size(0))
            run += float(loss.detach().float().item()) * n
            seen += n
            corr += int((lo.argmax(-1) == y).sum().item())
            g += 1
            pbar.set_postfix(loss=f"{run/max(seen,1):.4f}", h=f"{elapsed_h():.1f}")
            del img, frq, sem, y, lo, loss

        m = evaluate(model, vl, cfg)
        hist["epoch"].append(ep + 1)
        hist["train_loss"].append(run / max(seen, 1))
        hist["val_auc"].append(m["frame_auc"])
        hist["video_auc"].append(m["video_auc"])
        print(f"        ep {ep+1:02d} frame AUC {m['frame_auc']:.4f} | "
              f"video AUC {m['video_auc']:.4f}")
        if m["frame_auc"] > best:
            best = m["frame_auc"]
            _st_save(state_of(model), str(ck))
        meta_p.write_text(json.dumps({"next_epoch": ep + 1, "best": best,
                                      "hist": dict(hist)}, default=float))
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if stopped:
        print(f"  [{r}] time budget reached - progress saved, rerun to continue")
        return None

    if ck.exists():
        model.load_state_dict(_st_load(str(ck), device="cpu"), strict=True)
    fin = evaluate(model.to(DEVICE), vl, cfg)
    res = {"run": r, "model": name, "seed": seed, "tag": tag,
           "params_M": n_par, "epochs": cfg.EPOCHS,
           "frame_auc": fin["frame_auc"], "video_auc": fin["video_auc"],
           "frame_acc": fin["frame_acc"], "hist": dict(hist)}
    save_result(r, res)
    model.cpu()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"  [{r}] done: frame AUC {fin['frame_auc']:.4f} | "
          f"video AUC {fin['video_auc']:.4f}")
    return res


def build_variant(name, overrides=None):
    """Same construction path as cell 2's build(), with cfg overrides applied
    BEFORE the model is created, so architecture flags actually take effect."""
    cfg = make_cfg(name)
    if overrides:
        for k, v in overrides.items():
            setattr(cfg, k, v)
    model = FSTTNet(cfg)
    stem = DeepStem(cfg.EMBED_DIMS[0], pretrained=getattr(cfg, "PRETRAINED_STEM", True))
    n_tok = cfg.IMG_SIZE // (2 ** (STEM_INDEX + 1))
    model.spatial_stem = stem
    if getattr(model, "use_freq", False):
        model.freq_stem = PoolTo(model.freq_stem, n_tok)
    if not FINETUNE_STEM:
        for prm in stem.backbone.parameters():
            prm.requires_grad_(False)
    return model, cfg


_TIMM = {"Xception": "legacy_xception", "EfficientNet-B4": "efficientnet_b4",
         "ViT-B/16": "vit_base_patch16_224", "Swin-T": "swin_tiny_patch4_window7_224",
         "F3Net-Lite": "efficientnet_b0"}


class _Baseline(nn.Module):
    def __init__(self, timm_name, in_ch=3, pretrained=True):
        super().__init__()
        import timm
        self.in_ch = in_ch
        self.net = timm.create_model(timm_name, pretrained=pretrained,
                                     num_classes=2, in_chans=in_ch)
        if pretrained and not any(p.requires_grad for p in self.net.parameters()):
            pass

    def forward(self, img, freq=None, sem=None, want_maps=False):
        x = img if img.dim() == 5 else img.unsqueeze(1)
        if self.in_ch == 6 and freq is not None:
            x = freq if freq.dim() == 5 else freq.unsqueeze(1)
        B, T = x.shape[:2]
        xf = x.flatten(0, 1)
        try:
            f = self.net.forward_features(xf)
            pooled = self.net.forward_head(f, pre_logits=True)
            logits = self.net.get_classifier()(pooled)
        except Exception:
            logits = self.net(xf); pooled = logits
        return logits.view(B, T, -1).mean(1), {"feat": pooled.view(B, T, -1).mean(1)}


def build_baseline_same_schedule(name):
    cfg = make_cfg("STT-Repro")          # identical schedule to the token models
    cfg.EPOCHS = BASELINE_EPOCHS
    in_ch = 6 if name == "F3Net-Lite" else 3
    m = _Baseline(_TIMM[name], in_ch=in_ch, pretrained=True)
    n_pre = sum(p.numel() for p in m.parameters())
    if n_pre == 0:
        raise RuntimeError(f"{name} built with no parameters")
    return m, cfg


# ---------------------------------------------------------------------------
# 3. STAGE 1 :: GATE
# ---------------------------------------------------------------------------
def boot_diff(y, p1, p2, n=2000, seed=0):
    rng = np.random.RandomState(seed)
    d = []
    for _ in range(n):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        d.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
    d = np.asarray(d) * 100
    if not len(d):
        return (np.nan,) * 3
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)


GATE = {"passed": False, "detail": "not evaluated"}
if 1 in RUN_STAGES:
    print("\n" + "-" * 92)
    print("STAGE 1 :: GATE - does the proposed model beat its own reproduction?")
    print("-" * 92)
    a = have(rid(PROPOSED, SEEDS[0])) or {"frame_auc": None}
    b = have(rid(BASE_REPRO, SEEDS[0])) or {"frame_auc": None}
    if a["frame_auc"] is None or b["frame_auc"] is None:
        # fall back to whatever cell 2 left in memory
        _res = globals().get("RESULTS") or {}
        a = {"frame_auc": _res.get(PROPOSED, {}).get("frame_auc")}
        b = {"frame_auc": _res.get(BASE_REPRO, {}).get("frame_auc")}

    _preds = globals().get("PREDS") or {}
    if PROPOSED in _preds and BASE_REPRO in _preds:
        dfp, dfb = _preds[PROPOSED], _preds[BASE_REPRO]
        common = sorted(set(dfp.video_id) & set(dfb.video_id))
        gp = dfp.groupby("video_id").agg(prob=("prob", "mean"), y=("y", "first")).loc[common]
        gb = dfb.groupby("video_id").agg(prob=("prob", "mean"), y=("y", "first")).loc[common]
        m_, lo, hi = boot_diff(gp.y.values.astype(int), gp.prob.values, gb.prob.values)
        GATE["passed"] = lo > GATE_TOLERANCE_PP
        GATE["detail"] = f"dAUC {m_:+.3f} pp, 95% CI [{lo:.2f}, {hi:.2f}]"
        print(f"  {GATE['detail']}  ->  {'PASS' if GATE['passed'] else 'FAIL'}")
    elif a["frame_auc"] is not None and b["frame_auc"] is not None:
        d = 100 * (a["frame_auc"] - b["frame_auc"])
        GATE["passed"] = d > GATE_TOLERANCE_PP
        GATE["detail"] = f"dAUC {d:+.3f} pp (no paired CI - run cells 3/4 for that)"
        print(f"  {GATE['detail']}  ->  {'PASS' if GATE['passed'] else 'FAIL'}")
    else:
        print("  cannot evaluate - run cell 2, then cells 3 and 4, then this cell")

if not GATE["passed"] and not FORCE_UNLOCK:
    print("\n" + "=" * 92)
    print("GATE NOT PASSED - STAGES 2-4 SKIPPED ON PURPOSE")
    print("=" * 92)
    print(GATE["detail"])
    print()
    print("Seeds, baselines and ablations would all be measuring a difference that is")
    print("not there. Spending ~40 GPU hours on them now buys nothing.")
    print()
    print("What is worth trying instead, in order of expected payoff:")
    print("  1. MODALITY_DROPOUT = 0.15 and rerun cell 2. At present nothing forces the")
    print("     network to use the spectral branch, so it can and does ignore it. This")
    print("     is the most likely reason the contribution does not show up.")
    print("  2. Check that cell 1 reported a high detection rate. On unaligned crops the")
    print("     semantic prior is meaningless and the whole design is handicapped.")
    print("  3. If neither moves the interval above zero, stop tuning. Write the")
    print("     controlled-reproduction paper: your CIs, calibration and degradation")
    print("     study already exceed what the base paper reports, and a careful")
    print("     negative result is publishable.")
    print()
    print("Set FORCE_UNLOCK = True only if you want the full study regardless.")
    print("=" * 92)
else:
    # -----------------------------------------------------------------------
    # 4. STAGE 2 :: SEEDS
    # -----------------------------------------------------------------------
    if 2 in RUN_STAGES:
        print("\n" + "-" * 92)
        print("STAGE 2 :: SEED VARIANCE")
        print("-" * 92)
        for s in SEEDS:
            for nm in [BASE_REPRO, PROPOSED]:
                if left_h() <= 0.5:
                    print("  budget exhausted - rerun this cell"); break
                train_run(nm, s)

    # -----------------------------------------------------------------------
    # 5. STAGE 3 :: BASELINES, SAME SCHEDULE
    # -----------------------------------------------------------------------
    if 3 in RUN_STAGES:
        print("\n" + "-" * 92)
        print("STAGE 3 :: BASELINES UNDER ONE SCHEDULE")
        print("-" * 92)
        for nm in BASELINES:
            if left_h() <= 0.5:
                print("  budget exhausted - rerun this cell"); break
            try:
                train_run(nm, SEEDS[0], epochs=BASELINE_EPOCHS, is_baseline=True)
            except Exception as e:
                print(f"  ! {nm}: {type(e).__name__}: {e}")

    # -----------------------------------------------------------------------
    # 6. STAGE 4 :: CUMULATIVE ABLATION
    # -----------------------------------------------------------------------
    if 4 in RUN_STAGES:
        print("\n" + "-" * 92)
        print("STAGE 4 :: CUMULATIVE COMPONENT ABLATION")
        print("-" * 92)
        VARIANTS = [
            ("A: reproduction",        dict(USE_FREQ_STREAM=False, USE_TEMPORAL=False,
                                            ADAPTIVE_ALPHA=False)),
            ("B: + frequency stream",  dict(USE_FREQ_STREAM=True, USE_TEMPORAL=False,
                                            ADAPTIVE_ALPHA=False)),
            ("C: + adaptive alpha",    dict(USE_FREQ_STREAM=True, USE_TEMPORAL=False,
                                            ADAPTIVE_ALPHA=True)),
            ("D: + temporal (full)",   dict(USE_FREQ_STREAM=True, USE_TEMPORAL=True,
                                            ADAPTIVE_ALPHA=True)),
        ]
        for tag, ov in VARIANTS:
            if left_h() <= 0.5:
                print("  budget exhausted - rerun this cell"); break
            try:
                train_run(PROPOSED, SEEDS[0], cfg_overrides=ov,
                          epochs=ABLATION_EPOCHS, tag=tag.split(":")[0])
            except Exception as e:
                print(f"  ! {tag}: {type(e).__name__}: {e}")


# ---------------------------------------------------------------------------
# 7. COLLECT AND TABULATE
# ---------------------------------------------------------------------------
rows = []
for p in sorted(EXP.glob("*.json")):
    d = _jr = None
    try:
        d = json.loads(p.read_text())
    except Exception:
        continue
    if isinstance(d, dict) and "frame_auc" in d:
        rows.append(d)

print("\n" + "=" * 92)
print("RESULTS SO FAR")
print("=" * 92)
if not rows:
    print("no completed runs yet")
else:
    R = pd.DataFrame(rows)[["model", "seed", "tag", "params_M", "epochs",
                            "frame_auc", "video_auc"]]
    R[["frame_auc", "video_auc"]] = (R[["frame_auc", "video_auc"]] * 100).round(2)
    R = R.sort_values(["model", "tag", "seed"])
    display(R)
    R.to_csv(EXP / "all_runs.csv", index=False)

    multi = R[R.tag.fillna("") == ""].groupby("model").agg(
        n=("seed", "count"),
        frame_mean=("frame_auc", "mean"), frame_std=("frame_auc", "std"),
        video_mean=("video_auc", "mean"), video_std=("video_auc", "std")).round(2)
    if (multi.n > 1).any():
        print("\nSEED VARIANCE (mean +/- std over seeds)")
        display(multi)
        pack = ROOT / "manuscript_pack" / "tables"
        pack.mkdir(parents=True, exist_ok=True)
        multi.reset_index().to_csv(pack / "tableXII_seed_variance.csv", index=False)
        print(f"  saved -> {pack / 'tableXII_seed_variance.csv'}")

    ab = R[R.tag.fillna("") != ""]
    if len(ab):
        print("\nCOMPONENT ABLATION")
        display(ab[["tag", "frame_auc", "video_auc", "params_M"]])
        pack = ROOT / "manuscript_pack" / "tables"
        pack.mkdir(parents=True, exist_ok=True)
        ab.to_csv(pack / "tableVII_component_ablation.csv", index=False)
        print(f"  saved -> {pack / 'tableVII_component_ablation.csv'}")

print(f"\nelapsed {elapsed_h():.2f} h")
print("Rerun this cell in a new session to continue - finished runs are skipped.")
print("Then rerun cells 3 and 4 to refresh the tables, and CELL T to re-score.")
print("=" * 92)


---
## 7 · The decision

Open `manuscript_pack/tables/tableXIII_significance.csv`, FSTT-Net vs STT-Repro row.

**CI entirely above zero** — the contributions do measurable work on aligned crops. Run
cell 6 to completion, then write. A Transactions submission is defensible.

**CI still contains zero** — more tuning will not fix it. Two honest options:

1. **Change the architecture, not the hyperparameters.** The current design adds capacity
   *in parallel* to the spatial stream, so the network can route around the spectral
   branch. `MODALITY_DROPOUT` is the cheapest test of that diagnosis; a deeper fix is to
   make the spectral evidence a bottleneck rather than an addition.
2. **Reframe the paper.** Report it as a controlled reproduction: *the added streams do
   not improve intra-dataset accuracy on a saturated benchmark, but they degrade more
   gracefully under compression and blur.* Your confidence intervals, calibration curves
   and degradation study already exceed what the base paper reports. That is publishable
   in IEEE Access, Neurocomputing, Pattern Recognition Letters, or WIFS/IJCB.

Either way, report the measured numbers. A careful negative result with a controlled
reproduction is a real contribution; a claimed improvement a reviewer cannot reproduce
is not.